***Laue Patterns digital processing to get single pixel or roi-related counters value over a dataset of images***

***pixel intensity monitoring***

*** useful visualize grains in 2D Map (amesh dmesh scan) , dips in DAXM including quick DAXM (zf ascan dscan), 1D scan***

*** no laue pattern indexing / refinement ***


Author: J.-S. Micha

Last Revision:   sept 2026

tested with python3.12  ubuntu 24.04 jupyter-slurm

**Objectives**

- Load and get pixel intensities on single pixel intensity or from ROIs statistics (pixel max intensity, position of max, postion of center of mass, position of gaussian fitting peak position)
- Visualize corresponding 2D or 1D scalar profile (intensity, spot position, etc)
- For 2D map (mesh scan): compilation of several map allows spatial determination of scattering region (grain, crystal, fluorescing material). It can be used to provide set of Laue spots very likely to belong to the same grain (useful for indexing routine)
- For 1D scan (ascan): in case of DAXM measurements, one can segment the laue spots per depth (quick-DAXM) and then again provide set of Laue spots very likely to belong to the same grain (useful for indexing routine). One can plot as well resconstructed intensities as a function of depth. For RAINBOW method diamond's scan, intensity profiles exhibiting dips (related to corrresponding Laue spot energy) can be compiled for monitroing.

Examples with following data, namely:

- A 321216, Data at /data/visitor/a321216, Laue camera is sCMOS_4M

- ME17010, Data at /data/visitor/me1701, Laue camera is sCMOS

some ways to obtain pixel intensity info in a set of images and at different given roi locations

since august 2025,  use the conda 2025 environnement distribution that contains lauetools (more or less recent version) and a cell that points to the current source code of LaueTools

# SET & check environment

In [ ]:
!which python

In [ ]:
JUPYTER_LAB = True # False for jupyter hub or notebook

# at ESRF during experiment and 90 days after at /data/visitor
# OR for a long term porject at /data/projects
DATA_AT_ESRF_NICE=True   # False

# data can be /data/visitor or /data/projects  and  anywhere on linux ESRF machine or linux personnal laptop
ON_LINUX = True  # False

# ideally hdf5 files and data must be organized by subfolders as BLISS does
HDF5_LOGFILE_EXISTS = True  # False

In [ ]:
# just to have a trace of current dependencies version
# if ON_LINUX:
#     !jupyter --version

# IMPORT packages

In [ ]:
# Setting absolute path to LaueTools Modules if default version of the source code is not up to date
#  inly working for venv created environment, not conda !!!
if 0:
    import sys
    # for slurm machines
    # sys.path.insert(0,'/home/esrf/micha/lauetools_devNotebooks/lauetools')
    # # for lbm32gpu1 machine (test temp)
    # dev lauetools package
    #sys.path.insert(0,'/data/bm32/inhouse/STAFF/JSM/lauetools_devNotebooks/lauetools')
    #sys.path.insert(0,'/home/esrf/micha/lauetools_devNotebooks/lauetools')

    sys.path.insert(0,'/data/bm32/inhouse/lauetoolsenv2/lib/python3.12/site-packages')
    
    import LaueTools as LT
    print('code from', LT.__file__)

In [ ]:
# Matplotlib backend selection (JupyterLab vs. JupyterHub)
if not JUPYTER_LAB: # For JupyterHub
    %matplotlib notebook  
else: # For JupyterLab
    %matplotlib widget    

# Standard library imports
import os
import time
import copy
import itertools
import glob
from pathlib import Path

# Numerical and data handling
import numpy as np
import pandas as pd
import h5py

# Plotting and widgets
import matplotlib.pyplot as plt
import matplotlib
import ipywidgets as widgets
from ipywidgets import (
    FloatProgress, IntSlider, Button, HBox, VBox,
    Output, FloatRangeSlider, interact, interactive,
    fixed, interact_manual
)
from IPython.display import display
import fabio

# LaueTools imports
import LaueTools
import LaueTools.GUI.mosaic as MOS
import LaueTools.generaltools as GT
import LaueTools.IOimagefile as IOimage
import LaueTools.imageprocessing as Improc
import LaueTools.dict_LaueTools as DictLT
import LaueTools.IOLaueTools as IOLT
import LaueTools.readmccd as RMCCD
import LaueTools.blissdatafolderstructure as bf
import LaueTools.logfile_reader as iohdf5
import LaueTools.blissscan as bscan

# Print LaueTools paths
print('LaueTools location:', LaueTools.__file__)
print('LaueTools path:', LaueTools.__path__)

from LaueTools.imagescollector import (
    collectpixelvalue_singlefile,
    collectroissum_singlefile,
    collectroisptp_singlefile,
    collectroiarray_singlefile,
    collectroismax_singlefile,
    collectroisXYmax_singlefile,
    collectroisXYcenterofmass_singlefile,
    collectroisfitpeak_singlefile
)

In [ ]:
from tqdm import tqdm, tqdm_notebook
import itertools

import multiprocessing
from multiprocessing import active_children, cpu_count
print(f'{multiprocessing.cpu_count()} cpu(s) are available!\n')

In [ ]:
if 0: # just to check user with permission and plot ability
    print(os.getgroups())
    fig_,ax_ = plt.subplots()
    ax_.plot(np.arange(20))
    

# SET user-defined `ExperimentFolder`

parent folder of all raw data ot the experiment. For data at ESRF during and after experiment, the path should contain 'RAW_DATA'

In [ ]:
#Experiment number or id?   the subfolder of date will be added if there is one
#expId = 'a321216', 'ma6030', 'hc5386', 'a322860', 'a322864', 'blc15488'
expId = 'me1701'  # colombo et al
expId = 'ma6758'  # akamatsu et al Jan 2026
expId = 'blc17163'  # blc June 2026 
expId = 'bm322606'  # a321212  June 
expId = 'blc17179'  #qmax Alumina ZrO2 Pt and also DAXM BaTiO3 waiting to write in blc17194
expId = 'a321217'  #test daxm and intensities
expId = 'sc5805'  # nacre dauphin et al
#expId = 'ihmi1613'  #map 2D   ceramic guinebretiere
expId = 'a321220'  # guinebretiere MgO



## a321220  Guinebretiere et al  Mgo MgFe2O4

In [ ]:
if expId == 'a321220':  # 
    print(expId)
    
    CCDLabel = 'EIGER_4MCdTe'  # default sCMOS  = quad  (4M)  or 'sCMOS_4M'  binned 3x3 36M IMAGESTAR
    
    DATA_AT_ESRF_NICE = True
    HDF5_LOGFILE_EXISTS = True

    #ExperimentFolder = bf.getExperimentFolder(expId, data_at_esrf=DATA_AT_ESRF_NICE, expDate='20260915')
    ExperimentFolder = '/data/visitor/a321220/bm32/20260915/RAW_DATA/'
    ExperimentFolder

## sc5805   Dauphin et l nacre

In [ ]:
if expId == 'sc5805':  # 
    print(expId)
    
    CCDLabel = 'EIGER_4MCdTe'  # default sCMOS  = quad  (4M)  or 'sCMOS_4M'  binned 3x3 36M IMAGESTAR
    
    DATA_AT_ESRF_NICE = True
    HDF5_LOGFILE_EXISTS = True

    ExperimentFolder = bf.getExperimentFolder(expId, data_at_esrf=DATA_AT_ESRF_NICE)
    ExperimentFolder

## ih mi 1613  ihr  test guinebretiere nanoxml

In [ ]:
if expId == 'ihmi1613':  # 
    print(expId)
    
    CCDLabel = 'EIGER_4MCdTe'  # default sCMOS  = quad  (4M)  or 'sCMOS_4M'  binned 3x3 36M IMAGESTAR
    
    DATA_AT_ESRF_NICE = True
    HDF5_LOGFILE_EXISTS = True

    ExperimentFolder = bf.getExperimentFolder(expId, data_at_esrf=DATA_AT_ESRF_NICE)
    ExperimentFolder
    

## A321217 test micha

In [ ]:
if expId == 'a321217':  # 
    print(expId)
    
    CCDLabel = 'EIGER_4MCdTe'  # default sCMOS  = quad  (4M)  or 'sCMOS_4M'  binned 3x3 36M IMAGESTAR
    
    DATA_AT_ESRF_NICE = True
    HDF5_LOGFILE_EXISTS = True

    ExperimentFolder = bf.getExperimentFolder(expId, data_at_esrf=DATA_AT_ESRF_NICE)
    ExperimentFolder

## BLC 17179 qmax & BaTiO3  map and daxm

In [ ]:
if expId == 'blc17179':  # 
    print(expId)
    
    CCDLabel = 'EIGER_4MCdTe'  # default sCMOS  = quad  (4M)  or 'sCMOS_4M'  binned 3x3 36M IMAGESTAR
    
    DATA_AT_ESRF_NICE = True
    HDF5_LOGFILE_EXISTS = True
    if DATA_AT_ESRF_NICE:
        
        nicefolder = 'visitor'
        #nicefolder = os.path.join('bm32','inhouse')
    
        #--------------------------------------------
        ExperimentFolder = os.path.join('/data',nicefolder, f'{expId}/bm32/')
    
        listdates = os.listdir(ExperimentFolder)
        print('possible dates',listdates)
        if len(listdates)>1:
            GT.printyellow(f'\nBe careful, there are several dates ... => {listdates}\n')
        
        # to uncomment two lines to precise the date if there are several ones
        # ExperimentFolder= bf.setExperimentFolder_with_date(ExperimentFolder, expDate='20250212')
        # ExperimentFolder = os.path.join(ExperimentFolder,'RAW_DATA')
        
        ExperimentFolder= bf.setExperimentFolder_with_date(ExperimentFolder)
       
        print('ExperimentFolder set to ', ExperimentFolder)
    else:
        ExperimentFolder = userdefined_datafolder
        
    if os.path.exists(ExperimentFolder):
        GT.printgreen(f'\n"ExperimentFolder" exists ! : \n{ExperimentFolder}')
        
    if DATA_AT_ESRF_NICE and 'RAW_DATA' not in ExperimentFolder:
        GT.printyellow(f'\n"ExperimentFolder" does not contain "RAW_DATA"! Are you sure?')

## A321212 Cornelius Si NW

In [ ]:
if expId == 'bm322606':  # 
    print(expId)
    
    CCDLabel = 'EIGER_4MCdTe'  # default sCMOS  = quad  (4M)  or 'sCMOS_4M'  binned 3x3 36M IMAGESTAR
    
    DATA_AT_ESRF_NICE = True
    HDF5_LOGFILE_EXISTS = True
    if DATA_AT_ESRF_NICE:
        
        nicefolder = 'visitor'
        nicefolder = os.path.join('bm32','inhouse')
    
        #--------------------------------------------
        ExperimentFolder = os.path.join('/data',nicefolder, f'{expId}/bm32/')
    
        listdates = os.listdir(ExperimentFolder)
        print('possible dates',listdates)
        if len(listdates)>1:
            GT.printyellow(f'\nBe careful, there are several dates ... => {listdates}\n')
        
        # to uncomment two lines to precise the date if there are several ones
        # ExperimentFolder= bf.setExperimentFolder_with_date(ExperimentFolder, expDate='20250212')
        # ExperimentFolder = os.path.join(ExperimentFolder,'RAW_DATA')
        
        ExperimentFolder= bf.setExperimentFolder_with_date(ExperimentFolder)
       
        print('ExperimentFolder set to ', ExperimentFolder)
    else:
        ExperimentFolder = userdefined_datafolder
        
    if os.path.exists(ExperimentFolder):
        GT.printgreen(f'\n"ExperimentFolder" exists ! : \n{ExperimentFolder}')
        
    if DATA_AT_ESRF_NICE and 'RAW_DATA' not in ExperimentFolder:
        GT.printyellow(f'\n"ExperimentFolder" does not contain "RAW_DATA"! Are you sure?')

## blc17163 BaTiO3

In [ ]:
if expId == 'blc17163':  # test HERAUD
    print(expId)
    
    CCDLabel = 'EIGER_4MCdTe'  # default sCMOS  = quad  (4M)  or 'sCMOS_4M'  binned 3x3 36M IMAGESTAR
    
    DATA_AT_ESRF_NICE = True
    HDF5_LOGFILE_EXISTS = True
    if DATA_AT_ESRF_NICE:
        
        nicefolder = 'visitor'
        #nicefolder = os.path.join('projects','mapgrainxl')
    
        #--------------------------------------------
        ExperimentFolder = os.path.join('/data',nicefolder, f'{expId}/bm32/')
    
        listdates = os.listdir(ExperimentFolder)
        print('possible dates',listdates)
        if len(listdates)>1:
            GT.printyellow(f'\nBe careful, there are several dates ... => {listdates}\n')
        
        # to uncomment two lines to precise the date if there are several ones
        # ExperimentFolder= bf.setExperimentFolder_with_date(ExperimentFolder, expDate='20250212')
        # ExperimentFolder = os.path.join(ExperimentFolder,'RAW_DATA')
        
        ExperimentFolder= bf.setExperimentFolder_with_date(ExperimentFolder)
       
        print('ExperimentFolder set to ', ExperimentFolder)
    else:
        ExperimentFolder = userdefined_datafolder
        
    if os.path.exists(ExperimentFolder):
        GT.printgreen(f'\n"ExperimentFolder" exists ! : \n{ExperimentFolder}')
        
    if DATA_AT_ESRF_NICE and 'RAW_DATA' not in ExperimentFolder:
        GT.printyellow(f'\n"ExperimentFolder" does not contain "RAW_DATA"! Are you sure?')

## Lhuissier et al AL

In [ ]:
if expId == 'ma6758':  # test HERAUD
    print(expId)
    
    CCDLabel = 'EIGER_4MCdTe'  # default sCMOS  = quad  (4M)  or 'sCMOS_4M'  binned 3x3 36M IMAGESTAR
    
    DATA_AT_ESRF_NICE = True
    HDF5_LOGFILE_EXISTS = True
    if DATA_AT_ESRF_NICE:
        
        nicefolder = 'visitor'
        #nicefolder = os.path.join('projects','mapgrainxl')
    
        #--------------------------------------------
        ExperimentFolder = os.path.join('/data',nicefolder, f'{expId}/bm32/')
    
        listdates = os.listdir(ExperimentFolder)
        print('possible dates',listdates)
        if len(listdates)>1:
            GT.printyellow(f'\nBe careful, there are several dates ... => {listdates}\n')
        
        # to uncomment two lines to precise the date if there are several ones
        # ExperimentFolder= bf.setExperimentFolder_with_date(ExperimentFolder, expDate='20250212')
        # ExperimentFolder = os.path.join(ExperimentFolder,'RAW_DATA')
        
        ExperimentFolder= bf.setExperimentFolder_with_date(ExperimentFolder)
       
        print('ExperimentFolder set to ', ExperimentFolder)
    else:
        ExperimentFolder = userdefined_datafolder
        
    if os.path.exists(ExperimentFolder):
        GT.printgreen(f'\n"ExperimentFolder" exists ! : \n{ExperimentFolder}')
        
    if DATA_AT_ESRF_NICE and 'RAW_DATA' not in ExperimentFolder:
        GT.printyellow(f'\n"ExperimentFolder" does not contain "RAW_DATA"! Are you sure?')

## Cr/Zr A321216 data Ribart et al

In [ ]:
if expId == 'a321216':
    
    CCDLabel = 'sCMOS_4M'  # default sCMOS  = quad  (4M)  or 'sCMOS_4M'  binned 3x3 36M IMAGESTAR
    
    userdefined_datafolder =  '/my/folder/to/data'
    
    if DATA_AT_ESRF_NICE:
        
        nicefolder = 'visitor'
        #nicefolder = os.path.join('projects','mapgrainxl')
    
        #--------------------------------------------
        ExperimentFolder = os.path.join('/data',nicefolder, f'{expId}/bm32/')
    
        listdates = os.listdir(ExperimentFolder)
        print('possible dates',listdates)
        if len(listdates)>1:
            GT.printyellow(f'\nBe careful, there are several dates ... => {listdates}\n')
        
        # to uncomment two lines to precise the date if there are several ones
        # ExperimentFolder= bf.setExperimentFolder_with_date(ExperimentFolder, expDate='20250212')
        # ExperimentFolder = os.path.join(ExperimentFolder,'RAW_DATA')
        
        ExperimentFolder= bf.setExperimentFolder_with_date(ExperimentFolder)
       
    
        print('ExperimentFolder set to ', ExperimentFolder)
    else:
        ExperimentFolder = userdefined_datafolder
        
    if os.path.exists(ExperimentFolder):
        GT.printgreen(f'\n"ExperimentFolder" exists ! : \n{ExperimentFolder}')
        
    if DATA_AT_ESRF_NICE and 'RAW_DATA' not in ExperimentFolder:
        GT.printyellow(f'\n"ExperimentFolder" does not contain "RAW_DATA"! Are you sure?')

## A2O3 ME1701 data Colombo et al

In [ ]:
if expId == 'me1701':

    CCDLabel = 'sCMOS'  # default sCMOS  = quad  (4M)  or 'sCMOS_4M'  binned 3x3 36M IMAGESTAR
    
    userdefined_datafolder =  '/my/folder/to/data'
    
    if DATA_AT_ESRF_NICE:
        
        nicefolder = 'visitor'
        #nicefolder = os.path.join('projects','mapgrainxl')
    
        #--------------------------------------------
        ExperimentFolder = os.path.join('/data',nicefolder, f'{expId}/bm32/')
    
        listdates = os.listdir(ExperimentFolder)
        print('possible dates',listdates)
        if len(listdates)>1:
            GT.printyellow(f'\nBe careful, there are several dates ... => {listdates}\n')
        
        # to uncomment two lines to precise the date if there are several ones
        # ExperimentFolder= bf.setExperimentFolder_with_date(ExperimentFolder, expDate='20250212')
        # ExperimentFolder = os.path.join(ExperimentFolder,'RAW_DATA')
        
        ExperimentFolder= bf.setExperimentFolder_with_date(ExperimentFolder)
       
    
        print('ExperimentFolder set to ', ExperimentFolder)
    else:
        ExperimentFolder = userdefined_datafolder
        
    if os.path.exists(ExperimentFolder):
        GT.printgreen(f'\n"ExperimentFolder" exists ! : \n{ExperimentFolder}')
        
    if DATA_AT_ESRF_NICE and 'RAW_DATA' not in ExperimentFolder:
        GT.printyellow(f'\n"ExperimentFolder" does not contain "RAW_DATA"! Are you sure?')

# Browse data folders or hdf5 file, select a scan, and build rapidly scan parameter dictionnary `d`.

`d` to be used in next section

## [OPTIONS] list of folders and files by OS

In [ ]:
#ExperimentFolder = bf.getExperimentFolder(expId, data_at_esrf=DATA_AT_ESRF_NICE)
print('"ExperimentFolder" is set to:\n',ExperimentFolder)

In [ ]:
 bf.ls_folder(ExperimentFolder)

In [ ]:
bf.tree(ExperimentFolder,0)

In [ ]:
bf.tree(Path(ExperimentFolder)/'MH73_HT',0, truncatesize=15)

## read the hdf5 logfile: 

In [ ]:
# use the master (uppest level) h5 file
pathHDF5, HDF5_LOGFILE_EXISTS = bf.findmasterh5file(ExperimentFolder)
print(pathHDF5, HDF5_LOGFILE_EXISTS)

# or use a lower level h5 file
subfolder ='MH73_HT'
pathHDF5_subfile,  HDF5_LOGFILE_EXISTS= bf.findmasterh5file(os.path.join(ExperimentFolder,subfolder))
print(pathHDF5_subfile,  HDF5_LOGFILE_EXISTS)

In [ ]:
#logfile = iohdf5.H5file(pathHDF5, CCDLabel)
logfile = iohdf5.H5file(pathHDF5_subfile, CCDLabel)
print('selected logfile .h5 :',logfile.path)

In [ ]:
#logfile.getscans(node_substrings=['map2D'],verbose=1)
logfile.getscans(verbose=1)

In [ ]:
logfile.dfallscans

In [ ]:
d = logfile.build_dict_scan(78)

In [ ]:
d

In [ ]:
#logfile.selectedscans

##  [OPTION] Scan mining in dataframe list

### Location of mesh scans (2D map)

In [ ]:
dfmeshscans = None
if HDF5_LOGFILE_EXISTS:
    # ALL MESH SCANS  (2D Map)
    pd.set_option('max_colwidth', 200)
    dfmeshscans = dfallscans[dfallscans['scantype']=='amesh'][['sample_dataset_scanindex','fullcommand', 'imagefolder', 'motors']]
dfmeshscans

In [ ]:
# to see a specific item  use ** iloc **  (absolute index in dfmeshscans)
item_idx = 2

if item_idx<len(dfmeshscans):
    print(dfmeshscans.iloc[item_idx])
else:
    GT.printyellow('this item index does not exist')

### Location of zf or thf scans (DAXM)

In [ ]:
# ALL ZF SCANS (DAXM Wire scans) or Diamond scan (thf)
if HDF5_LOGFILE_EXISTS:

    motorname = 'zf'
    print('location of images for scans of %s'%motorname)
    dfallscans.loc[dfallscans['motors'] == motorname]['imagefolder']
    
dfallscans

In [ ]:
dfzfscans = None
if HDF5_LOGFILE_EXISTS:

    pd.set_option('max_colwidth', 200)
    dfzfscans = dfallscans[dfallscans['motors']=='zf'][['sample_dataset_scanindex','fullcommand', 'imagefolder', 'motors']]
dfzfscans

### Location of scans for specified sample

In [ ]:
# narrow the search of specific folder

selected_scans = None

#df = dfzfscans
df = dfallscans

if expId == 'me1701':
    str_in_dataset = 'A45'  #'TRUE_COM' #'A45'
    motor_in_motors =  'zf' #'thf'
if expId == 'a321216':
    str_in_dataset = 'TRUE_COM' 
    motor_in_motors =  'thf'

cond_dataset  = df['sample_dataset_scanindex'].str.contains(str_in_dataset)
cond_motor  = df['fullcommand'].str.contains(motor_in_motors)

cond = cond_dataset & cond_motor

if HDF5_LOGFILE_EXISTS:
    selected_scans=df[cond][
                        ['sample_dataset_scanindex','fullcommand', 'imagefolder', 'motors']]

selected_scans

### retrieve bliss command and data from image and `dfallscans` from hdf5 file

In [ ]:
if expId == 'me1701':
    queryimage = os.path.join(d290['folder'],'img_0400.tif')
    print('image', queryimage)

    blisscommand, imagedate, dfallscans_index = IOimage.fromscmosdate2blisscommand(queryimage, dfallscans)
    print('blisscommand',blisscommand)
    print('imagedate',imagedate)
    print('dfallscans_index', dfallscans_index)

#### Example 1: Select the scan collecting images at querydate

In [ ]:
import datetime

querydate = datetime.datetime(2024,6,18,23,33,30)  # y m d  h min sec
print('querydate',querydate)

mask = (dfallscans['end_time'] > querydate.isoformat()) & (dfallscans['start_time'] <= querydate.isoformat())
dfallscansgooddate = dfallscans.loc[mask]

print(dfallscansgooddate.fullcommand.values)
dfallscansgooddate

#### Example 2: Select the scan collecting images at during the period

In [ ]:
# period is defined by  [querydate-timespan, querydate+timespan]
querydate = datetime.datetime(2024,6,18,23,33,30)  # y m d  h min sec
timespan = 60 # min

minidate = datetime.datetime.fromtimestamp(querydate.timestamp()-timespan*60)
maxidate = datetime.datetime.fromtimestamp(querydate.timestamp()+timespan*60)
mask2 = (dfallscans['end_time'] < maxidate.isoformat()) & (dfallscans['start_time'] > minidate.isoformat() )
dfallscansgoodperiod = dfallscans.loc[mask2]

#print(dfallscansgoodperiod.fullcommand.values)
print(dfallscansgoodperiod[['fullcommand', 'imagefolder', 'start_time']])
dfallscansgoodperiod

### Rapid access to image folder  in dev

In [ ]:
# get images_subfolder from dataframe index and fullpath image folder
dfallscans_idx = 290
dataframescanslist = dfallscans  # whatever ... dfzfscans, dfallscans

#*********************************
#to see a specific item property  use ** loc **
fullpath_imagefolder = dataframescanslist.loc[dfallscans_idx]['imagefolder']

images_subfolder = bf.setimages_subfolder(fullpath_imagefolder, rootfolder='RAW_DATA')
print('for item #%d'%dfallscans_idx)
print('images_subfolder is (to be copied afterwards if needed):')
print(images_subfolder)

## [OPTION] SET manually scan parameters (not recommended), just for illustration now

In [ ]:
try:
    print('ExperimentFolder',ExperimentFolder)
except NameError:
    ExperimentFolder = '/home/micha'
    print('ExperimentFolder set to', ExperimentFolder)

IMAGES_IN_SINGLE_FOLDER= True  # True is default

#  RARE CASES --------images for a single scan saved and spread over several folder----------------------------
if not IMAGES_IN_SINGLE_FOLDER: # not general
    # images dispatched in multiple folders !
    indexrange=(2611,2661)
    subfolder = '18320-1/18320-1_0001'

    listimagefolder=[os.path.join(ExperimentFolder,subfolder,'scan%04d'%ii) for ii in range(indexrange[0],indexrange[1]+1)]
    mapdimensions=(len(listimagefolder),1)
    dictscanshfoc={
    'folder':listimagefolder,
        'prefix': 'img_',
        'listindices' : 0,
        'mapdimensions': mapdimensions,
        'CCDLabel' :'sCMOS' ,
        'peaklistfile': None,
        'collector': 'pixelval',
        'nbimagesperline': mapdimensions[0],
    }

elif IMAGES_IN_SINGLE_FOLDER:
    pass

# GENERAL CASE : for multiple images in the same folder

#ExperimentFolder#
#"/data/visitor/a321206/bm32/20250603/RAW_DATA/ZrO2/ZrO2_map2D/scan0001/"

# MAP 2D
#amesh yech -1.30821 -1.26821 80 xech -8.64697 -8.56697 160 0.04
mapdimensions=(61,61)   # = (nb of steps+1, nb of steps +1 )
dictmap2D={
'folder':os.path.join(ExperimentFolder,'ZrO2/ZrO2_map2D/scan0001/'),
    'prefix': 'img_',
    'listindices' : np.arange(0,mapdimensions[0]*mapdimensions[1]),
    'mapdimensions': mapdimensions,
    'CCDLabel' :'sCMOS' ,
    'peaklistfile': None,
    'collector': 'pixelval',
    'nbimagesperline': mapdimensions[0],
    'fastaxis':'yech',  # motor that moves first to form the first line
    'slowaxis':'xech', # motor that moves once to go from one line to the other
    'scantype':'map',
}

# MAP 2D
#amesh yech -1.30821 -1.26821 80 xech -8.64697 -8.56697 160 0.04
mapdimensions=(301,100)   # = (nb of steps+1, nb of steps +1 )
dictmapbig2D={
'folder':os.path.join(ExperimentFolder,'ZrO2/ZrO2_ZrO2_1330C/scan0002/'),
    'prefix': 'img_',
    'listindices' : np.arange(0,mapdimensions[0]*mapdimensions[1]),
    'mapdimensions': mapdimensions,
    'CCDLabel' :'sCMOS' ,
    'peaklistfile': None,
    'collector': 'pixelval',
    'nbimagesperline': mapdimensions[0],
    'fastaxis':'yech',  # motor that moves first to form the first line
    'slowaxis':'xech', # motor that moves once to go from one line to the other
    'scantype':'map',
}


#amesh 
mapdimensions=(11,11)
dictmap2Dbig={
'folder':os.path.join(ExperimentFolder,'wednesday/wednesday_A42ZTAA_A_test_0002/scan0001'),
    'prefix': 'img_',
    'listindices' : np.arange(0,mapdimensions[0]*mapdimensions[1]),
    'mapdimensions': mapdimensions,
    'CCDLabel' :'sCMOS' ,
    'peaklistfile': None,
    'collector': 'pixelval',
    'nbimagesperline': mapdimensions[0],
    'fastaxis':'yech', 
    'slowaxis':'xech',
    'scantype':'map',
}


#  DAXM   A45/A45_line1daxms/scan0005 
mapdimensions=(420,1)

dictdaxm={
'folder':os.path.join(ExperimentFolder,'A45/A45_line1daxms/scan0005'),
    'prefix': 'img_',
    'listindices' : np.arange(0,mapdimensions[0]*mapdimensions[1]),
    'mapdimensions': mapdimensions,
    'CCDLabel' :'sCMOS' ,
    'peaklistfile': None,
    'collector': 'pixelval',
    'nbimagesperline': mapdimensions[0],
    'fastaxis':'zf',  # motor that moves first to form the first line}
    'scantype':'daxm',
}

# DAXM Calibration
mapdimensions=(420,1)

dictdaxmGe={
'folder':os.path.join(ExperimentFolder,'A45ZTAA/A45ZTAA_Gedaxm/scan0001'),
    'prefix': 'img_',
    'listindices' : np.arange(0,mapdimensions[0]*mapdimensions[1]),
    'mapdimensions': mapdimensions,
    'CCDLabel' :'sCMOS' ,
    'peaklistfile': None,
    'collector': 'pixelval',
    'nbimagesperline': mapdimensions[0],
    'fastaxis':'zf',
    'scantype':'daxm'
}

mapdimensions=(420,1)
d290 = {'scantype': 'daxm', 'start_time': '2024-11-15T00:25:21.473512+01:00', 
        'end_time': '2024-11-15T00:29:28.756940+01:00', 
        'sample_dataset_scanindex': 'A45ZTAA_A45ZTAAwire1_daxms_1',
        'fullcommand': 'ascan zf -2.563000 -2.143000 419 0.35', 
        'scanindex': '1', 'motors': 'zf', 
        'localhdf5file': '/data/visitor/me1701/bm32/20241113/RAW_DATA/A45ZTAA/A45ZTAA_A45ZTAAwire1_daxms/A45ZTAA_A45ZTAAwire1_daxms.h5', 
        'imagefolder': '/data/visitor/me1701/bm32/20241113/RAW_DATA/A45ZTAA/A45ZTAA_A45ZTAAwire1_daxms/scan0001', 
        'folder': '/data/visitor/me1701/bm32/20241113/RAW_DATA/A45ZTAA/A45ZTAA_A45ZTAAwire1_daxms/scan0001', 
        'prefix': 'img_', 'listindices': np.arange(0,mapdimensions[0]*mapdimensions[1]), 
        'nbimagesperline': mapdimensions[0], 
        'mapdimensions': mapdimensions, 'peaklistfile': None, 
        'fastaxis': 'zf', 'slowaxis': None, 
        'collector': 'pixelval', 'CCDLabel': 'sCMOS'}


# rainbow scan 
mapdimensions=(501,1)

dictrainbow={
'folder':os.path.join(ExperimentFolder,'Ni3_11pt/Ni3_11pt_point_0012/scan0001'),
    'prefix': 'img_',
    'listindices' : np.arange(0,mapdimensions[0]*mapdimensions[1]),
    'mapdimensions': mapdimensions,
    'CCDLabel' :'sCMOS' ,
    'peaklistfile': None,
    'collector': 'pixelval',
    'nbimagesperline': mapdimensions[0],
    'fastaxis':'thf',
    'scantype':''
}

dictrainbow2={
'folder':os.path.join(ExperimentFolder,'Ni3_rainbow/Ni3_rainbow_point_0002/scan0001'),
    'prefix': 'img_',
    'listindices' : np.arange(0,mapdimensions[0]*mapdimensions[1]),
    'mapdimensions': mapdimensions,
    'CCDLabel' :'sCMOS' ,
    'peaklistfile': None,
    'collector': 'pixelval',
    'nbimagesperline': mapdimensions[0],
    'fastaxis':'thf',
    'scantype':''
}

## [OPTION] SET dict `d`  to one from above dictionnaries or from iohdf5.build_dict_scan() (see previous section)

In [ ]:
# ***********FINAL CHOICE **************
if expId == 'a321217':
    d=d15
    d=d23
    d=d0
   
if expId == 'blc17179':
    d=d59
    d=d461
    d=d465
    d=d5
if expId == 'bm322606':
    d=d25
    d=d26
    d=d27
    #d=d30
    d=d33
if expId == 'blc17163':
   
    d= d9  #   BaTiO3 map
    d =d10  #  BaTiO3 large map

    d =d12  #  FE large map
if expId == 'ma6758':
    d= d48  # daxm   650 micron   daxm   expp 1 sec
    d= d89  # daxm   650 micron   daxm  expo 5 sec
   

if expId == 'me1701':
    d= d289  # Ge
    #
    d= d290   # alumina

# experimental  .... CHoose workflow

In [ ]:
s = bscan.BlissScan(d)
#print('s',s)

In [ ]:
s.possible_workflows()

In [ ]:
s.launch_workflow('mosaic')

# DEFINE and CHOOSE Counters (ROIs) location: from peaksearch, evenly spaced (gridroi) or randomly chosen

## [OPTION 1] peaksearch on single image (No peakshape FIT): integer coordinates

### compute `roiposition` from peaksearch

In [ ]:
imageindex = 1300

Data_for_localMaxima = 'auto_background'

#Manual entry
if expId == 'ihmi1613':
    imageindex=3760//2 

    IntensityThreshold = 100
    
if expId == 'a321217':
    imageindex=5  

    IntensityThreshold = 100
    Data_for_localMaxima = None
elif expId == '322812':
    imageindex=0  # Al phase
    
    #imageindex=6  # Al2Cu phase
    IntensityThreshold = 500
elif expId == 'me1701':
    imageindex=0
    IntensityThreshold = 2000 #for Al2O3 phase
elif expId == 'hc6287':
    imageindex=0
else:
    #imageindex=0
    IntensityThreshold = 1500  # 200 for Ge wafer

#--------------end of user input--------------------
ADD_ROIS_FOR_DAXM_BACKGROUND= False
if isinstance(d['folder'],list):
    folder = d['folder'][0]
else:
    folder = d['folder']
intensities = None
roiposition = None
r = RMCCD.PeakSearch(os.path.join(folder,f"{d['prefix']}%04d.{d['suffix']}"%imageindex),
                    CCDLabel=d['CCDLabel'],
                    IntensityThreshold=IntensityThreshold,
                    Data_for_localMaxima=Data_for_localMaxima,
                    fit_peaks_gaussian=0,
                    local_maxima_search_method=0,
                    boxsize=5,
                    Saturation_value= 6000000,  
                    PeakSizeRange=(.2, 1),
                    maxPixelDistanceRejection=5,
                    NumberMaxofFits=20000,
                    verbose=0)  # high value to for 1000s spots
if r is None:
    GT.printyellow('Nothing found. Try to change "IntensityThreshold" neither too small nor too large (first: 1000-5000 ?)')
if r is not None:
    roiposition = r[-1]
    nb_peaks_found = len(roiposition)
    if nb_peaks_found>0:
        GT.printgreen(f"nb peaks {len(roiposition)}')
    intensities = r[0][:,2]
    
    #hottest pixel:
    hottestspot_index = np.argsort(r[0][:,2])[::-1][0]
    print('hottest pixel : [X, Y, intensity above background] is:\n', r[0][hottestspot_index][:3])

In [ ]:
MASK_REGIONS = False
if MASK_REGIONS:
    XY= roiposition
    #Xtest, Ytest=[2324,3521]
    Xtest, Ytest=[2226,3670]
    xcond =np.fabs(XY[:,0]-Xtest)<5
    ycond =np.fabs(XY[:,1]-Ytest)<10
    cond = np.logical_and(xcond,ycond)
    np.where(cond)

In [ ]:
SORTED_BY_INTENSITY = True
if SORTED_BY_INTENSITY:# sorted by decreasing intensity
    sortedindices = np.argsort(r[0][:,2])[::-1]
    speaks = r[0][sortedindices]
    roiposition = speaks[:,:2]
    intensities = r[0][:,2][sortedindices]
    print('   X, \t\t\tY, \t\tIntensity above BackGround')
    print(np.column_stack((roiposition, intensities)))

### GUI: laue pattern image browser + `roiposition` from peaksearch

In [ ]:
fullpath = os.path.join(folder,f"{d['prefix']}%04d.{d['suffix']}"%imageindex)
os.path.exists(fullpath)

In [ ]:
from ipywidgets import interact, interactive, fixed, interact_manual

In [ ]:
import os
import fabio
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive

# Assume `d` is a dictionary containing:
# - 'prefix': Prefix for image filenames
# - 'suffix': Suffix for image filenames
# - 'mapdimensions': Tuple (nslow, nfast) representing rows and columns
# - 'listindices': List of valid indices (if needed)
# - 'folder': Path to the folder containing images

# Extract dimensions
nslow, nfast = d['mapdimensions']  # to be cchecked for the convention
col0 = nfast//2
row0 = nslow//2

# Initialize figure and axis
figp, axp = plt.subplots()
plt.show(block=False)  # Show the figure in non-blocking mode

# Variable to store the clicked pixel
clicked_pixel = None

# Output widget for printing clicked coordinates
output = Output()
display(output)

# Function to handle click events
# Function to handle click events
def on_click(event):
    global clicked_pixel
    if event.inaxes == axp:  # Ensure the click is within the image axes
        x, y = float(event.xdata), float(event.ydata)
        clicked_pixel = (int(x), int(y))
        with output:
            output.clear_output()
            print(f"Clicked pixel: (x={x:.1f}, y={y:.1f})")

# Connect the click event to the figure
figp.canvas.mpl_connect('button_press_event', on_click)


axp.imshow(imgdata, vmin = 0, vmax =2000)#, cmap=plt.cm.OrRd)

# Function to plot the image based on rowindex and colindex
def plotimage(rowindex=0, colindex=0, vmin=1000, vmax=2000, show_rois=False):
    global clicked_pixel
    # Calculate imageindex
    imageindex = rowindex * nfast + colindex

    # Construct the full path
    fullpath = os.path.join(folder, f"{d['prefix']}%04d.{d['suffix']}" % imageindex)

    # Check if the file exists
    if not os.path.exists(fullpath):
        print(f"\n'fullpath' does not exist!:\n{fullpath}")
        return

    # Open the image and get data
    with fabio.open(fullpath) as img:
        imgdata = img.data


    print('imageindex', imageindex)

    # Clear the axis and set limits
    ymin, ymax = axp.get_ylim()
    xmin, xmax = axp.get_xlim()
    axp.clear()
    axp.set_xlim(xmin, xmax)
    axp.set_ylim(ymin, ymax)

    # Adjust limits if needed
    ymin, ymax, xmin, xmax = int(ymin), int(ymax), int(xmin), int(xmax)
    if ymin > ymax:
        ymin, ymax = ymax, ymin

    # Set the title
    if 'RAW_DATA' in fullpath:
        figtitle = fullpath.rsplit('/RAW_DATA/')[1]
    else:
        figtitle = fullpath
    figp.suptitle(f"{ExperimentFolder}\n{figtitle}")

    # Display the image
    axp.imshow(imgdata, vmin=vmin, vmax=vmax, cmap=plt.cm.BuGn)

    # Show ROIs if enabled
    if show_rois:
        offsetXYtoplot = (1, 1)
        for pt in roiposition:
            axp.scatter(pt[0] - offsetXYtoplot[0], pt[1] - offsetXYtoplot[1], marker='+', color='r')

    plt.draw()  # Redraw the figure

# Create a button to print the clicked pixel
button = Button(description="Save Clicked Pixel")

def on_button_click(b):
    global clicked_pixel
    with output:
        output.clear_output()
        if clicked_pixel is not None:
            print(f"Saved clicked pixel: {clicked_pixel}")
        else:
            print("No pixel clicked yet.")

button.on_click(on_button_click)

# Display the button and output
display(button)

# Create interactive sliders
interactive(
    plotimage,
    colindex=IntSlider(min=0, max=nfast-1, step=1, value=col0, continuous_update=False, tooltip=f"along {d['fastaxis']}"),
    rowindex=IntSlider(min=0, max=nslow-1, step=1, value=row0, continuous_update=False, tooltip=f"along {d['slowaxis']}"),
    vmin=(0, 50),
    vmax=(100, 5000),
    show_rois=[False, True]
)

In [ ]:
10*0.075/100*180/3.14159

In [ ]:
print('clicked_pixel', clicked_pixel)
d['roicenter']=clicked_pixel  # for section 7 mosaic plot

In [ ]:
# [OPTION] [HELPER] find the position of a roi given XY pixels coordinates
if 0:
    XY= roiposition
    #Xtest, Ytest=[2324,3521]
    Xtest, Ytest=[1156,1825]
    Xtest, Ytest=[918,1730]
    
    xtolerance, ytolerance = 5, 10
    
    #--------------------------------------
    xcond =np.fabs(XY[:,0]-Xtest)<xtolerance
    ycond =np.fabs(XY[:,1]-Ytest)<ytolerance
    cond = np.logical_and(xcond,ycond)
    potential_index = np.where(cond)[0]
    
    potential_index, XY[potential_index]

#### useful tools to sort spots (according to Y) for DAXM

In [ ]:
SORT_BY_Y=False

# FOR Diamond could be useful (scan thf)
# SORT_BY_Y = True

if d['scantype']=='daxm':  # mandatory tof the next algorithms
    GT.printgreen(f'DAXM scan!')
    SORT_BY_Y = True

#--------------------------------
if SORT_BY_Y:
    # sort by increasing Y
    sortedYindices = np.argsort(roiposition[:,1])
    roiposition= np.take(roiposition,sortedYindices, axis=0)
    intensities = np.take(intensities,sortedYindices, axis=0)
    GT.printgreen(f'Sorted {len(roiposition)} peaks by intensity increasing pixel Y in "roiposition" ')

#### [OPTION] Modify ROIs list

In [ ]:
MANUAL_MODIFY= False
if MANUAL_MODIFY:# manual modification of roi (overwrite)
    roiposition[-1]=np.array([744,700])
#     roiposition[1]=np.array([1116,853])
#     roiposition[2]=np.array([424,1936])

In [ ]:
MANUAL_ADD= False

if MANUAL_ADD:
    roiposition= np.concatenate((np.array([[744,700],[900,958],[550,360]]),roiposition), axis=0)
    # you may need to add elements `intensities` array
    intensities= np.concatenate((np.array([1,1,1]),intensities), axis=0)

In [ ]:
print('total number of spots in list: ',len(roiposition))
np.set_printoptions(suppress=True,precision=3)
print('   X,      Y,      Intensity above BackGround')
print(np.column_stack((roiposition, intensities)))
np.set_printoptions(suppress=True,precision=7)

### CAN be SKIPPED! [ADVANCED LEVEL FOR DAXM] Build list of Background ROIS close to current ROIS locations

For each ROI, it will find location of background pixels (to estimate background fluorescence level).

**Create background roi: `bckroiposition` and update `roiposition` (will contain peak position for its first half and background position in its second half)**

each XY peak roi has got a pixel for bckground estimation, with the same Y value.



In [ ]:
# True to check step down relative position of shadowing of fluo or scattering signals
# (Advanced DAXM with fluorescence signal)
# DEFAULT is False
ADD_ROIS_FOR_DAXM_BACKGROUND = True

if d['scantype']=='daxm' and ADD_ROIS_FOR_DAXM_BACKGROUND:
    #imageindex = 1427

    print('imageindex',imageindex)
    fullpath = os.path.join(folder,f'{d['prefix']}%04d.{d['suffix']}'%imageindex)
    with fabio.open(fullpath) as img:
        imgdata = img.data
    imgdata[1336]

In [ ]:
if d['scantype']=='daxm' and ADD_ROIS_FOR_DAXM_BACKGROUND:
    def predictpixelforbackground(peakroiXY,imagedata2D, halfsize=9):
        """locate where background can be estimated,
        along an horizontal intensity cross section line"""
        x, y = peakroiXY
        
        poordetection = False
        insidecamera = True

        largestpixel = 2016 # sCMOS
        largestpixel = 2050 # EIGER
        
        if x<=halfsize+1 or x >= largestpixel-halfsize or y<=halfsize+1 or y >= largestpixel-halfsize:
            poordetection = True
            insidecamera = False
            Xbckg = 0
        else:
            xx = np.arange(int(x)-halfsize,int(x)+halfsize+1)
    
            IcrossX = imagedata2D[int(y)][xx[0]:xx[-1]+1]
    
            # locate bckg pixel
            pop, bins = np.histogram(IcrossX, bins=15)
            xbck = np.where(IcrossX<bins[1])
            xbckclose_idx = np.argsort(np.fabs(xbck[0]-halfsize))[0]
            Xbckg = xx[xbckclose_idx]
    
            amp = np.ptp(IcrossX)
            if amp < 100:
                poordetection = True
            mini = np.amin(IcrossX)
    
            Ibckg = IcrossX[xbckclose_idx]
            
            if Ibckg > mini+0.2*amp:
                poordetection = True
    
        return Xbckg, poordetection, insidecamera

In [ ]:
# GUI to set automatic parameters to find pixel on right or left horizontally for bckg

#if d['scantype']=='daxm' and ADD_ROIS_FOR_DAXM_BACKGROUND:
from ipywidgets import interact, interactive, fixed, interact_manual

peak_index = 300

peakroiXY = roiposition[peak_index]
x, y = peakroiXY

halfsize = 9

print(x,y)
assert x>10
assert x < 2010
xx = np.arange(int(x)-halfsize,int(x)+halfsize+1)

IcrossX = imgdata[int(y)][xx[0]:xx[-1]+1]
# print(xx, len(xx))
# print(IcrossX)
# locate bckg pixel
pop, bins = np.histogram(IcrossX, bins=15)
xbck = np.where(IcrossX<bins[1])
xbckclose_idx = np.argsort(np.fabs(xbck[0]-halfsize))[0]
Xbckg = xx[xbckclose_idx]

figl,axl = plt.subplots()
axl.plot(xx, IcrossX, '-o')
axl.grid()
axl.axvline(Xbckg, color='k')

def plotxprofile(peak_index=0, halfsize=9):
    axl.clear()
    
    peakroiXY = roiposition[peak_index]
    x,y = peakroiXY
    Xbckg, poordetection, insidecamera = predictpixelforbackground(peakroiXY,imgdata,halfsize)

    if insidecamera:
        xx = np.arange(int(x)-halfsize,int(x)+halfsize+1)
        IcrossX = imgdata[int(y)][xx[0]:xx[-1]+1]
        
        axl.plot(xx, IcrossX, '-o')
        axl.grid()
        if not poordetection:
            axl.axvline(Xbckg, color='k')
        axl.set_title('roiposition at x,y = %d, %d'%(int(x), int(y)))
    else:
        axl.plot(1000*np.ones(halfsize*2+1), '-o')
        axl.set_title('roiposition at x,y = %d, %d\n too close from border'%(int(x), int(y)))
interactive(plotxprofile, peak_index=(0,len(roiposition)-1),
            halfsize=(3,23))

#### build array of bckgposition

In [ ]:
if d['scantype']=='daxm' and ADD_ROIS_FOR_DAXM_BACKGROUND:

    bckgposition= np.zeros_like(roiposition)
    print(bckgposition.shape)
    nbgood = 0
    indexgood =  []
    for k, roixy in enumerate(roiposition):
        Ybckg=roixy[1]
        res =predictpixelforbackground(roixy,imgdata, halfsize=5)
        print(res)
        Xbckg,poordetection,incamera = res
        
        if incamera and not poordetection:
            print('*****>',[Xbckg, Ybckg])
            bckgposition[k]=[Xbckg, Ybckg]
            nbgood+=1
            indexgood.append(k)

In [ ]:
if d['scantype']=='daxm' and ADD_ROIS_FOR_DAXM_BACKGROUND:

    print('nb background pixel sensors found: %d  over %d rois in roiposition'%(nbgood, len(roiposition)))
    print('index of roi with bckg pixel sensor',indexgood)
    bckgposition

In [ ]:
if d['scantype']=='daxm' and ADD_ROIS_FOR_DAXM_BACKGROUND:
    len(bckgposition)

#### be careful when resetting `roiposition` to `roiposition`+`bckgposition`

In [ ]:
# TODO  collect intensity profile for bckgposition
if d['scantype']=='daxm' and ADD_ROIS_FOR_DAXM_BACKGROUND:

    roiposition = np.concatenate((roiposition, bckgposition),axis=0)
    print('Now `roiposition` should be twice longer ... and contains ROI locating at the baseline of each ROI')
    print(roiposition.shape)
    # forming fake intensities data for background pixel
    d['intensities'] = np.concatenate((d['intensities'],d['intensities']))
    print(d['intensities'].shape)

##  [OPTION 2] Read peak positions in .fit file

In [ ]:
fitfile = '/data/visitor/a321201/bm32/20250212/RAW_DATA/ech16_1_R2/ech16_1_R2_Z1_GOI_Zr_7_Core_daxm/scan0001/dat_img_0000_LT_0_fitnb_1.fit'

In [ ]:
graindata = IOLT.readfile_fit(fitfile)[4]
XYgrain = graindata[:,7:9]
intensities_fitfile = graindata[:,1]

# sorting by increasing Y for DAXM
sortedYindices = np.argsort(XYgrain[:,1])
XYgrain= np.take(XYgrain,sortedYindices, axis=0)
XYgrain

##  [OPTION 3] get all pixels in a roi

In [ ]:
# XcenterROI, YcenterROI = 886, 1462  # spot 1
# XcenterROI, YcenterROI = 1640, 1260  # spot 2
# XcenterROI, YcenterROI = 1546, 857   # spot 3

# fluo  spot 1
XcenterROI, YcenterROI = 802, 1492 
# # fluo  spot 2
# XcenterROI, YcenterROI = 1732, 1333 


hbx = 5  # halfboxsize // X pixel
hby = 19  # halfboxsize // Y pixel

#-------------------------------------
if hbx%2==1 and hby%2==1:
    yv,xv = np.meshgrid(np.arange(-hby,hby+1), np.arange(-hbx,hbx+1))
    
    #print(xv,yv)
    roipixels = np.array([xv, yv]).T.reshape(((2*hbx+1)*(2*hby+1),2))
    roipixels += np.array([XcenterROI, YcenterROI])

roipixels

##  [OPTION 4] get roi position from a built regular array of rois

### set `gridroi`  ROI position

In [ ]:
# quadbin1  :  maxnbpixels = 2000  # quad scmos binning 2X2  4M pixels
# imstarbin2 :  maxnbpixels = 3000    # scmos_9M binning hardware 2X2  9M pixels   gsense6060 imagestar165

# ROIs are in whole detector area 
gridroi = GT.buildgridroi(spacing=100,maxnbpixels=2060)

# ROIs are in the half lower part of detector area 
#gridroi = GT.buildgridroi_bottom(spacing=15,maxnbpixels=2060)
# set general parameter (which is filled  by peaksearch)
intensities_gridrois = None

gridroi, gridroi.shape

### [OPTION] plot markers if needed

In [ ]:
nbrois_grid = gridroi.shape[0]
try:
    maximage = np.amax(imgdata)
except NameError:
    imgdata = 1000*np.ones((2200, 2200))

if nbrois_grid<1000:
    fig,ax = plt.subplots()
    
    if imgdata is None:
        imgdata = 1000*np.ones((2000,2000))
    #ax.imshow(np.log10(imgdata), vmin = 3, vmax = 3.5, cmap=plt.cm.inferno)
    ax.imshow(imgdata, vmin = 1000, vmax =6000, cmap=plt.cm.OrRd)
    for pt in gridroi:
        ax.scatter(pt[0],pt[1],marker='+',color='r')
        ax.set_title('')

    plt.show()
else:
    GT.printyellow(f'nb of rois in the grid ({nbrois_grid}) is too large to be reasonnably plot')

# SET centers of ROI: setting `d['peaklist']`

SET one flag to True to choose the ROIs center location origin

(part of roi positions can also be selected)

In [ ]:
USE_PEAKSEARCH_LIST = False  # True for DAXM  False for 2D Map  (could include first half of ROI center peak and second half of neighbouring peaks
USE_GRIDROI_LIST = True  # False for DAXM  True for 2D Map
USE_RANDOM_POSITION_LIST = False
USE_ROIPIXELS = False
USE_FITFILE_PEAKLIST = False

# -------------------------------------------
lflag = [USE_PEAKSEARCH_LIST,USE_GRIDROI_LIST,USE_RANDOM_POSITION_LIST,USE_ROIPIXELS,USE_FITFILE_PEAKLIST]

if sum(lflag)==0:
    GT.printyellow('One "USE_###" flag at least should be True')
elif sum(lflag)>1:
    GT.printred('Please set only one "USE_###" flag to True')
#-------------------------------------------
#read pixel positions of peaks from key 'peaklistfile'
if 0:
    if 'peaklist' not in d or d['peaklist'] is None:
        if d['peaklistfile'] is not None:# default 
            iohdf5.getpeaklist(d)
        print(d['peaklist'])

        # use the variable gridroi
alreadyselected = 0
if USE_RANDOM_POSITION_LIST: # or input them: data = array of peaks position
    
    # RANDOM ROIS
    nbofrois = 300
    maxXYpixelvalue = 2000 # given by the dimension of the detector
    iohdf5.getpeaklist(d, data=np.random.randint(maxXYpixelvalue,size=(nbofrois,2)))
    alreadyselected += 1
if USE_GRIDROI_LIST:
    
    # regular spaced ROIS
    iohdf5.getpeaklist(d, data=gridroi)
    alreadyselected += 1
    intensities = None  # patch
if USE_PEAKSEARCH_LIST:
    # ROIS from peak peaksearch above
    iohdf5.getpeaklist(d, data=roiposition)
    alreadyselected += 1

if USE_ROIPIXELS:
    
    iohdf5.getpeaklist(d, data=roipixels)

if USE_FITFILE_PEAKLIST:
    
    iohdf5.getpeaklist(d, data=XYgrain)
    alreadyselected += 1
    d['intensities']=intensities_fitfile

if alreadyselected > 1:
    GT.printred(f'You must select a single type of list! Check there is only one True value at the top of the cell!')
if 0:#  ---
    iohdf5.sortpeaklist(d)
    print(d['peaklist'])
    
if 0:  # select the first number of rois 
    selectnb=20
    d['peaklist']=d['peaklist'][:selectnb+1]
    
GT.printgreen(f"\nnb of rois : {len(d['peaklist'])}")
print('current peaklist', d['peaklist'])

if 1:  # reset manually some roi centers
    d['peaklist'][0]=[990,840] 
    #d['peaklist'][1]=[582,840]
    
if intensities is not None:
    d['intensities']=intensities
    if ADD_ROIS_FOR_DAXM_BACKGROUND:
        d['intensities'] = np.concatenate((d['intensities'],d['intensities']))
    print('values of last peaksearch peaks intensities are in d["intensities"]')

In [ ]:
d['peaklist']

## final dict for exp. parameters (folder and rois)

In [ ]:
np.amax(d['peaklist'], axis=0)

# COLLECT Mosaic Multiprocessing 

each 2D small part around the same single pixel position `d['roicenter']`is taken over all images and rearranged to form form a mosaic of local 2D signal

In [ ]:
# rapid Manual entry
# mapdimensions = (21,86)  # fast  // xech and Xpixel,  slow axis  // yech and -Ypixel
if d:
    #d['roicenter']=d['peaklist'][3500]
    d['roicenter']=[1410,1595]
    d['roicenter']=[1554,786]

    d['roicenter']=[532,1982]
    d['roicenter'] =  [1590, 1560]
    d['roicenter'] =  [1211, 1928]
    d['roicenter'] =  [67, 1307]
    d['roicenter'] =  [1154, 1220]
    d['roicenter'] =  [1000, 848]  # RT
    d['roicenter'] =  [980, 800]   # 1250C
    #d['roicenter'] = clicked_pixel

In [ ]:
if __name__=='__main__':
    
    collector = 'mosaic'
    nbcpus= 64 # > 1 !!
    
    # for roimax or roiXYmax
    boxsize_X=25  # along pixel X
    boxsize_Y=25  # along pixel Y
    
    # images to be handled
    listindices = None#  None means take indices in d[['listindices']]
    nbcompletelines = None  #10  # None all lines when scan is completed
    
    #---------------------Multiprocessing-----------------------------------------------
    dictparam=d

    maxnbcpus = cpu_count()
    if nbcpus is None:
        nbcpus = maxnbcpus
    else:
        nbcpus = max(2,min(nbcpus,maxnbcpus))
    print('nbcpus', nbcpus)
    
    roicenter=dictparam.get('roicenter', None)
    prefix=dictparam.get('prefix', None)
    folder=dictparam.get('folder', None)
    CCDLabel=dictparam.get('CCDLabel', None)

    if roicenter is None:
        GT.printred(f"d['roicenter'] is not set to extract pixel intensities around it")
    else:
        xroi, yroi = d['roicenter']
        maxdimension = 2015
        if xroi-boxsize_X<0 or xroi+boxsize_X>maxdimension or yroi-boxsize_Y<0 or yroi+boxsize_Y>maxdimension:
            GT.printred(f"d['roicenter'] {d['roicenter']} is too close from detector frame border for the boxsize : {(boxsize_X, boxsize_Y)}")
    
    if listindices is None:
        listindices=dictparam['listindices']
        if nbcompletelines is not None:
            listindices=dictparam['listindices'][:dictparam['nbimagesperline']*nbcompletelines]
            print(dictparam['mapdimensions'][0],nbcompletelines)

            # warning  : reset d values !!
            d['mapdimensions']=(dictparam['mapdimensions'][0],nbcompletelines)
            d['listindices'] =listindices
            
    if not isinstance(folder, list):  # iteration over images on the same folder
        nbimages = len(listindices)
        listfolders = itertools.repeat(folder)
        
    else: # iteration over a list of folders
        print('Multiple folders screened')
        nbimages = len(folder)
        listfolders = folder
        listindices = itertools.repeat(dictparam['listindices'])
    
    #----------------------------------
    
    t00 = time.time()
    print(f'using {collector} as collector: {nbimages} images, {nbcpus} cpu(s)')
    
    args_mosaic = zip(listindices,
                   itertools.repeat(roicenter),
                   itertools.repeat(prefix),
                   listfolders,
                  itertools.repeat(boxsize_X),
                  itertools.repeat(boxsize_Y),
                   itertools.repeat(CCDLabel),
                   )
    
    #print('args_pixelvalue',[elem for elem in args_pixelvalue])
    
    with multiprocessing.Pool(nbcpus) as pool:
            
        allresults = pool.starmap(collectroiarray_singlefile,
                                      tqdm(args_mosaic, total=nbimages,
                                      desc='sum values progress bar:'), chunksize=1)

    allresults = np.array(allresults)

    elapsedtime=time.time()-t00
    print(f'total time is {elapsedtime:.3f} sec for {nbimages} images and {nbcpus} cpu(s)')

    children = active_children()
    print(f'Active children: {len(children)}')

    d['allresults']=allresults
    print('Results are in allresults or allresults key...\n\n!!! ----- Collection done ;-) ----!!')
    
print('allresults.shape = (nbimages x boxsize x boxsize)', allresults.shape)

## rearranging results for 2D mesh scan (map)

In [ ]:
# plot mosaic from 2D map
if d['scantype']=='map':
    # processing and rearranging collected ROIs imagelets
    dimfast, dimslow = d['mapdimensions']
    print('axis dimensions: dimslow, dimfast',dimslow, dimfast)
    mosaic = np.zeros((dimslow, dimfast, 2*boxsize_Y+1, 2*boxsize_X+1))
    print('mosaic.shape', mosaic.shape)
    dict_map_imageindex ={}

    print(d['listindices'])
    
    sm = mosaic.shape
    bigimage = np.zeros((sm[0]*sm[2],sm[1]*sm[3]))
    
    
    if dimfast > 0:
        for map_imageindex, absolute_imageindex in enumerate(d['listindices']):
            
            imap, jmap = map_imageindex // dimfast, map_imageindex % dimfast
    
            dict_map_imageindex[map_imageindex] = [absolute_imageindex,
                                                    map_imageindex,
                                                    imap, jmap]
            
            raw = allresults[map_imageindex,:,:]
            
            #datcrop = np.flipud(raw).T
            datcrop = raw
            
            mosaic[imap,jmap] = datcrop #datcrop.T #np.flipud(datcrop).T
            # for 2D map
            bigimage[imap*sm[2]:(imap+1)*sm[2], jmap*sm[3]:(jmap+1)*sm[3]] = np.flipud(datcrop)
    
    if 1:
        mosaictranspose = mosaic.transpose((0, 3, 1, 2))
        mosaicflat = mosaictranspose.reshape((dimfast * (2 * boxsize_X + 1), dimslow * (2 * boxsize_Y + 1)))
    
    print('mosaictranspose.shape',mosaictranspose.shape)
    print('mosaicflat.shape',mosaicflat.shape)
    print('pixel ', d['roicenter'])
    #dat_dims = (nb_lines * (2 * boxsize_Y + 1), nb_col * (2 * boxsize_col + 1))

## rearranging mosaic plot for 1D scan (Daxm)

In [ ]:
# plot mosaic for 1D or DAXM scan
if d['scantype']=='daxm':

    xdim_plot = 20
    print('nb of imagelets in horizontal direction (// pixel x)',xdim_plot)
    
    dimfast = d['mapdimensions'][0]
    print("dimfast",dimfast)
    print(mosaic.shape)
    print(f"find a divisor of {dimfast} to set 'xdim_plot' as the number of imagelets per line in the final mosaicimage")
    
    ydim_plot = dimfast//xdim_plot
    #print('xdim_plot, ydim_plot', xdim_plot, ydim_plot)
    sm = mosaic.shape
    #print('sm',sm)
    mosaicimage = np.zeros((ydim_plot*sm[2],xdim_plot*sm[3]))
    
    n1,n2 =  mosaicimage.shape
    
    #print('n1,n2',n1,n2)
    
    for map_imageindex, absolute_imageindex in enumerate(d['listindices']):
            
        imap, jmap = map_imageindex // xdim_plot, map_imageindex % xdim_plot
    
        # dict_map_imageindex[map_imageindex] = [absolute_imageindex,
        #                                         map_imageindex,
        #                                         imap, jmap]
    
        datcrop = allresults[map_imageindex,:,:]
        #print('map_imageindex, imap, jmap',map_imageindex, imap, jmap)
        #print('datcrop.shape', datcrop.shape)
        #mosaic[imap,jmap] = datcrop
        # for 2D map
        mosaicimage[imap*sm[2]:(imap+1)*sm[2], jmap*sm[3]:(jmap+1)*sm[3]] = np.flipud(datcrop)

## [OPTION] picking results and/or saving figure

In [ ]:
# results can be written in user-defined folder or in 'PROCESSED_DATA'
if not isinstance(folder, list):   # single folder
    #genfolder = folder
    genfolder = bf.createmirrorfolder(folder)
else:
    genfolder = os.path.join(os.path.split(ExperimentFolder)[0],'PROCESSED_DATA')

# in projects/mapgrainxl  there is no 'PROCESSED_DATA'
genfolder = folder

if os.path.isdir(genfolder):
    GT.printgreen(f'\nAnalysis results will be written is "genfolder":\n {genfolder}')

In [ ]:
# use genfolder to write output  results
import pickle

if 1: # SAVE
    dictresults=copy.copy(d)
    dictresults['allresults']=allresults
    dictresults['mosaic']=mosaic
    dictresults['bigimage']=bigimage
    with open(os.path.join(genfolder,'%s_roicenter_X_%d_Y_%d.pickle'%(collector,int(roicenter[0]),int(roicenter[1]))), 'wb') as f:
        pickle.dump(dictresults, f)

if 0: #LOAD
    if input('Are you sure to load a previous file and overwrite dictresults?') in ('y','yes','Y','YES','o','O'):
        
        folder = '/data/visitor/ihma346/bm32/20230304/SiCnuit/SiCnuit_bulle2/scan0001'
        collector = 'fitpeakXY'
        #folder = d['folder']

        #with open(os.path.join(folder,'%s_20counters.pickle'%collector), 'rb') as f:
        with open(os.path.join(genfolder,'fitpeakXY_20counters'+'.pickle'), 'rb') as f:
            dictresults=pickle.load(f)

        d=dictresults
        allresults = d['allresults']
        mosaic = d['mosaic']
        bigimage = d['bigimage']


## [OPTION] visualisation of imagelet

In [ ]:
# just to test a single imagelet
imageindex = 0
vmax= None

print(imageindex//dimfast,imageindex%dimfast)
#-------------
if imageindex not in d['listindices']:
    raise ValueError('imageindex is not in the data !\n Try an other "imageindex" value')
roicenter = d['roicenter']
figt, axt = plt.subplots()
data2D  = mosaic[imageindex//dimfast,imageindex%dimfast]
if vmax is not None: vmax = max(np.amin(data2D)+1, vmax)
axt.imshow(data2D, origin='upper', vmax=vmax)
axt.set_title(f'roicenter at ({roicenter[0]}, {roicenter[1]})')
axt.set_xlabel('Pixel X')
axt.set_ylabel('Pixel Y')

print('data2D.shape', data2D.shape)
print('initial roi size (X, Y)',2*boxsize_X+1, 2*boxsize_Y+1)

In [ ]:
17000/151

In [ ]:
print(bigimage.shape) # "Ypixel * slow", Xpixel * fast"
mosaic.shape # slow , fast , Ypixel, Xpixel

In [ ]:
# find the hottest imagelet   indices in mosaic darray
fullboxY = mosaic.shape[2]
fastdim = mosaic.shape[1]
fullboxX = mosaic.shape[3]

pixel_slowdirection, pixel_fastdirection = np.argmax(bigimage)//(fullboxX*fastdim), np.argmax(bigimage)%(fullboxX*fastdim)
print(pixel_slowdirection, pixel_fastdirection)
i_slow_max = pixel_slowdirection//fullboxY
i_fast_max = pixel_fastdirection//fullboxX

i_slow_max, i_fast_max

## GUI browser imagelet

In [ ]:
imageindex0 = 100
dimfast = d['mapdimensions'][0]
roicenter = d['roicenter']
i_x0, i_y0 = imageindex0//dimfast,imageindex0%dimfast  # slow, fast

print(i_x0, i_y0)

figt, axt = plt.subplots()
axt.imshow(mosaic[i_x0, i_y0], origin='upper')

slider_ix = widgets.IntSlider(value=i_x0, min=0, max=mosaic.shape[0]-1, step=1)
slider_iy = widgets.IntSlider(value=i_y0, min=0, max=mosaic.shape[1]-1, step=1)
slider_vmin = widgets.IntSlider(value=0, min=-100, max=0, step=1)
slider_vmax = widgets.IntSlider(value=1000, min=10, max=60000, step=1)
output = widgets.Output()

def handle_slider_change(change):
    with output:
        output.clear_output()
        print(f"The new slider value is: {change.new}")

slider_ix.observe(handle_slider_change, 'value')
slider_iy.observe(handle_slider_change, 'value')

widgets.VBox([slider_ix,slider_iy, slider_vmin, slider_vmax, output])

def plotroi(i_slow, i_fast, vmin=0, vmax=8000, scale='linear'):
    ymin, ymax = axt.get_ylim()
    xmin, xmax = axt.get_xlim()

    i_x, i_y = i_slow, i_fast

    axt.clear()
    
    axt.set_xlim(xmin, xmax)
    axt.set_ylim(ymin, ymax)
    
    
    ymin, ymax, xmin, xmax = int(ymin),int(ymax),int(xmin),int(xmax)
    if ymin>ymax:
        yminc = ymin
        ymin = ymax
        ymax = yminc

    imageindex = i_x*dimfast+i_y

    if scale == 'linear':
        dplot = mosaic[i_x, i_y]
        vminplot = vmin
        vmaxplot = vmax
    elif scale == 'log':
        dplot = np.log10(mosaic[i_x, i_y])
        vminplot = np.log10(vmin)
        vmaxplot = np.log10(vmax)
            
    axt.imshow(dplot, origin='upper', vmin=vminplot, vmax=vmaxplot)
    axt.set_xlabel('pixel X')
    axt.set_ylabel('pixel Y')
    tt = f'roicenter at pixel ({roicenter[0]}, {roicenter[1]}) '
    tt += f'image index {imageindex}'
    axt.set_title(tt)

    print('max:', np.amax(dplot))
    print('min:', np.amin(dplot))

widgets.interact(plotroi, i_slow=slider_ix,i_fast=slider_iy,
                 vmin = slider_vmin, vmax = slider_vmax, scale=['linear', 'log'])
print(f'i_slow // {d["slowaxis"]}   i_fast // {d["fastaxis"]}')

## visualisation mosaic for 2D scan (map)

In [ ]:
mosaic.shape

In [ ]:
# plot mosaic from 2D map
if d['scantype']=='map':
    vmax =  1000 # None

    #bigimage.shape # "Ypixel * slow", Xpixel * fast"
    #mosaic.shape # slow , fast , Ypixel, Xpixel
    def format_coord(x, y):
        col = int(x)
        row = int(y)
        if col >= 0 and col < bigimage.shape[1] and row >= 0 and row < bigimage.shape[0]:
            #cnt_idx = fig.gca()
            i, j = col//mosaic.shape[3], row//mosaic.shape[2]
            img_idx = 0+ mosaic.shape[1]*j+i
            return "x=%1.4f, y=%1.4f, imageid=%d" % (x, y,img_idx)
        else:
            return "x=%1.4f, y=%1.4f" % (x, y)


    
    
    print(bigimage.shape, )
    figmosaic, axmosaic = plt.subplots(figsize=(8,8))
    axmosaic.format_coord = format_coord
    axmosaic.imshow(bigimage, origin='lower', vmax=1000, cmap='GnBu')
    axmosaic.set_xlabel(f"fastaxis {d['fastaxis']} // pixelX")  # ok for fdscan2d
    axmosaic.set_ylabel(f"slowaxis {d['slowaxis']} // pixelY")
    title = ''
    title += '%s\n'%d['imagefolder']
    title += 'roicenter X,Y = (%d,%d)'%(d['roicenter'][0],d['roicenter'][1])
    axmosaic.set_title(title)
    axmosaic.format_coord = format_coord

## visualisation mosaic from 1D scan (DAXM)

In [ ]:
if d['scantype']=='daxm':
    #print(mosaicimage.shape)
    figmosaic, axmosaic = plt.subplots(figsize=(10,10))
    axmosaic.imshow(mosaicimage, origin='lower')
    axmosaic.set_xlabel(d['fastaxis'])
    axmosaic.set_ylabel(d['slowaxis'])
    title = ''
    title += '%s\n'%d['imagefolder']
    title += 'roicenter X,Y = (%d,%d)'%(d['roicenter'][0],d['roicenter'][1])
    axmosaic.set_title(title)

In [ ]:
# saving figure
outputfolder = genfolder
#outputfolder =  d['folder']
 
# ----  output filename business  ------------------
roicenter = d['roicenter']
nameout = d.get('title','')
tt =time.asctime().strip()
tt = tt.replace(' ','_')
tt=tt.replace(':','_')
coltime="%s_roicenter_X_%d_Y_%d_%s"%(collector,int(roicenter[0]),int(roicenter[1]),tt)

if nameout != '':
    nameout+='_'+coltime
else:
    nameout=coltime
plt.savefig(os.path.join(outputfolder,nameout+'.png'))
print('output folder --->', outputfolder)
print('figure filename --->',nameout+'.png')

#   COLLECT ROI counters multiprocessing  (including grqain imaging)

(setting collector, boxsizes)

In [ ]:
# some helping command
# !ls -alp {d['folder']}
# d['peaklist']
# d['listindices']
# d['fullcommand']
# d['CCDLabel']

In [ ]:
# if 0: # in case of trouble with multiprocessing
#     print(pool)
#     del pool

In [ ]:
#!ls {d['folder']}

In [ ]:
print('map dimensions:',d['mapdimensions'])
print('folder of Laue images:',d['folder'])
imageindexmax_in_folder = GT.get_largest_index_in_folder(d['folder'], filename_prefix=d['prefix'],
                                                         filename_suffix=d['suffix'])
print('largest image index in this folder:',imageindexmax_in_folder)

In [ ]:
### tests of elementary function
if 0:
    folder=d['folder']
    #folder=d481['folder']
    peaklist=d.get('peaklist', np.array([[1000,1000],[1200,1200]]))
    prefix=d.get('prefix', d['prefix'])
    CCDLabel=d.get('CCDLabel', 'sCMOS')

    print('folder', folder)
    print('CCDLabel',CCDLabel)


    # for _i in range(0,10):
    #     print('_i', _i)
    #     filename = prefix+'%04d'%_i+'.tif'
    #     val = IOimage.pixelvalat(os.path.join(folder, filename),xy=np.array([[1000.,1000.],[1200,1200]]))
    #     print('val', val)
if 0:
    for _i in range(0,4):
        print('_i', _i)
        filename = prefix+'%04d'%_i+f".{d['suffix']}"
        print('filename',filename)
        val = collectpixelvalue_singlefile(_i, np.array([[2050,2100],[1200,1200]]), prefix, folder, CCDLabel=CCDLabel)
        print('val', val)
if 0:
    for _i in range(0,12):
        print('_i', _i)
        peaklist_test = pk
        peaklist_test = peaklist
        
        filename = prefix+'%04d'%_i+f".{d['suffix']}"
        val = collectroismax_singlefile(_i, peaklist_test[:1530],prefix, folder,
                                        boxsize_row=1,boxsize_line=1,CCDLabel=CCDLabel)
        print('val', val)

In [ ]:
# checking if we have more images than expected
if imageindexmax_in_folder != max(d['listindices']):
    GT.printyellow(f'\nBe careful, this folder created by a scan may contain unexpected additional images')
    GT.printyellow(f"\nPlease use d['listindices'] to properly read the right number of images")
else:
    GT.printgreen(f'\nThis folder containing images created by a scan contains {max(d["listindices"])+1} images as expected from dict `d`')

In [ ]:
if 'intensities' in d:
    print( d['intensities'].shape)
nbrois = d['peaklist'].shape[0]
print(f'There are {nbrois} ROI centers (or peaklist) selected on detector {d['CCDLabel']}')

In [ ]:
maxdistancefromborder = 60 # will be the half boxsize used in the next cell!
if 1:
    framedim = DictLT.dict_CCD[d['CCDLabel']][0]
    print('framedim  (pixelY, pixelX)',framedim)
    
    pk0 = d['peaklist']

    print('maximum values (X, Y):',np.amax(d['peaklist'], axis=0))

    d['initialpeaklist'] = copy.copy(pk0)
    
    print('Checking and keeping peaks far from detector frame border')
    print('before',len(pk0))
    pk, totakeindices = RMCCD.filter_points_far_from_border(pk0, maxdistancefromborder,
                                                            maxdistancefromborder, d['CCDLabel'])
    print('after', len(pk))
    print(f"updating peaks list of d['peaklist']")
    d['peaklist'] = np.ndarray.astype(pk, np.int32)

    print('maximum values (X, Y):',np.amax(d['peaklist'], axis=0))
    
    if 'intensities' in d:
        if d['intensities'] is not None:
            d['intensities']= np.take(d['intensities'],totakeindices , axis=0)
    minX, maxX = np.min(d['peaklist'][:,0]),np.amax(d['peaklist'][:,0])
    minY, maxY = np.min(d['peaklist'][:,1]),np.amax(d['peaklist'][:,1])
    print(f'X ROIs center are comprised between: {minX}, {maxX}')
    print(f'Y ROIs center are comprised between: {minY}, {maxY}')


In [ ]:
np.amax(d['peaklist'], axis=0)

In [ ]:
if __name__=='__main__':
    import itertools

    # True will deduce the last completed lines and modify map dimensions accordingly, 
    # False: consider the scan completed (all images stored) or one can define our own listindices
    ONLINE=False  # True  
    # -----------------INPUT ---------------------------------------
    # advised for 2D map
    #collector = 'roimax','roiXYmax' 'fitpeakXY' 'XYcenterofmass'
    # advised for daxm
    #collector = 'roimax', 'pixelval' 'sum'
    collector = 'roimax'

    nbcpus= 64 # > 1 !!

    computerrorbars = False

    # HALF boxsize (odd value to have centered box on central pixel)
    boxsize_X=50 # along pixel X  
    boxsize_Y=50  # along pixel Y
    
    if  d['scantype'] == 'daxm':
        collector = 'roimax'
        boxsize_X=3  # 1
        boxsize_Y=1
        
    elif collector == 'pixelval':
        boxsize_X=None  # meaningless
        boxsize_Y=None

    
    dictparam=d

    folder=dictparam.get('folder', None)
    
    # images to be handled
    listindices = None#  None means take indices in d[['listindices']]
    nbcompletelines = None #10  # None all lines when scan is completed
    
    if ONLINE:
        nbcompletelines = GT.get_largest_index_in_folder(d['folder'], filename_prefix=d['prefix'],
                                                         filename_suffix=d['suffix'])//d['mapdimensions'][0]

    
    #---------------------Multiprocessing -----------------------------------------------
    if maxX + boxsize_X>framedim[1] or minX - boxsize_X<0:
        raise ValueError('boxsize_X is too large')
    if maxY + boxsize_Y>framedim[0] or minY - boxsize_Y<0:
        raise ValueError('boxsize_Y is too large')
    
    if collector == 'fitpeakXY':
        computerrorbars = False # True
        
    maxnbcpus = cpu_count()
    if nbcpus is None:
        nbcpus = maxnbcpus
    else:
        nbcpus = max(2,min(nbcpus,maxnbcpus))
    print('nbcpus', nbcpus)
    
    peaklist=dictparam.get('peaklist', None)
    prefix=dictparam.get('prefix', None)
    
    CCDLabel=dictparam.get('CCDLabel', 'sCMOS')
    
    nbpeaks = len(dictparam['peaklist'])
    
    if listindices is None:
        listindices=dictparam['listindices']
        if nbcompletelines is not None:
            listindices=dictparam['listindices'][:d['nbimagesperline']*nbcompletelines]
            
    if not isinstance(folder, list):  # iteration over images on the same folder
        nbimages = len(listindices)
        listfolders = itertools.repeat(folder)
        
    else: # iteration over a list of folder
        print('Multiple folders screened')
        nbimages = len(folder)
        listfolders = folder
        listindices = itertools.repeat(dictparam['listindices'])
    
    
    #----------------------------------
    
    t00 = time.time()
    print(f'using {collector} as collector: {nbimages} images, {nbpeaks} roi positions, {nbcpus} cpu(s)')
    
    args_pixelvalue = zip(listindices,
                   itertools.repeat(peaklist),
                   itertools.repeat(prefix),
                   listfolders,
                   itertools.repeat(CCDLabel),
                   )

    args_roimax = zip(listindices,
                   itertools.repeat(peaklist),
                   itertools.repeat(prefix),
                   listfolders,
                  itertools.repeat(boxsize_X),
                  itertools.repeat(boxsize_Y),
                   itertools.repeat(CCDLabel),
                   )

    args_roimax_errorbars = zip(listindices,
                   itertools.repeat(peaklist),
                   itertools.repeat(prefix),
                   listfolders,
                  itertools.repeat(boxsize_X),
                  itertools.repeat(boxsize_Y),
                   itertools.repeat(CCDLabel),
                    itertools.repeat(computerrorbars)
                   )
    
    #print('args_pixelvalue',[elem for elem in args_pixelvalue])
    
    with multiprocessing.Pool(nbcpus) as pool:
    
        assert len(peaklist)>0

        if collector == 'pixelval': # 1 scalar /roi
            print("collectpixelvalue_singlefile")
            allresults = pool.starmap(collectpixelvalue_singlefile,
                                      tqdm(args_pixelvalue, total=nbimages,
                                      desc='pixel values progress bar:'))
            
        elif collector == 'sum': # 1 scalar /roi
            
            allresults = pool.starmap(collectroissum_singlefile,
                                      tqdm(args_roimax, total=nbimages,
                                      desc='sum values progress bar:'), chunksize=1)
        elif collector == 'roimax':

            allresults = pool.starmap(collectroismax_singlefile,
                                      tqdm(args_roimax, total=nbimages,
                                      desc='max values progress bar:'), chunksize=1)
            #collectroismax_singlefile.__defaults__=(peaklist,prefix,folder,boxsize_X,boxsize_Y,CCDLabel)
            #allresults = pool.map(collectroismax_singlefile, tqdm(listindices))
            
        elif collector == 'roiXYmax': #  2 values/roi

            allresults = pool.starmap(collectroisXYmax_singlefile,
                                      tqdm(args_roimax, total=nbimages,
                                      desc='pixel positions progress bar:'))

        elif collector == 'fitpeakXY':  # 7 values/roi
            allresults = pool.starmap(collectroisfitpeak_singlefile,
                                      tqdm(args_roimax_errorbars, total=nbimages,
                                      desc='fitted positions progress bar:'), chunksize=1)
        elif collector == 'XYcenterofmass': #  2 values/roi
            allresults = pool.starmap(collectroisXYcenterofmass_singlefile,
                                      tqdm(args_roimax, total=nbimages,
                                      desc='pixel positions progress bar:'))

    allresults = np.array(allresults)

    elapsedtime=time.time()-t00
    print(f'total time is {elapsedtime:.3f} sec for {nbimages} images, {nbpeaks} roi positions and {nbcpus} cpu(s)')

    children = active_children()
    print(f'Active children: {len(children)}')

    d['allresults']=allresults
    GT.printgreen('Results are in allresults or allresults key...\n\n!!! ----- Collection done ;-) ----!!')


In [ ]:
allresults.shape

## pickling results 

In [ ]:
ExperimentFolder, folder

### create mirror folder in PROCESSED_DATA and GALLERY

In [ ]:
folder= d['folder']

print('raw data folder (where images are saved from a given scan)', folder)
#---------------------------------------------
# creating Folder for processed data
if not isinstance(folder, list):   # single folder
    # write in PROCESSED_DATA
    genfolder, galleryfolder = bf.createmirrorfolder(folder)

elif 'projects' in ExperimentFolder:
    # in projects/mapgrainxl  there is no 'PROCESSED_DATA'
    genfolder = folder
    galleryfolder = os.path.join(os.path.split(ExperimentFolder)[0],'GALLERY')
    
else:
    genfolder = os.path.join(os.path.split(ExperimentFolder)[0],'PROCESSED_DATA')
    galleryfolder = os.path.join(os.path.split(ExperimentFolder)[0],'GALLERY')

os.makedirs(genfolder, exist_ok=True)
os.makedirs(galleryfolder, exist_ok=True)

print('analysis results will be written is "genfolder":\n',genfolder)
print('images results will be written is "galleryfolder":\n',galleryfolder)

In [ ]:
#---------------------------------------------
# use genfolder to write output  results
import pickle

if 1: # SAVE

    nbcounters = len(d['peaklist'])
    dictresults=copy.copy(d)
    dictresults['allresults']=allresults

    picklefile = os.path.join(genfolder,'%s_%dcounters.pickle'%(collector,nbcounters))

    with open(picklefile, 'wb') as f:
        pickle.dump(dictresults, f)

    GT.printgreen(f'picklefile saved in {picklefile}')

if 0: #LOAD
    if input('Are you sure to load a previous file and overwrite dictresults?') in ('y','yes','Y','YES','o','O'):
        
        folder = '/data/visitor/ihma346/bm32/20230304/SiCnuit/SiCnuit_bulle2/scan0001'
        collector = 'fitpeakXY'
        #folder = d['folder']

        #fullpathpickle = os.path.join(genfolder,'%s_20counters.pickle'%collector)
        fullpathpickle = '/data/visitor/blc17179/bm32/20260617/PROCESSED_DATA/BaTiO3/BaTiO3_map2D/scan0002/roimax_740counters.pickle'
        with open(fullpathpickle, 'rb') as f:
            dictresults=pickle.load(f)
        allresults = dictresults['allresults']
        d=dictresults
        collector= d['collector']

In [ ]:
d

## Arrange collected data. SET SET scalar X or Y component for `collector` = 'roiXYmax' or 'fitpeakXY'

BUILD `tr` array from collected `allresults`

In [ ]:
#peak component selection  (only if 'roiXYmax' or 'fitpeakXY' collected data)


if collector in ('fitpeakXY','roiXYmax', 'test'):
    peakposcomponent = input('Which pixel coordinate do you choose ? (X, Y)')
    # peakposcomponent='X'
    # peakposcomponent='Y'
    print(f'peakposcomponent is {peakposcomponent}')

print('(nb images, nb rois, nb vals / roi)',allresults.shape)
print('collector:',collector)

if collector in ('sum','roimax','pixelval',):
    tr = np.transpose(allresults)
    
    collectortitle=collector #just for visualisation
    
    spatialcoord = ''
    
elif collector in ('fitpeakXY',):
    # bke, amp, xfit, yfit, std1, std2, orientangle,  [errorbars] ?
    bkg  = allresults[:,:,0]
    amp = allresults[:,:,1]
    xfit, yfit = np.transpose(allresults[:,:,2:4])
    
    spatialcoord = peakposcomponent
    
    if peakposcomponent == 'X':
        tr = xfit
    if peakposcomponent == 'Y':
        tr = yfit
    collectortitle=collector #just for visualisation
else:
    tr_x = np.transpose(allresults[:,:,0])
    tr_y = np.transpose(allresults[:,:,1])
    
    spatialcoord = peakposcomponent
    
    # selecting X value
    if peakposcomponent=='X':
        tr = tr_x
    else: # selecting Y value
        tr = tr_y
    collectortitle=collector+'_%s'%peakposcomponent #just for visualisation

print('nb rois:',tr.shape[0])
print('nb images:',tr.shape[1])
print('map dimensions:',d['mapdimensions'])
print('map nb of dimensions:',len(d['mapdimensions']))
print('Chosen scalar to map in 2D:', collectortitle)
print('scan type:', d['scantype'])
if d['scantype']=='map':
    GT.printgreen('Your data are 2D, so use rather section VISUALIZE 2D case')
elif d['scantype']=='daxm':
    GT.printgreen('Your data are 1D, so use rather section VISUALIZE 1D case')


# VISUALIZE: 2D case (amesh dmesh fdmap)

this section is useful only if d['scantype']=='map'

In [ ]:
# Advice
if not d['scantype']=='map':
    print(f'2D visualisation is not appropriate for your scan since d["scantype"]="{d["scantype"]}" is not "map".\nGo better to the next section!')

### some helpers functions

In [ ]:
def getxechyech(imageindex,dictparam):
    fullcommand = dictparam['fullcommand']
    print('fullcommand :',fullcommand)
    assert imageindex>0
    
    if fullcommand.startswith('amesh'):
        _, fastmotor, fm_min, fm_max, fm_nbsteps, slowmotor, sm_min, sm_max, sm_nbsteps, expo = fullcommand.split()
    elif fullcommand.startswith('fscan2d'):
        _, slowmotor, sm_min, sm_stepsize, sm_dimension, fastmotor, fm_min, fm_stepsize, fm_dimension, _, expo = fullcommand.split()
        fm_nbsteps = int(fm_dimension) -1
        sm_nbsteps = int(sm_dimension) -1
        fm_max = float(fm_min) + fm_nbsteps*float(fm_stepsize)
        sm_max = float(sm_min) + sm_nbsteps*float(sm_stepsize)
        
    dim = (int(fm_nbsteps)+1, int(sm_nbsteps)+1)
    if imageindex>dim[0]*dim[1]:
        GT.printred('imageindex too large!')
        return None
    fm_step = (float(fm_max)-float(fm_min))/int(fm_nbsteps)
    sm_step = (float(sm_max)-float(sm_min))/int(sm_nbsteps)
    fmname = fastmotor
    smname = slowmotor
    
    index_row, index_col = imageindex//dim[0], imageindex%dim[0]
    fm_val = float(fm_min) + index_col*float(fm_step)
    sm_val = float(sm_min) + index_row*float(sm_step)
    return {fmname: fm_val, smname:sm_val}

def convertindex2Dto1D(fastindex, slowindex, mapdimensions:'(nfast, nslow)'=None):
    nf, _ = mapdimensions
    return slowindex*nf+fastindex

def convertindex1Dto2D(index, mapdimensions:'(nfast, nslow)'=None):
    nf, _ = mapdimensions
    fastindex = index%nf
    slowindex = index//nf
    return fastindex,slowindex

getxechyech(55,d)

### multiple plot of map obtained from several roi counters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Checkbox, Output
from IPython.display import display

# --- Select counters ---
list_idx_counters = range(24, 24 + 23)  # Example: 23 counters starting from 24
list_idx_counters = range(3788, 3788 + 15)  # Example: 23 counters starting from 24

autocontrast = True  # Default: autocontrast enabled
nbcols = 4  # Number of columns for subplots

# --- Validate counters ---
nbcounters = len(list_idx_counters)
print(f"Number of counters: {nbcounters}")
print(f"Highest selected index: {max(list_idx_counters)}")

if max(list_idx_counters) > len(tr):
    raise ValueError(f"Maximum counter index is {len(tr) - 1}")

# --- Subplot layout ---
nbimagesperline = d['nbimagesperline']
nrows = (nbcounters + nbcols - 1) // nbcols  # Ceiling division

# --- Dynamic figure size ---
fastdim, slowdim = d['mapdimensions']
subplot_width_inches = slowdim / 100  
subplot_height_inches = fastdim / 100  
fig_width = nbcols * subplot_width_inches
fig_height = nrows * subplot_height_inches + 1.5  # Extra space for widgets

# --- Create figure and subplots ---
fig2, axs = plt.subplots(
    nrows=nrows,
    ncols=nbcols,
    sharex=True,
    sharey=True,
    figsize=(fig_width, fig_height)
)

# --- Flatten axes for iteration ---
axs_flat = axs.flatten() if nrows > 1 else [axs]

# --- Missing plot placeholder ---
if d['fastaxis'] in ('xech', 'xps'):
    missingplot = np.zeros_like(tr[0]).reshape((-1, nbimagesperline))
else:  # 'yech' or 'yps'
    missingplot = np.zeros_like(tr[0]).reshape((-1, nbimagesperline)).T

# --- Coordinate formatting ---
def format_coord(x, y):
    fastdim, slowdim = d['mapdimensions']
    col, row = int(x + 0.5), int(y + 0.5)

    if d['fastaxis'] in ('xech', 'xps'):
        if 0 <= col < fastdim and 0 <= row < slowdim:
            img_idx = nbimagesperline * row + col
    elif d['fastaxis'] in ('yech', 'yps'):
        if 0 <= col < slowdim and 0 <= row < fastdim:
            img_idx = nbimagesperline * col + row
        else:
            return f"x={x:.4f}, y={y:.4f}, intensity"
    else:
        return f"x={x:.4f}, y={y:.4f}, intensity"

    return f"x={x:.4f}, y={y:.4f}, imageid={img_idx}, intensity"

# --- Initial vmin and vmax values ---
initial_vmin = 0
initial_vmax = 100

# --- Store clicked points ---
clicked_points_list = []

# --- Store image indices for each subplot ---
image_indices = {}

# --- Plot data ---
def update_plots(autocontrast, vmin, vmax, save_clicked_points):
    global image_indices
    image_indices = {}  # Reset image indices
    k = 0
    for i, ax in enumerate(axs_flat):
        row = i // nbcols  # Current row index
        col = i % nbcols   # Current column index

        if k < nbcounters:
            _idx = list_idx_counters[k]
            roidata = tr[_idx]

            # Reshape and transpose logic
            transdata = roidata.reshape((-1, nbimagesperline))
            if d.get('fastaxis') in ('xech', 'xps'):
                xlabel, ylabel = d['fastaxis'], d['slowaxis']
            elif d.get('fastaxis') in ('yech', 'yps'):
                transdata = transdata.T
                xlabel, ylabel = d['slowaxis'], d['fastaxis']
            else:
                xlabel, ylabel = 'xech', 'yech'

            # Apply vmin and vmax
            current_vmin = None if autocontrast else vmin
            current_vmax = None if autocontrast else vmax

            if autocontrast:
                if d['CCDLabel'] == 'sCMOS':
                    current_vmin, current_vmax = 1010, max(1010, 0.75 * np.amax(roidata))
                else:
                    current_vmin, current_vmax = 0, max(200, 0.25 * np.amax(roidata))

            ax.clear()
            ax.imshow(transdata, origin='lower', vmin=current_vmin, vmax=current_vmax)
            ax.format_coord = format_coord
            ax.text(0.5, 0.05, f"{_idx}",
                    ha='center', va='center', fontsize=7, color='red',
                    transform=ax.transAxes)

            # Store image index for this subplot
            image_indices[ax] = _idx

            # Only set xlabel for the bottom row
            if row == nrows - 1:
                ax.set_xlabel(xlabel)
            else:
                ax.set_xticklabels([])  # Remove x-tick labels for non-bottom rows

            # Only set ylabel for the first column
            if col == 0:
                ax.set_ylabel(ylabel)
            else:
                ax.set_yticklabels([])  # Remove y-tick labels for non-first columns

            k += 1
        else:
            ax.clear()
            ax.imshow(missingplot, origin='lower')
            ax.text(0.5, 0.5, 'no data',
                    ha='center', va='center', fontsize=5, color='red',
                    transform=ax.transAxes)
            ax.axis('on')

        

    # Connect click event to save points if checkbox is checked
    if save_clicked_points:
        fig2.canvas.mpl_connect('button_press_event', on_click)
    else:
        fig2.canvas.mpl_disconnect(fig2.canvas.mpl_connect('button_press_event', on_click))

    plt.subplots_adjust(hspace=0.05, wspace=0.05)


    fig2.suptitle(d['imagefolder'], fontsize=8)
    fig2.canvas.draw_idle()

# --- Click event handler ---
def on_click(event):
    if event.inaxes:
        ax = event.inaxes
        x, y = event.xdata, event.ydata
        print(f'point: x={x:.0f}, y={y:.0f}')
        if ax in image_indices:
            _idx = image_indices[ax]
            # Calculate imageid using format_coord logic
            coord_text = format_coord(x, y)
            # Extract imageid from the formatted string
            imageid = int(coord_text.split("imageid=")[1].split(",")[0]) if "imageid=" in coord_text else None
            clicked_points_list.append((int(x), int(y), imageid))
            print(f"Saved point: x={x:.0f}, y={y:.0f}, imageid={imageid}")

# --- Create widgets ---
autocontrast_checkbox = Checkbox(
    value=True,
    description='Autocontrast',
    disabled=False,
    indent=False
)

save_clicked_points_checkbox = Checkbox(
    value=False,
    description='Save Clicked Points',
    disabled=False,
    indent=False
)

vmin_slider = FloatSlider(
    value=initial_vmin,
    min=-1000,
    max=1000,
    step=1,
    description='vmin:',
    disabled=False,
    continuous_update=True
)

vmax_slider = FloatSlider(
    value=initial_vmax,
    min=-1000,
    max=1000,
    step=1,
    description='vmax:',
    disabled=False,
    continuous_update=True
)

# --- Link widgets to update function ---
interact(
    update_plots,
    autocontrast=autocontrast_checkbox,
    vmin=vmin_slider,
    vmax=vmax_slider,
    save_clicked_points=save_clicked_points_checkbox
)

# --- Initial plot ---
update_plots(autocontrast=True, vmin=initial_vmin, vmax=initial_vmax, save_clicked_points=False)

# --- Display the figure ---
plt.show()


In [ ]:
clicked_points_list

In [ ]:
def getposition(d, img_idx_list):
    """
    Retrieve (fastaxis, slowaxis) coordinates for a list of image indices.

    Args:
        d (dict): Dictionary containing metadata (e.g., 'fullcommand', 'localhdf5file', 'scanindex', 'fastaxis', 'slowaxis').
        img_idx_list (list or int): A list of image indices or a single image index.

    Returns:
        list: A list of dictionaries, where each dictionary contains the (fastaxis, slowaxis) coordinates for the corresponding img_idx.
              If img_idx_list is a single integer, returns a list with one dictionary.
    """
    # Convert single img_idx to a list for uniform processing
    if isinstance(img_idx_list, int):
        img_idx_list = [img_idx_list]

    # Initialize the list to store results
    position_list = []

    # Retrieve fastaxis and slowaxis data
    if d['fullcommand'].startswith('fscan2d'):
        if d['fullcommand'].split()[5] in ('yps', 'yech'):
            with h5py.File(d['localhdf5file'], locking=False) as f:
                fastaxis_value = f[f'{d["scanindex"]}.1/instrument/{d["fastaxis"]}/data'][()]
                slowaxis_value = f[f'{d["scanindex"]}.1/instrument/{d["slowaxis"]}/value'][()]

    fa = d['fastaxis']
    sa = d['slowaxis']

    # Iterate over each img_idx in the list
    for img_idx in img_idx_list:
        position_list.append({
            fa: float(np.round(fastaxis_value[img_idx], 4)),
            sa: float(np.round(slowaxis_value[img_idx], 4))
        })

    return position_list

_,_,img_idx = clicked_points_list[-1]
list_img_idx = [_idx for _,_,_idx in clicked_points_list]
getposition(d, img_idx),getposition(d, list_img_idx)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Checkbox, Button, Output
from IPython.display import display

# --- Select counters ---
list_idx_counters = range(0, 0+ 23)  # Example: 23 counters starting from 24
autocontrast = True  # Default: autocontrast enabled
nbcols = 4  # Number of columns for subplots

# --- Validate counters ---
nbcounters = len(list_idx_counters)
print(f"Number of counters: {nbcounters}")
print(f"Highest selected index: {max(list_idx_counters)}")

if max(list_idx_counters) > len(tr):
    raise ValueError(f"Maximum counter index is {len(tr) - 1}")

# --- Subplot layout ---
nbimagesperline = d['nbimagesperline']
nrows = (nbcounters + nbcols - 1) // nbcols  # Ceiling division

# --- Dynamic figure size ---
fastdim, slowdim = d['mapdimensions']
subplot_width_inches = slowdim / 100
subplot_height_inches = fastdim / 100
fig_width = nbcols * subplot_width_inches
fig_height = nrows * subplot_height_inches + 2.0  # Extra space for widgets

# --- Create figure and subplots ---
fig2, axs = plt.subplots(
    nrows=nrows,
    ncols=nbcols,
    sharex=True,
    sharey=True,
    figsize=(fig_width, fig_height)
)

# --- Flatten axes for iteration ---
axs_flat = axs.flatten() if nrows > 1 else [axs]

# --- Missing plot placeholder ---
if d['fastaxis'] in ('xech', 'xps'):
    missingplot = np.zeros_like(tr[0]).reshape((-1, nbimagesperline))
else:  # 'yech' or 'yps'
    missingplot = np.zeros_like(tr[0]).reshape((-1, nbimagesperline)).T

# --- Coordinate formatting ---
def format_coord(x, y):
    fastdim, slowdim = d['mapdimensions']
    col, row = int(x + 0.5), int(y + 0.5)

    if d['fastaxis'] in ('xech', 'xps'):
        if 0 <= col < fastdim and 0 <= row < slowdim:
            img_idx = nbimagesperline * row + col
            return img_idx
    elif d['fastaxis'] in ('yech', 'yps'):
        if 0 <= col < slowdim and 0 <= row < fastdim:
            img_idx = nbimagesperline * col + row
            return img_idx
    return None

# --- Initial vmin and vmax values ---
initial_vmin = 0
initial_vmax = 50000

# --- Global list to store selected regions ---
selected_regions = []

# --- Track the last active subplot (for click) ---
last_active_subplot = None

# --- Function to update the last active subplot ---
def on_click(event):
    global last_active_subplot
    if event.inaxes:
        last_active_subplot = event.inaxes

# Connect the click event to track the last active subplot
fig2.canvas.mpl_connect('button_press_event', on_click)

# --- Function to capture the selected region and compute image indices as a 2D array ---
def capture_selected_region(b):
    global selected_regions, last_active_subplot
    print("Capture button clicked!")  # Debug message

    if last_active_subplot is None:
        print("No subplot was clicked. Please click on a subplot first.")
        return

    # Find the index of the last active subplot
    subplot_index = None
    for i, ax in enumerate(axs_flat):
        if ax == last_active_subplot:
            subplot_index = i
            break

    if subplot_index is None or subplot_index >= nbcounters:
        print("No valid subplot found or subplot index out of range.")
        return

    ax = last_active_subplot
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    # Iterate over all pixels in the visible region
    x_min, x_max = int(np.floor(xlim[0])), int(np.ceil(xlim[1]))
    y_min, y_max = int(np.floor(ylim[0])), int(np.ceil(ylim[1]))

    # Create a 2D array for the visible region
    region_array = np.full((y_max - y_min + 1, x_max - x_min + 1), None, dtype=object)

    for x in range(x_min, x_max + 1):
        for y in range(y_min, y_max + 1):
            img_idx = format_coord(x, y)
            region_array[y - y_min, x - x_min] = img_idx

    # Calculate barycenter (center of the rectangle)
    barycenter_x = (x_min + x_max) / 2
    barycenter_y = (y_min + y_max) / 2
    barycenter_img_idx = format_coord(barycenter_x, barycenter_y)

    # Calculate area (number of valid imageindex elements)
    area = np.count_nonzero(region_array != None)

    # Append region info to selected_regions
    selected_regions.append({
        'subplot_index': list_idx_counters[subplot_index],
        'region_array': region_array,
        'barycenter_imageindex': barycenter_img_idx,
        'barycenter_x': barycenter_x,
        'barycenter_y': barycenter_y,
        'area': area
    })

    print(f"Captured region for subplot {subplot_index}. Total regions: {len(selected_regions)}")

# --- Create a button widget for capturing the selected region ---
capture_button = Button(description="Capture Selected Region")
capture_button.on_click(capture_selected_region)

# --- Plot data ---
def update_plots(autocontrast, vmin, vmax):
    k = 0
    for i, ax in enumerate(axs_flat):
        row = i // nbcols  # Current row index
        col = i % nbcols   # Current column index

        if k < nbcounters:
            _idx = list_idx_counters[k]
            roidata = tr[_idx]

            # Reshape and transpose logic
            transdata = roidata.reshape((-1, nbimagesperline))
            if d.get('fastaxis') in ('xech', 'xps'):
                xlabel, ylabel = d['fastaxis'], d['slowaxis']
            elif d.get('fastaxis') in ('yech', 'yps'):
                transdata = transdata.T
                xlabel, ylabel = d['slowaxis'], d['fastaxis']
            else:
                xlabel, ylabel = 'xech', 'yech'

            # Apply vmin and vmax
            current_vmin = None if autocontrast else vmin
            current_vmax = None if autocontrast else vmax

            if autocontrast:
                if d['CCDLabel'] == 'sCMOS':
                    current_vmin, current_vmax = 1010, max(1010, 0.75 * np.amax(roidata))
                else:
                    current_vmin, current_vmax = 0, max(200, 0.25 * np.amax(roidata))

            ax.clear()
            im = ax.imshow(transdata, origin='lower', vmin=current_vmin, vmax=current_vmax)
            ax.text(0.5, 0.05, f"{_idx}",
                    ha='center', va='center', fontsize=7, color='red',
                    transform=ax.transAxes)

            # Only set xlabel for the bottom row
            if row == nrows - 1:
                ax.set_xlabel(xlabel)
            else:
                ax.set_xticklabels([])  # Remove x-tick labels for non-bottom rows

            # Only set ylabel for the first column
            if col == 0:
                ax.set_ylabel(ylabel)
            else:
                ax.set_yticklabels([])  # Remove y-tick labels for non-first columns

            k += 1
        else:
            ax.clear()
            ax.imshow(missingplot, origin='lower')
            ax.text(0.5, 0.5, 'no data',
                    ha='center', va='center', fontsize=5, color='red',
                    transform=ax.transAxes)
            ax.axis('on')

    plt.subplots_adjust(hspace=0.05, wspace=0.05)
    fig2.canvas.draw_idle()

# --- Create widgets ---
autocontrast_checkbox = Checkbox(
    value=True,
    description='Autocontrast',
    disabled=False,
    indent=False
)

vmin_slider = FloatSlider(
    value=initial_vmin,
    min=-1000,
    max=50000,
    step=100,
    description='vmin:',
    disabled=False,
    continuous_update=True
)

vmax_slider = FloatSlider(
    value=initial_vmax,
    min=-1000,
    max=50000,
    step=100,
    description='vmax:',
    disabled=False,
    continuous_update=True
)

# --- Link widgets to update function ---
interact(
    update_plots,
    autocontrast=autocontrast_checkbox,
    vmin=vmin_slider,
    vmax=vmax_slider
)

# --- Display the button widget ---
display(capture_button)

# --- Initial plot ---
update_plots(autocontrast=True, vmin=initial_vmin, vmax=initial_vmax)

# --- Display the figure ---
plt.show()

# --- Print the list of selected regions (for testing) ---
for region_idx, region in enumerate(selected_regions):
    print(f"Region {region_idx + 1}:")
    print(f"  Subplot index: {region['subplot_index']}")
    print(f"  Barycenter (x, y): ({region['barycenter_x']:.2f}, {region['barycenter_y']:.2f})")
    print(f"  Barycenter imageindex: {region['barycenter_imageindex']}")
    print(f"  Area (number of valid imageindices): {region['area']}")
    print(f"  Region array shape: {region['region_array'].shape}")

In [ ]:
# --- Print the list of selected regions (for testing) ---
for region_idx, region in enumerate(selected_regions):
    print(f"Region {region_idx + 1}:")
    print(f"  Subplot index: {region['subplot_index']}")
    print(f"  Barycenter (x, y): ({region['barycenter_x']:.2f}, {region['barycenter_y']:.2f})")
    print(f"  Barycenter imageindex: {region['barycenter_imageindex']}")
    print(f"  Area (number of valid imageindices): {region['area']}")
    print(f"  Region array shape: {region['region_array'].shape}")
    region_array=region['region_array']
    print(f"  Sample of region array: \n{np.array([
            [region_array[0, 0], region_array[0, -1]],
            [region_array[-1, 0], region_array[-1, -1]]
        ])}")  # Print 4 corners

## save figure and peaks/rois coordinates

In [ ]:
import time

outputfolder = galleryfolder
#outputfolder =  d['folder']
 
# ----  output filename business  -------
nameout = d.get('title','')
tt =time.asctime().strip()
tt = tt.replace(' ','_')
tt=tt.replace(':','_')
coltime="%s_%s_%s"%(collector,spatialcoord,tt)

if nameout != '':
    nameout+='_'+coltime
else:
    nameout=coltime
#---------------------------
print(nameout)
#--------------------------
lcnt = tuple(list_idx_counters)
xsel, ysel = np.take(d['peaklist'],lcnt , axis=0).T
ct_indices = np.array(lcnt)
selected_peaks=np.array([ct_indices,xsel,ysel]).T
coordfile = "roicoordinates_%s.dat"%nameout

with open(os.path.join(d['folder'],coordfile),'w') as f:
    np.savetxt(f, selected_peaks, header='roi_index, X, Y')
# -------------------

plt.savefig(os.path.join(outputfolder,nameout+'.png'))
print('output folder --->', outputfolder)
print('figure filename --->',nameout+'.png')
print('roi coordinates filename ---->',coordfile)

### single map from 1 counter

In [ ]:
# Select counters
idx_counter =0
outputfolder = galleryfolder

# Output filename business
nameout = d.get('title', '')
tt = time.asctime().strip().replace(' ', '_').replace(':', '_')
coltime = f"{collector}_{spatialcoord}_{tt}"
nameout = f"{nameout}_{coltime}" if nameout else coltime

print('nbimagesperline', nbimagesperline)
xy = np.take(d['peaklist'], idx_counter, axis=0)

def format_coord2(x, y):
    if d['fastaxis'] in ('yech', 'yps'):
        col = int(y + 0.5)
        row = int(x + 0.5)
        nfast, nslow = d['mapdimensions']
    else:
        col = int(x + 0.5)
        row = int(y + 0.5)
        nslow, nfast = d['mapdimensions']

    if 0 <= col < nfast and 0 <= row < nslow:
        img_idx = nbimagesperline * row + col
        return f"x={x:1.4f}, y={y:1.4f}, imageid={img_idx}"
    return f"x={x:1.4f}, y={y:1.4f}"

# Create an output widget to display the plot
plot_output = widgets.Output()

# Display the plot in the output widget
with plot_output:
    fig3, ax = plt.subplots(figsize=(8, 6))
    dd = tr[idx_counter]

    if autocontrast:
        vmin, vmax = np.amin(dd), np.amax(dd)
    else:
        vmin, vmax = None, None

    # Reshape data based on fastaxis
    if 'fastaxis' not in d:
        transdata = roidata.reshape((-1, nbimagesperline))
        xlabel, ylabel = 'xech', 'yech'
    else:
        if d['fastaxis'] in ('xech', 'xps'):
            transdata = roidata.reshape((-1, nbimagesperline)).T
            xlabel, ylabel = d['fastaxis'], d['slowaxis']
        elif d['fastaxis'] in ('yech', 'yps'):
            transdata = roidata.reshape((-1, nbimagesperline))
            xlabel, ylabel = d['fastaxis'], d['slowaxis']

    # Plot the data
    im = ax.imshow(transdata, origin='lower', vmin=vmin, vmax=vmax)
    ax.set_title(f"{collectortitle} {spatialcoord}\nroi #{idx_counter} centered on X,Y = {xy[0]:.1f}, {xy[1]:.1f}")
    ax.format_coord = format_coord2
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

    # Add colorbar
    fig3.colorbar(im, ax=ax)

# Display the output widget (this will show the plot)
display(plot_output)

# Save the plot and data
roicounterfile = f"roi_{idx_counter}_x_{int(xy[0])}_y_{int(xy[1])}_{nameout}"
fig3.savefig(os.path.join(outputfolder, f"{roicounterfile}.png"))

header = f"{roicounterfile} {collectortitle} {d['folder']} mapdimensions={d['mapdimensions']}"
with open(os.path.join(outputfolder, f"{roicounterfile}.dat"), 'w') as f:
    np.savetxt(f, dd.reshape((-1, nbimagesperline)), header=header)

print('saving folder ---->', outputfolder)
print('plot saved in ---->', f"{roicounterfile}.png")
print('data saved in ---->', f"{roicounterfile}.dat")

### [OPTION] saving several individual plots

In [ ]:
if 1:
    # --- Inputs ---
    nbmaxcounters = tr.shape[0]
    idx_counter_list = np.arange(24,25)  # Replace with your list of counters
    outputfolder = galleryfolder
    autocontrast = False  # Set to False if you want manual vmin/vmax
    adddatetofilename = False
    
    
    # --- ------------ ---
    if adddatetofilename:
        nameout = d.get('title', '')
        tt = time.asctime().strip().replace(' ', '_').replace(':', '_')
        coltime = f"{collector}_{spatialcoord}_{tt}"
        nameout = f"{nameout}_{coltime}" if nameout else coltime
    else:
        nameout=''
    # --- Ensure output folder exists ---
    os.makedirs(outputfolder, exist_ok=True)
    
    # --- Loop over idx_counter_list ---
    for idx_counter in idx_counter_list:
        print('idx_counter',idx_counter)
        xy = np.take(d['peaklist'], idx_counter, axis=0)
        dd = tr[idx_counter]  # Data for the current idx_counter
    
        # Reshape and transpose logic
        transdata = dd.reshape((-1, nbimagesperline))
        if d.get('fastaxis') in ('xech', 'xps'):
            xlabel, ylabel = d['fastaxis'], d['slowaxis']
        elif d.get('fastaxis') in ('yech', 'yps'):
            transdata = transdata.T
            xlabel, ylabel = d['slowaxis'], d['fastaxis']
        else:
            xlabel, ylabel = 'xech', 'yech'
    
        # --- Set vmin/vmax ---
        if autocontrast:
            vmin, vmax = np.amin(dd), np.amax(dd)
        else:
            vmin, vmax = None, None
    
        # --- Create figure and plot ---
        fig, ax = plt.subplots(figsize=(8, 6))
        im = ax.imshow(transdata, origin='lower', vmin=vmin, vmax=vmax)
        # Adjust tick font size for both x and y axes
        ax.tick_params(axis='both', labelsize=4)  # Set tick font size to 8
        
        ax.set_title(f"{collectortitle}\n {spatialcoord}roi #{idx_counter} at X,Y = {xy[0]:.1f}, {xy[1]:.1f}", fontsize=6)
        ax.set_xlabel(xlabel, fontsize=6)
        ax.set_ylabel(ylabel, fontsize=6)
        #fig.colorbar(im, ax=ax)
    
        # --- Save plot and data ---
        roicounterfile = f"roi_{idx_counter}_x_{int(xy[0])}_y_{int(xy[1])}{nameout}"
        fig.savefig(os.path.join(outputfolder, f"{roicounterfile}.png"), dpi=300, bbox_inches='tight', pad_inches=0.5)
        plt.close(fig)  # Close the figure to free memory
    
        # --- Save data ---
        # header = f"{roicounterfile} {collectortitle} {d['folder']} mapdimensions={d['mapdimensions']}"
        # with open(os.path.join(outputfolder, f"{roicounterfile}.dat"), 'w') as f:
        #     np.savetxt(f, dd.reshape((-1, nbimagesperline)), header=header)
    
        if idx_counter%10==0: print(f"Saved plot (and data) for idx_counter={idx_counter} in {outputfolder}")
    
    print("All plots and data saved successfully.")

## GUI for 2D case, to browse over maps from all roi counters

In [ ]:
print('nb of Roi counters', ', nb of images')
print(tr.shape)
print('scan full command',d['fullcommand'])
print('nbimagesperline',d['mapdimensions'][0])
print("d['mapdimensions']",d['mapdimensions'])
print("d['fastaxis']",d['fastaxis'])

In [ ]:
np.amax(tr),
print('intensity is the largest at immage',np.argmax(np.amax(tr, axis=0)))
np.argmax(tr, axis=0)
np.argmax(tr[:, 673])

In [ ]:
# -------------- GUI to browse counters ---------------
# ---- Input ----
idx_counter0 = 0
autocontrast0 = True
nbmaxcounters = tr.shape[0]
print('nbmaxcounters', nbmaxcounters)

# ---- Output filename business ----
nameout = d.get('title', '')
tt = time.asctime().strip()
tt = tt.replace(' ', '_').replace(':', '_')
coltime = "%s_%s_%s" % (collector, spatialcoord, tt)
nameout = f"{nameout}_{coltime}" if nameout else coltime
nbimagesperline = d['nbimagesperline']


# --- Coordinate formatting ---
def format_coord3(x, y):
    fastdim, slowdim = d['mapdimensions']
    col, row = int(x + 0.5), int(y + 0.5)

    if d['fastaxis'] in ('xech', 'xps'):
        if 0 <= col < fastdim and 0 <= row < slowdim:
            img_idx = nbimagesperline * row + col
            return img_idx
    elif d['fastaxis'] in ('yech', 'yps'):
        if 0 <= col < slowdim and 0 <= row < fastdim:
            img_idx = nbimagesperline * col + row
            return img_idx
    return None

# ---- Create an output widget for the plot ----
plot_output = widgets.Output()

# ---- Plot function ----
def plotmap2D(idx_counter=idx_counter0, vmin=0, vmax=1200, autocontrast=autocontrast0, scale='linear'):
    global currentroi_idx, currentdd, xy, fig

    currentroi_idx = idx_counter
    dd = np.copy(tr[idx_counter])
    currentdd = dd
    xy = np.take(d['peaklist'], idx_counter, axis=0)

    posmax = np.argmax(dd)
    ratiomaxonavg = np.amax(dd) / np.mean(dd)
    print('idx_counter', idx_counter, 'max intensity', np.amax(dd), 'max /avg', ratiomaxonavg,
          'at imageindex, (col, row)', posmax, posmax // nbimagesperline, posmax % nbimagesperline)

    if ratiomaxonavg > 1.1:
        GT.printgreen('DETECTION')

    if autocontrast:
        _max = np.amax(dd)
        _min = np.amin(dd)
        vmin, vmax = _min, _max

    # Reshape and transpose logic
    transdata = dd.reshape((-1, nbimagesperline))
    if d.get('fastaxis') in ('xech', 'xps'):
        xlabel, ylabel = d['fastaxis'], d['slowaxis']
    elif d.get('fastaxis') in ('yech', 'yps'):
        transdata = transdata.T
        xlabel, ylabel = d['slowaxis'], d['fastaxis']
    else:
        xlabel, ylabel = 'xech', 'yech'

    tdata = transdata

    if scale == 'log':
        _vmax = np.log10(vmax)
        _vmin = np.log10(vmin)
        _tdata = np.log10(tdata)
    elif scale == 'sqrt':
        _vmax = np.sqrt(np.fabs(vmax - 1000))
        _vmin = np.sqrt(np.fabs(vmin - 1000))
        _tdata = np.sqrt(np.fabs(tdata - 1000))
    elif scale == 'atan':
        nor = 60000 / np.pi * 2
        _vmax = np.arctan(np.fabs(vmax - 1000) / nor)
        _vmin = np.arctan(np.fabs(vmin - 1000) / nor)
        _tdata = np.arctan(np.fabs(tdata - 1000) / nor)
    else:
        _vmax = vmax
        _vmin = vmin
        _tdata = tdata

    # Clear the output and create a new figure
    plot_output.clear_output(wait=True)
    with plot_output:
        fig, ax = plt.subplots(figsize=(4, 4))
        im = ax.imshow(_tdata, origin='lower', vmax=_vmax, vmin=_vmin, aspect=1)
        txttitle = f"{collectortitle} {spatialcoord}\nroi #{idx_counter} centered on X,Y = {xy[0]:.1f}, {xy[1]:.1f}"
        ax.set_title(txttitle)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.format_coord = format_coord3
        plt.show()

# ---- Set initial vmin/vmax ----
if collector == 'sum':
    vmax0 = 30000000
    vminmax0 = vmax0 / 2.
else:
    vmax0 = 15000
    vminmax0 = 1010

# ---- Create interactive widget ----
ppi = interactive(
    plotmap2D,
    idx_counter=(0, nbmaxcounters - 1),
    vmin=(0, vminmax0),
    vmax=(100, vmax0),
    autocontrast=[False, True],
    scale=['linear', 'log', 'sqrt', 'atan']
)

# ---- Save button ----
buttonsave = widgets.Button(description="Save Plot")
output = widgets.Output()

def on_button_clicked(b):
    fname = os.path.join(genfolder, f'map_roi_{currentroi_idx}')
    fig.savefig(fname)
    np.save(fname, currentdd)
    with output:
        output.clear_output()
        print(f'plot saved in:\n{fname}')

buttonsave.on_click(on_button_clicked)

# ---- Layout ----
layout = widgets.VBox([
    widgets.HBox([ppi, buttonsave]),
    plot_output,
    output
])

# ---- Display the layout ----
display(layout)

# VISUALIZE: 1D case (1D scan (DAXM rainbow technique )or 1D map, time scan, loopscan) : plot and save

In [ ]:
50*151

In [ ]:
# Advice
if d['scantype']=='map':
    GT.printyellow(f'1D visualisation may not be appropriate for your scan since d["scantype"]="{d["scantype"]}" is "map".')
else:
    GT.printgreen(f'This 1D visualisation section is appropriate for your scan since d["scantype"]="{d["scantype"]}".')

In [ ]:
#pathHDF5, d

### [OPTION] read monitor and  set `monitor_data`

In [ ]:
MONITORDATA_AVAILABLE = False

if 1:
    # read h5file for monitor correction...
    import h5py

    with h5py.File(d['localhdf5file'], 'r', locking=False) as f:
        monitor_data = f[f'{d['scanindex']}.1/measurement/mon'][()]
    #scanObj=iohdf5.Scan_hdf5(pathHDF5, d['sample_dataset_scanindex'], collectallscans=True)


    #logfilereader.getscanprops_lowest_hdf5(hdf5fullpath, 'P3_TTh1_longdia_3')
    
    #monitor_data = scanObj.getcounterdata('mon')

    fig000, ax000 = plt.subplots(figsize=(6,3))
    ax000.plot(monitor_data)
    ax000.set_xlabel('image index')
    ax000.set_ylabel('monitor intensity')
    ax000.set_title(f"Monitor \n"+d['sample_dataset_scanindex'])
    ax000.grid()

    MONITORDATA_AVAILABLE = True

    
    nbimagesperline = d['mapdimensions'][0]
    monitor_data2D = monitor_data.reshape((-1,nbimagesperline))
    print(monitor_data2D.shape)


In [ ]:
15000/85000, monitor_data.shape

In [ ]:
# reading thf angle for Diamond scan in bliss hdf5 file
# TODO

In [ ]:
# Single counter plot
# for 1D dimension data (not sample 2D map)
# ------------   INPUT  (roi or counter index ----------------
idx_ct = 10

# --------------------------------------
if d['mapdimensions'][1]==1:
    
    fig00, ax00 = plt.subplots(figsize=(5,3))
    ax00.plot(tr[idx_ct])
    ax00.set_xlabel('image index')
    ax00.set_ylabel('intensity')
    ax00.text(0.5, 0.85, '#counter: %d'%idx_ct,
                        horizontalalignment='center',
                        verticalalignment='center',
                        fontsize=20, color='red',
                        transform=ax00.transAxes)
    pos = d['peaklist'][idx_ct]
    ax00.text(0.5, 0.05, 'ROI location X,Y: %d, %d'%(int(pos[0]), int(pos[1])),
                    horizontalalignment='center',
                    verticalalignment='center',
                    fontsize=8, color='red',
                    transform=ax00.transAxes)
    strcollector = collector
    if collector == 'XYcenterofmass':
        strcollector += ' : %s'%peakposcomponent
        
    ax00.set_title(strcollector)
    ax00.grid()

### GUI to browse single roi intensity profile

In [ ]:
# Single counter plot
# for 1D dimension data (not sample 2D map)
# ------------   INPUT  (roi or counter index ----------------
idx_counter0 = 0
# --------------------------------------

maxnbcounters = tr.shape[0]
    
# to locate approximately BUT AUTOMATICALLY a dip position
from ipywidgets import interact, interactive, fixed, interact_manual

figgui, axgui = plt.subplots(figsize=(5,3))

axgui.set_xlabel('image index')
axgui.set_xlabel('intensity')
axgui.grid()

print('min max data1d', np.amin(monitor_data), np.amax(monitor_data))

def plot1Dcounter(ymin=7000,ymax=10000, idx_counter=idx_counter0, normalizebymax=False, normbymonitor=False):
    """wirescan_index"""
    # xlim = ax1D.get_xlim()
    axgui.clear()
    axgui.grid()
    
    expotime = 15
    offsetmonitor = 31
    pedestal = 0
    
    # ax1D.set_xlim(*xlim)
    data1d = np.copy(tr[idx_counter])
    strangedata = False
    if np.amax(data1d)<1.1:
        strangedata = True
    print('min max data1d  before', np.amin(data1d), np.amax(data1d))
    print('idx_counter', idx_counter)
    
    if normbymonitor:
        data1d =(data1d-pedestal)/(monitor_data-expotime*offsetmonitor)
        #ax00.set_ylim(ymin/1000.,1.05)
        
    ma = np.amax(data1d)
    if normalizebymax:
        data1d *=1./ma
        #ax00.set_ylim(ymin/1000.,1.05)
    else:
        pass
        #ax00.set_ylim(ymin,ymax)
    
    #ax00.plot(monitor_data/4., '-', c='orange')
    print('min max data1d', np.amin(data1d), np.amax(data1d))
    
    axgui.plot(data1d)
    
    if strangedata:
        axgui.text(0.5, 0.5, 'STRANGE DATA', horizontalalignment='center',
                                            verticalalignment='center',
                                            fontsize=60, color='red',
                                            transform=ax00.transAxes)
        
    axgui.text(0.5, 0.85, '#counter: %d'%idx_counter,
               horizontalalignment='center', verticalalignment='center',
               fontsize=20, color='red', transform=ax00.transAxes)
    
    pos = d['peaklist'][idx_counter]
    axgui.text(0.5, 0.05, 'ROI location X,Y: %d, %d'%(int(pos[0]), int(pos[1])),
                horizontalalignment='center',
                verticalalignment='center',
                fontsize=8, color='red',
                transform=ax00.transAxes)
    strcollector = collector
    if collector == 'XYcenterofmass':
        strcollector += ' : %s'%peakposcomponent

    axgui.set_title(strcollector)
    
interactive(plot1Dcounter, ymin=(1,10000), ymax=(100,90000), idx_counter=(0,maxnbcounters-1),normalizebymax=[True,False], normbymonitor=[True, False])

In [ ]:
# several counters
# for 1D dimension data (not sample 2D map)

NORMALIZE_BY_MONITOR = False

if 1:
    # ------------   INPUT : set of roi or counter indices ----------------
    first_idx_ct = 0
    nbcounterstoplot = 10
    
    # -----------------------------------------
    if nbcounterstoplot>=40:
        GT.printyellow(f'  "nbcounterstoplot"={nbcounterstoplot} is not reasonable!')
    print("requested list of indices", np.arange(first_idx_ct,first_idx_ct+nbcounterstoplot))
    if d['mapdimensions'][1]==1:
        fig7, ax7 = plt.subplots(nrows=nbcounterstoplot, figsize=(5,9), sharex=True, sharey=False)
        
        strcollector = collector
        if collector == 'XYcenterofmass':
            strcollector += ' : %s'%peakposcomponent
        
        ax7[0].set_title(strcollector)
        ax7[-1].set_xlabel('image index')
        
        for k, ip in enumerate(tr[first_idx_ct:first_idx_ct+nbcounterstoplot]):

            # eiger4n cdte
            pedestal = 0
            offsetmonitor = 31.7
            expotime = 5
            if NORMALIZE_BY_MONITOR:
                ip =(ip-pedestal)/(monitor_data-expotime*offsetmonitor)

            ax7[k].plot(ip)
            #scale and do not consider outliers
            ymean = np.mean(ip)
            ystd = np.std(ip)
            ymax= ymean + 3*ystd
            ymin= ymean - 3*ystd
            
            ax7[k].set_ylim(ymin, ymax)
            ax7[k].text(0.5, 0.85, 'counter: %d'%(k+first_idx_ct),
                            horizontalalignment='center',
                            verticalalignment='center',
                            fontsize=20, color='red',
                            transform=ax7[k].transAxes)
            pos = d['peaklist'][k+first_idx_ct]
            ax7[k].text(0.5, 0.05, 'X,Y: %d, %d'%(int(pos[0]), int(pos[1])),
                            horizontalalignment='center',
                            verticalalignment='center',
                            fontsize=8, color='red',
                            transform=ax7[k].transAxes)

            plt.show()

### CAN BE SKIPPED if not using additional ROIs for background estimation around peaks on ROI

In [ ]:
if ADD_ROIS_FOR_DAXM_BACKGROUND:
    # IF BACKGROUND ROIs WERE DEFINED
    # several counters (roiposition and bckgposition)  tr[:len(tr)/2]  and tr[len(tr)/2:]
    # for 1D dimension data (monitoring) (not sample map)
    
    if d['mapdimensions'][1]==1:
        
        # ------------   INPUT  ----------------
        nbcounterstoplot = 20
        first_relative_idx_ct = 0
        # --------------------------------------
        
        fig, ax = plt.subplots(nrows=nbcounterstoplot, figsize=(7,12), sharex=True, sharey=False)
        
        strcollector = collector
        if collector == 'XYcenterofmass':
            strcollector += ' : %s'%peakposcomponent
        
        ax[0].set_title(strcollector)
        
        indexgood = np.arange(len(tr))
        
        for k,ix in enumerate(indexgood[first_relative_idx_ct:first_relative_idx_ct+nbcounterstoplot]):
            ip =tr[ix]
            ax[k].plot(ip)
            if roiposition[ix+len(tr)//2][0]!=0.:
                #print('I ve got corresponding background signal')
                ax[k].plot(tr[ix+len(tr)//2])
            #scale and do not consider outliers
            ymean = np.mean(ip)
            ystd = np.std(ip)
            ymax= ymean + 3*ystd
            ymin= ymean - 4*ystd
            ax[k].set_ylim(ymin, ymax)
            ax[k].text(0.5, 0.85, 'counter: %d'%(ix),
                            horizontalalignment='center',
                            verticalalignment='center',
                            fontsize=20, color='red',
                            transform=ax[k].transAxes)
            pos = d['peaklist'][ix]
            posbckg = d['peaklist'][ix+len(tr)//2]
            ax[k].text(0.5, 0.05, 'X,Y: %d, %d  bckg X,Y: %d %d'%(int(pos[0]), int(pos[1]),posbckg[0],posbckg[1]),
                            horizontalalignment='center',
                            verticalalignment='center',
                            fontsize=8, color='red',
                            transform=ax[k].transAxes)

        plt.show()

In [ ]:
if 0:
    # for 1D dimension data (monitoring) (not sample map) and long scan
    # single counter (divided in several parts)
    idx_ct = 4
    nbscandivisions=10
    
    #----------------------------------------------
    nbpts = len(tr[idx_ct])
    if d['mapdimensions'][1]==1:
        
        fig, ax = plt.subplots(nrows=nbscandivisions, figsize=(7,15), sharey=False)
        fig.suptitle(f'#counter : {idx_ct}')
        ax[-1].set_xlabel('image index')
        for k in range(nbscandivisions):
            ax[k].plot(np.arange(k*nbpts//nbscandivisions,(k+1)*nbpts//nbscandivisions),
                       tr[idx_ct][k*nbpts//nbscandivisions:(k+1)*nbpts//nbscandivisions],'o-')

## 1D Case: DAXM wire scans. Finding dips automatically

In [ ]:
def smooth(y, box_pts):
    box = np.ones(box_pts)/box_pts
    y_smooth = np.convolve(y, box, mode='same')
    return y_smooth

### [OPTION] [SAND BOX]

In [ ]:
# POOR and not working model to get the ABSOLUTE Depth  
# TODO see LaueTools/Daxm/Geometry ...!!

# simple model of triangulation
# get depth from pixel and wire position 
# dd = 79 mm, H (of wire) along detector direction = 1 mm, pixelsize = 0.0734  , camera positionned ycen= 1000
# wire0 = position of wire in between ycen and sample  in image index unit
# stepw =0.001 mm

#depth = stepw*(w-w0)/(Ycen-Y)/pixelsize + H/(dd-H)*(pixelsize*(Ycen-Y)+stepw*(w-w0))
def getdepthcrude(Y, w,w0=300,stepw=0.001, H=0.7,dd=82,Ycen=1000,pixelsize=0.0734):
    depth = stepw*(w-w0)/(Ycen-Y)/pixelsize + H/(dd-H)*(pixelsize*(Ycen-Y)+stepw*(w-w0))
    return depth

In [ ]:
# to locate approximately BUT AUTOMATICALLY a dip position 
# (DEFAULT wire diameter about 50 µm and ~ 50 steps or images long dip in intensity profile
from ipywidgets import interact, interactive, fixed, interact_manual

# select counters
nbmaxcounters = tr.shape[0]

idx_counter0 = 0

fig1D, ax1D = plt.subplots(figsize=(6,4))
mini = np.amin(tr[idx_counter0])
avg = np.mean(tr[idx_counter0])
vmin, vmax = avg-0.5,avg+0.5
vmin, vmax = None, None  # default

ax1D.grid()
ax1D.set_ylim(-0.05,0.3)

maxtr = np.amax(tr)

xy = np.take(d['peaklist'],idx_counter0, axis=0)
#plt.plot(tr[idx_counter])
#plt.title("%s %s\nroi #%d centered on X,Y =%.1f, %.1f"%(collectortitle,spatialcoord,idx_counter,xy[0],xy[1]))
xx = np.arange(tr.shape[1])

# dipf is + - designed for wire of 50 µm in diameter producing  + - 50 steps (images) long dip in pixel intensity profile
dipf = np.concatenate((np.zeros(5),np.linspace(-.1,-1,10),-np.ones(45),np.linspace(-1,-.1,10),np.zeros(5)))

def plot1Dcounter(idx_counter=idx_counter0, wirescan_index=0, ymin=-200,ymax=10000, boxsizesmooth=11):
    """wirescan_index"""
    # xlim = ax1D.get_xlim()
    ax1D.clear()
    ax1D.grid()
    ax1D.set_ylim(ymin,ymax)
    # ax1D.set_xlim(*xlim)
    dd= tr[idx_counter][wirescan_index*d['mapdimensions'][0]:(wirescan_index+1)*d['mapdimensions'][0]]
#     param = np.polyfit(xx, dd, 1)
#     p1d = np.poly1d(param)
#     ave = p1d(xx)
    # NORMALIZATION By MONITOR
    # pedestal = 0
    # offsetmonitor = 31.7
    # expotime = 5
    # dd =(dd-pedestal)/(monitor_data-expotime*offsetmonitor)
    
    if 0 and np.amax(dd)<=1100:
        dd= np.zeros_like(dd)
        ave = np.zeros_like(ave)
    
    #sig1 = -(dd-ave)
    sig1 = - (dd - np.mean(dd))
    
    # print(np.mean(dd))
    # print(np.std(sig1))
    sig = sig1-np.amin(sig1)
    sig = sig/np.amax(sig)
    
    ddsmoothed = smooth(dd, boxsizesmooth)
    ddprime = -(ddsmoothed[1:]-ddsmoothed[:-1])  # negative of difference

    ax1D.plot(dd)
    ax1D.plot(ddsmoothed, '-', c='orange')

    #     ax1D.plot(sig1)
#     ax1D.plot(1000*sig)
    #ax1D.plot(100*ddprime+np.mean(dd)*.75)
    ax1D.axhline(np.mean(dd)*.75, color ='r')
#     ax1D.plot(sig)
    
    cv = np.convolve(sig,-dipf)
    
    
    centerdip = np.argmax(cv)-dipf.shape[0]//2
    
    #print('center dip at :',centerdip)
    #ddiff = np.diff(sig)
    #ddiff = np.gradient(sig)
    #ddiff =- np.mean(ddiff)
    #ddiff /= np.std(ddiff)
    #ax1D.plot(5000*ddiff)
    #ax1D.plot(100*ddiff*sig[:600])
    
    xy = np.take(d['peaklist'],idx_counter, axis=0)
    
    #print('estimated dip middle position (image index)',centerdip, ', estimated depth: %.4f mm'%getdepthcrude(xy[1],centerdip))

    txttitle = "%s %s\nroi #%d centered on X,Y =%.1f, %.1f\n"%(collectortitle,
                                                            spatialcoord,idx_counter,xy[0],xy[1])
    txttitle += f'centerdipr @{centerdip}'
    txttitle += f' approx depth: {np.round(getdepthcrude(xy[1],centerdip),3)} mm'
    
    plt.title(txttitle)
    ax1D.axvline(centerdip, color ='k')
    ax1D.set_xlabel('image index')
    #plt.show()
    
interactive(plot1Dcounter, idx_counter=(0,nbmaxcounters-1), wirescan_index=(0,d['mapdimensions'][1]-1), 
            ymin=(-200,2900), ymax=(1,int(maxtr*1.02)), boxsizesmooth=(3,50))

In [ ]:
# calculate image difference

# idx_counter = 0

# wirescan_index = 0  #!= 0 in case of sequence of zf scan

# dd= tr[idx_counter][wirescan_index*d['mapdimensions'][0]:(wirescan_index+1)*d['mapdimensions'][0]]
# ddsmoothed = smooth(dd, 9)
# ddprime = -(ddsmoothed[1:]-ddsmoothed[:-1])  # negative of difference

# print('min max of yaxis', np.amin(ddprime),np.amax(ddprime))

# figprime, axprime = plt.subplots()
# axprime.plot(np.arange(1,len(ddsmoothed)),ddprime )
# axprime.set_xlabel('image index')
# axprime.grid()
# axprime.set_xlim(0,380)
# axprime.set_ylim(-350,400)

In [ ]:
# Algorithm to get dip position (without np.diff  or np.gradient)

idx_counter0 = 20
dd= tr[idx_counter0]
    
param = np.polyfit(xx, dd, 1)
p1d = np.poly1d(param)

ave = p1d(xx)

if np.amax(dd)<=1100:
    dd= np.zeros_like(dd)
    ave = np.zeros_like(ave)

sig1 = -(dd-ave)
#sig = sig/np.std(sig)

print(np.mean(dd))
print(np.std(sig1))
sig = sig1-np.amin(sig1)
sig = sig/np.amax(sig)
#sig = trtemplate
#sig = tr[362]

cv = np.convolve(sig,-dipf)

print('position dip', np.argmax(cv))
figc, axc = plt.subplots()
axc.plot(sig)
axc.plot(-dipf)
axc.plot(.01*cv, 'o')
axc.grid()

In [ ]:
#np.argmax(cv)-dipf.shape[0]//2, cv

### BUILD array of dip positions (in image index unit) for all roi intensity profiles: `posdip`

apply the algorithm designed above

In [ ]:
tr.shape

In [ ]:
tr[0,:].shape

In [ ]:
from scipy import signal

# vectorized method to find dip position from an array of intensity profiles
nbcounters = tr.shape[0]

pedestal = 0
offsetmonitor = 31.7
expotime = 5
tr =(tr-pedestal)/(monitor_data-expotime*offsetmonitor)

# ----------------------------------------------------
sig1 = - (tr - np.mean(tr, axis=1).reshape((nbcounters,1)))
sig0 = sig1-np.amin(sig1, axis=1).reshape((nbcounters,1))  
trprepared = sig0/np.amax(sig0, axis=1).reshape((nbcounters,1))
# sig = sig1-np.amin(sig1)
# sig = sig/np.amax(sig)

# must be all equal
print('these dimensions must be equal')
print(sig0.shape, sig1.shape, trprepared.shape)

# build a kernel to apply convolve2D but only getting results along one 1D 
dipf = np.concatenate((np.zeros(5),np.linspace(-.1,-1,10),-np.ones(45),np.linspace(-1,-.1,10),np.zeros(5)))

nbcounters = tr.shape[0]
idx0_counter= 0
dipfprepared = np.concatenate((dipf, np.zeros_like(dipf))).reshape((2,len(dipf)))
res = signal.convolve2d(trprepared[idx0_counter:idx0_counter+nbcounters,:],-dipfprepared)
print('convoluted profiles dimensions', res.shape)

print(trprepared[idx0_counter:idx0_counter+nbcounters,:].shape,dipfprepared.shape,res.shape)
# https://stackoverflow.com/questions/28898858/python-apply-along-axis-of-multiple-arrays/28904614#28904614
# subindexcounter = 0
# figcc, axcc = plt.subplots()
# axcc.plot(trprepared[idx0_counter+subindexcounter])
# axcc.plot(-dipfprepared[0])
# axcc.plot(.01*res[subindexcounter], 'o')
# axcc.grid()

# print('for counter', idx0_counter+subindexcounter)
# print('X,Y ', d['peaklist'][ idx0_counter+subindexcounter])
# print('dip middle image index',np.argmax(res[subindexcounter])-dipf.shape[0]//2)

posdip = np.argmax(res[:-1], axis=1)-dipf.shape[0]//2

In [ ]:
posdip

In [ ]:
# useless plot when abscissa (the Laue index) even when sorted by increasing Y 
# if 0:
#     figdd, axdd = plt.subplots(figsize=(10,4))
#     axdd.plot(posdip, 'o')
#     axdd.grid()
#     axdd.set_xlabel('index laue spot')
#     axdd.set_ylabel('position of dip (image index)')

## Quick DAXM (1D Case) Ge Calibration ONLY: wire(s) parameters. Improve rapid image index wireindex => depth   (calibration)

it allows to determine wire parameters: WPGe (for instance = [270, 85, 290, -295, 265, -585, 320, -1050])

In [ ]:
titlescan = d.get('sample_dataset_scanindex','DAXM calibration Zr grain')
XY = d['peaklist']
# peak intensity distribution
I = d['intensities']
print('statistics intensity',np.median(I), np.amax(I), np.amin(I))
print('XY.shape', XY.shape)
print('titlescan',titlescan)

In [ ]:
fig0, ax0=plt.subplots()
ax0.hist(XY[:,1], bins=80)

In [ ]:
### superimpose Laue spots belonging to the same grain
titleplot = '{expId} single grain\n' + titlescan
# GUI   set model 
ADDLAUESPOTMARKERS = False

# -----------------------------------------------------------
#MinIntensityThreshold = -1000
#s_idx = np.where(I>MinIntensityThreshold)[0]

s_idx = np.arange(len(XY))
XY = d['peaklist'][s_idx]

Xexp, Yexp = XY.T

# W4  Y from 0 380  top quarter  posdip = 200/(420-90)  Y + 86
# W3  Y 520 - 920
# W2  Y 1060 - 1480
# W1  Y 1570  -2000
Ytheo = np.arange(2048)

# model of Wire params

# initial Wires Parameters trajectory 
#
WP0 = [270, 85, 310, -325, 265, -585, 320, -1050]  # me1701
WP0 = [400, 525, 435, -485, 440, -955, 435, -1410]  # ma6758

dip4 = WP0[0]/(450)*Ytheo +WP0[1]
dip3 = WP0[2]/(450)*Ytheo +WP0[3]
dip2 = WP0[4]/(450)*Ytheo +WP0[5]
dip1 = WP0[6]/(450)*Ytheo +WP0[7]

figdd, axdd = plt.subplots(figsize=(5,3))
axdd.plot(Yexp,posdip[s_idx], 'o')  # exp dip
axdd.plot(Ytheo,dip4, '-r')
axdd.plot(Ytheo,dip3, '-r')
axdd.plot(Ytheo,dip2, '-r')
axdd.plot(Ytheo,dip1, '-r')

axdd.set_xlim(Yexp[0]-10, Yexp[-1]+10)
axdd.set_ylim(np.amin(posdip[s_idx])-10, np.amax(posdip[s_idx])+10)

def addmarkeronspots(listindices, arraypts, ax, color='k'):
    selectedspots = np.take(arraypts,listindices, axis=0)
    _, Yselected = selectedspots.T
    
    selectedposdip = posdip[s_idx][listindices]
    ax.scatter(Yselected, selectedposdip, marker='x',c=color, s=200, alpha=0.5)
    #print(Yselected, selectedposdip)
    
if ADDLAUESPOTMARKERS:
    
    addmarkeronspots(spotsofg0_inXY, XY, axdd)
    addmarkeronspots(spotsofg1_inXY, XY, axdd, color='g')

def plotposdip(w4_slope=WP0[0], w4_origordinate=WP0[1],
              w3_slope=WP0[2], w3_origordinate=WP0[3],
              w2_slope=WP0[4], w2_origordinate=WP0[5],
              w1_slope=WP0[6], w1_origordinate=WP0[7], showspotindex=False):
    
    ymin, ymax = axdd.get_ylim()
    xmin, xmax = axdd.get_xlim()

    axdd.clear()
    #--------------------------------
    dip4 = w4_slope/450.*Ytheo + w4_origordinate
    dip3 = w3_slope/450.*Ytheo + w3_origordinate
    dip2 = w2_slope/450.*Ytheo + w2_origordinate
    dip1 = w1_slope/450.*Ytheo + w1_origordinate
    
    axdd.plot(Yexp,posdip[s_idx], 'o', alpha=0.1)  # exp dip
    axdd.plot(Ytheo,dip4, '-r')
    axdd.plot(Ytheo,dip3, '-r')
    axdd.plot(Ytheo,dip2, '-r')
    axdd.plot(Ytheo,dip1, '-r')
    axdd.grid()
    axdd.set_xlabel('Y laue spot')
    axdd.set_ylabel('position of dip (image index)')
    axdd.set_ylim(0, tr.shape[1])
    
    if ADDLAUESPOTMARKERS:
    
        addmarkeronspots(spotsofg0_inXY, XY, axdd)
        addmarkeronspots(spotsofg1_inXY, XY, axdd, color='g')

    print('Current W4, W3, W2, W1 parameters :')
    print([w4_slope, w4_origordinate, w3_slope,w3_origordinate,
              w2_slope, w2_origordinate,w1_slope,w1_origordinate])
    
    if showspotindex:
        # add annotations one by one with a loop
        for _k, _yexp in enumerate(Yexp):
            alldata_txt = '#spot %d %.1f, %.1f'%(_k, Xexp[_k],_yexp)
            index_txt = '%d'%_k
            
            annottxt = index_txt
            axdd.text(
                  _yexp+0.2,
                  posdip[s_idx][_k],
                  annottxt,
                  ha='left',
             )

    axdd.set_xlim(xmin, xmax)
    axdd.set_ylim(ymin, ymax)

    axdd.set_title(titleplot)
    
interactive(plotposdip, w4_slope=(100,400,5), w4_origordinate=(-500,600,5),
           w3_slope=(100,600,5), w3_origordinate=(-800,200,5),
           w2_slope=(100,600,5), w2_origordinate=(-1300,0,5),
           w1_slope=(100,600,5), w1_origordinate=(-1500,0,5), showspotindex=[True, False])

In [ ]:
# copy and paste the list above 'W4,W3, W2, W1 parameters' to set WP
WPGe = [270, 85, 290, -295, 265, -585, 320, -1050]  # d289  ME1701

WPAl = [400, 525, 435, -485, 440, -955, 435, -1410]  # d89  ma6758

### [OPTIONAL PLOT]

In [ ]:
#  OPTIONAL
# from dip, Y =>  wire involved and depth

# Todo find W4 W3 W2 W1 automatically from WP, get wire step from dict d


# From previous plot

# W4  Y from 0 to 670  
# W3  Y 750 - 1440
# W2  Y 1500 - 2000
# W1  Y ---- -----

Wireparams= np.zeros((5,2))
for k, val in enumerate([4,3,2,1]):
    Wireparams[val]= (WP[2*k]/450.,WP[2*k+1])
    
Yexp = XY[:,1]
def roughdepth(Y, posdip, W4=(0,670),W3=(750,1440),W2=(1500,2000),W1=(3000,4000), Wireparams=Wireparams, wirestep=1):
    """Estimate the depth of spots (pixel coordinates Y) and its image index posdip
    
    If spot at Y is masked by at least two wires then default value is -200
    
    Wi is the Y Laue spots range where spots are masked only by wire #i
    etc...
    WireParameters are the 8 parameters (2 per wire) describing the line between posdip and Y Laue spot
    wirestep: wire scan step default is 1 micron
    """
    cond4 = Y < W4[1]
    cond3 = np.logical_and(Y>=W3[0],Y<=W3[1])
    cond2 = np.logical_and(Y>=W2[0],Y<=W2[1])
    cond1 = np.logical_and(Y>=W1[0],Y<=W1[1])
    condlist = [cond4, cond3, cond2, cond1]
    
    
    choicelist = [posdip-(Wireparams[4][0]*Y+Wireparams[4][1]),
                     posdip-(Wireparams[3][0]*Y+Wireparams[3][1]),
                        posdip-(Wireparams[2][0]*Y+Wireparams[2][1]),
                        posdip-(Wireparams[1][0]*Y+Wireparams[1][1])]
                        
    return np.select(condlist, choicelist*wirestep, -200)

roughdepth(Yexp, posdip)



In [ ]:
figddd, axddd = plt.subplots()
                                     
axddd.plot(Yexp,roughdepth(Yexp, posdip), 'o')  

axddd.grid()
axddd.set_xlabel('Y laue spot')
axddd.set_ylabel('estimated depth (microns)')
axddd.set_title('Wire calibration from Ge')



## Quick DAXM (1D Case) data of SAMPLE OF INTEREST 

In [ ]:
#d

In [ ]:
titlescan = d.get('sample_dataset_scanindex',d['folder'])
XY = d['peaklist']
# peak intensity distribution
I = d['intensities']
print('statistics intensity',np.median(I), np.amax(I), np.amin(I))
print('XY.shape', XY.shape)
depthtolerance =None
print('titlescan',titlescan)

In [ ]:
### superimpose Laue spots belonging to the same grain

# GUI   set model 
ADDLAUESPOTMARKERS = False
MinIntensityThreshold = -1000


#-------------------------------------------------------
s_idx = np.where(I>MinIntensityThreshold)[0]
print(f'spots indices above intensity is larger than {MinIntensityThreshold}')
print('s_idx', s_idx)
XY = d['peaklist'][s_idx]
Xexp, Yexp = XY.T
Ytheo = np.arange(2048)

# model of Wire params
# initial Wires Parameters trajectory 
#d['sample_dataset_scanindex'] = 2050_EBM_daxm_Al_daxm_step50um_fromsurfacemoins100um_1
WP0 = [375, 60, 310, -325, 300, -700, 320, -1050] # close to WP of Ge  (section 13.1.1)
WP0 = [315, 60, 310, -325, 300, -690, 320, -1045]
#WP0 =[270, 85, 290, -295, 265, -585, 320, -1050]  d290 ME1701
#WP0 = WPAl 


dip4 = WP0[0]/(450)*Ytheo +WP0[1]
dip3 = WP0[2]/(450)*Ytheo +WP0[3]
dip2 = WP0[4]/(450)*Ytheo +WP0[5]
dip1 = WP0[6]/(450)*Ytheo +WP0[7]

figdd, axdd = plt.subplots(figsize=(5,3))
axdd.plot(Yexp,posdip[s_idx], 'o')  # exp dip
axdd.plot(Ytheo,dip4, '-r')
axdd.plot(Ytheo,dip3, '-r')
axdd.plot(Ytheo,dip2, '-r')
axdd.plot(Ytheo,dip1, '-r')

axdd.set_xlim(Yexp[0]-10, Yexp[-1]+10)
axdd.set_ylim(np.amin(posdip[s_idx])-10, np.amax(posdip[s_idx])+10)


def addmarkeronspots(listindices, arraypts, ax, color='k'):
    selectedspots = np.take(arraypts,listindices, axis=0)
    _, Yselected = selectedspots.T
    
    selectedposdip = posdip[s_idx][listindices]
    ax.scatter(Yselected, selectedposdip, marker='x',c=color, s=200, alpha=0.5)
        
if ADDLAUESPOTMARKERS:
    
    addmarkeronspots(spotsofg0_inXY, XY, axdd)
    #addmarkeronspots(spotsofg1_inXY, XY, axdd, color='g')

def plotposdip(w4_slope=WP0[0], w4_origordinate=WP0[1],
              w3_slope=WP0[2], w3_origordinate=WP0[3],
              w2_slope=WP0[4], w2_origordinate=WP0[5],
              w1_slope=WP0[6], w1_origordinate=WP0[7], showspotindex=False):
    
    ymin, ymax = axdd.get_ylim()
    xmin, xmax = axdd.get_xlim()

    axdd.clear()

    #--------------------------------
    dip4 = w4_slope/450.*Ytheo + w4_origordinate
    dip3 = w3_slope/450.*Ytheo + w3_origordinate
    dip2 = w2_slope/450.*Ytheo + w2_origordinate
    dip1 = w1_slope/450.*Ytheo + w1_origordinate
    
    axdd.plot(Yexp,posdip[s_idx], 'o', alpha=0.1)  # exp dip
    axdd.plot(Ytheo,dip4, '-g')
    axdd.plot(Ytheo,dip3, '-g')
    axdd.plot(Ytheo,dip2, '-g')
    axdd.plot(Ytheo,dip1, '-g')
    axdd.grid()
    axdd.set_xlabel('Y laue spot')
    axdd.set_ylabel('position of dip (image index)')
    axdd.set_ylim(0, tr.shape[1])

    if depthtolerance is not None:
        for k in range(4):
            ar = np.array(line_at_depth_wire[k])
            axdd.plot(Ytest,ar, '--r')  # central line
            axdd.plot(Ytest,ar+depthtolerance, '-r')
            axdd.plot(Ytest,ar-depthtolerance, '-r')
    
    
    if ADDLAUESPOTMARKERS:
    
        addmarkeronspots(spotsofg0_inXY, XY, axdd)
        addmarkeronspots(spotsofg1_inXY, XY, axdd, color='g')

    print('Current W4, W3, W2, W1 parameters :')
    print([w4_slope, w4_origordinate, w3_slope,w3_origordinate,
              w2_slope, w2_origordinate,w1_slope,w1_origordinate])
    
    if showspotindex:
        # add annotations one by one with a loop
        for _k, _yexp in enumerate(Yexp):
            alldata_txt = '#spot %d %.1f, %.1f'%(_k, Xexp[_k],_yexp)
            index_txt = '%d'%_k
            
            annottxt = index_txt
            axdd.text(
                  _yexp+0.2,
                  posdip[s_idx][_k],
                  annottxt,
                  ha='left',
             )
                
    axdd.set_xlim(xmin, xmax)
    axdd.set_ylim(ymin, ymax)
    
interactive(plotposdip, w4_slope=(100,400,5), w4_origordinate=(-500,600,5),
           w3_slope=(100,600,5), w3_origordinate=(-800,200,5),
           w2_slope=(100,600,5), w2_origordinate=(-1300,0,5),
           w1_slope=(100,600,5), w1_origordinate=(-1500,0,5), showspotindex=[True, False])

In [ ]:
# copy and paste the list above 'W4,W3, W2, W1 parameters' to set WP for sample of interest data
WP =[270, 85, 290, -295, 265, -585, 320, -1050] 

WP = [270, 85, 435, -495, 440, -965, 435, -1415]  #d48
WP = [400, 525, 435, -485, 440, -955, 435, -1410] # d89
WP = [270, 245, 270, -95, 270, -440, 270, -795]  # d15 a321217  LaB6

### Capture spots index at the vicinity of straight line of wire shadows

In [ ]:
assert posdip.shape[0] == XY.shape[0]

In [ ]:

posdip.shape, XY.shape, Ytest.shape

In [ ]:
# Query and capture spots coming from grain at depthprobe within depthtolerance

depthprobe = 95 # micron
depthtolerance = 2 # micron

XYtest = copy.copy(XY)




Ytest = XYtest[:,1]
_posdip = copy.copy(posdip)


spots_at_depth = []

#wire shadowing abacus: line eq   Y = WP[2*i]/450.*X + WP[2*i+1]
selectedwires =(4,3,2,1)
#selectedwires = (2,)

line_at_depth_wire = []
for wireindex in (selectedwires):
    idx_in_WP = 4-wireindex
    
    line_at_depth_wire.append((WP[2*idx_in_WP]/450.*Ytest + WP[2*idx_in_WP+1]+depthprobe).tolist())
        # print(f'central dip position  w# {idx_in_WP}',(WP[2*idx_in_WP]/450.*Ytest + WP[2*idx_in_WP+1]+depthprobe).tolist())
    cond = np.fabs(_posdip - (WP[2*idx_in_WP]/450.*Ytest + WP[2*idx_in_WP+1]+depthprobe))<depthtolerance
    spots_at_depth+=np.where(cond)[0].tolist()
# found spots at given depth
nbspotsfound = len(spots_at_depth)
GT.printgreen(f'{nbspotsfound} spots found coming from grain(s) at depth (along the beam) {depthprobe} +/- {depthtolerance} µm')
XY_at_depth = np.take(XYtest, spots_at_depth, axis=0)
intensities_at_depth = np.take(intensities, spots_at_depth, axis=0)
XY_at_depth

In [ ]:
# sort by intensity 
ix_sorted = np.argsort(intensities_at_depth)[::-1]
XY_at_depth_sorted = np.take(XY_at_depth,ix_sorted,axis=0 )
XYI_at_depth = np.column_stack((XY_at_depth_sorted, intensities_at_depth[ix_sorted]))

print('X, Y I for spots at considered depth')
XYI_at_depth

In [ ]:
# Remove undesired spots--------------------
minimumY = 0  # 538
XYI_at_depth = XYI_at_depth[XYI_at_depth[:,1]>minimumY]
# _Ytest = XYtest[:,1]
# _posdip = copy.copy(posdip)
# # keep only some pts
# MinY= 538
# idx = np.where(_Ytest>MinY)[0]
# XYtest = np.take(XYtest, idx, axis=0)
# _posdip = np.take(_posdip, idx)

In [ ]:
fig11, ax11 = plt.subplots()
ax11.scatter(XYI_at_depth[:,0],XYI_at_depth[:,1])
ax11.set_ylim(2100,0)
ax11.set_xlabel('pixelX')
ax11.set_ylabel('pixelY')

In [ ]:
fig12, ax12 = plt.subplots()

import LaueTools.indexingImageMatching as IIM
import LaueTools.LaueGeometry as LaueGeo

calibdetfile = '/data/visitor/a321217/bm32/20260707/RAW_DATA/LaB6_2/LaB6_2_gedaxm/scan0001/calibGe001_crude_eiger4M_july2026_A321217.det'
detectorplaneparameters = [99.66981616892743, 1077.5005899952898, 1177.652469333325, 0.4096519569722474, 0.4229933899487947]
tth, chi = LaueGeo.calc_uflab(XYI_at_depth[:,0],XYI_at_depth[:,1],detectorplaneparameters, pixelsize=0.075)

gnomonx, gmonony = IIM.ComputeGnomon_2([tth, chi])

ax12.scatter(gnomonx, gmonony)
ax12.grid()

ax12.set_xlabel('gnomonX')
ax12.set_ylabel('gnomonY')

###   Reading laue spot XY coordinates from an exhaustive peak list in a .cor file

In [ ]:
# uncomment to list files in d['folder']
#!ls {d['folder']}

In [ ]:
## read .cor file
from LaueTools.IOLaueTools import readfile_cor, writefile_cor

#"/data/visitor/ma6758/bm32/20260312/RAW_DATA/Al/Al_Al650um_daxm_h1.7_expo_5_nbsteps_650_hmicro1p7mm/scan0002/dat_eiger4m_0584_LT_2.cor"
#resfit=readfile_fit(os.path.join(ExperimentFolder,'A45/A45_line1daxms/scan0005','dat_img_0000_Al2O3_grain1.fit'))
#fullpathcorfilename= '/data/visitor/ma6758/bm32/20260312/RAW_DATA/Al/Al_Al650um_daxm_h1.7_expo_1_nbsteps_650_hmicro1p7mm_0020/scan0002/dat_eiger4m_0642_LT_1.cor'
fullpathcorfilename= "/data/visitor/ma6758/bm32/20260312/RAW_DATA/Al/Al_Al650um_daxm_h1.7_expo_5_nbsteps_650_hmicro1p7mm/scan0002/dat_eiger4m_0584_LT_2.cor"
#fullpathcorfilename = os.path.join(d['folder'],'dat_img_0005_4900peaks.cor')
#fullpathcorfilename = os.path.join(d['folder'],'dat_img_0000_LT_0.cor')
corfiledata=readfile_cor(fullpathcorfilename)
nbspotscorfile=len(corfiledata[0])
GT.printgreen(f'Read {nbspotscorfile} spots in .cor file')

In [ ]:
XYcorfile = corfiledata[0][:,2:4]
spots_in_corfile, spots_in_XY_at_depth = GT.getPairsbetweenTwoSets(XYcorfile, XYI_at_depth[:,:2], dist_tolerance=2)
GT.printgreen(f'Found {len(spots_in_corfile)} spots among {nbspotscorfile} in .cor file')

In [ ]:
# pixel XY location on detector of surface grains spots
nbtodisplay = 20

brightest_spots_in_corfile = []
print('spot index in cor file,  [X,Y], [X,Y] cor file')
for _k in range(len(spots_in_corfile)):
    ix_cor = spots_in_corfile[_k]
    brightest_spots_in_corfile.append(ix_cor)
    if _k<nbtodisplay:
        print(ix_cor, XY_at_depth[spots_in_XY_at_depth[_k]], XYcorfile[ix_cor])
print('... And %d others'%(len(spots_in_corfile)-nbtodisplay))

print('\n---------\nIn %s '%fullpathcorfilename)
print('\nSome first brightest spots coming from depth ', depthprobe)
brightest_spots_in_corfile = np.array(brightest_spots_in_corfile)
brightest_spots_in_corfile

In [ ]:
# SOLUTION 1 write a .cor file containing these spots only
_, th, chi, x, y, I , _,CCDcalibDict=readfile_cor(fullpathcorfilename,
                                         output_CCDparamsdict=True,
                                         output_only5columns=True)
# allspots_data, CCDcalibDict
outputcorfilename = 'Grains_at_100microns'
selected_data=[]
for dd in (2*th, chi, x, y, I):
    selected_data.append(np.take(dd, brightest_spots_in_corfile,axis=0))


of = writefile_cor(outputcorfilename, *selected_data, param=CCDcalibDict,
                                            initialfilename=fullpathcorfilename,
                                            dirname_output=d['folder'])
print(os.path.join(d['folder'],of))

In [ ]:
#!ls -alpt {d['folder']}

In [ ]:
!cat {os.path.join(d['folder'],'shortdat_img_0005_4900peaks_grain0.fit')}

In [ ]:
# SOLUTION 2 index by looking at distances between these but matching rate is calculated with all spots

In [ ]:
fullpath = os.path.join(folder,f'{d['prefix']}%04d.{d['suffix']}'%imageindex)

from ipywidgets import interact, interactive, fixed, interact_manual

with fabio.open(fullpath) as img:
    imgdata = img.data
    
fig,ax = plt.subplots(figsize=(6,6))
#fig.suptitle('%s\n%s'%(ExperimentFolder,fullpath.rsplit('/RAW_DATA/')[1]))
#ax.imshow(np.log10(imgdata), vmin = 3, vmax = 3.5, cmap=plt.cm.inferno)
ax.imshow(imgdata, vmin = 1000, vmax =6000, cmap=plt.cm.OrRd)
for pt in XYI_at_depth[:,:2]:
    ax.scatter(pt[0],pt[1],marker='+',color='r')
    
def plotimage(imageindex=0,vmin=1000, vmax=2000, showroi=False):
    # TO IMPROVE  see wrokflow JSM   laueimproc
    
    ymin, ymax = ax.get_ylim()
    xmin, xmax = ax.get_xlim()

    ax.clear()
    
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    
    ymin, ymax, xmin, xmax = int(ymin),int(ymax),int(xmin),int(xmax)
    if ymin>ymax:
        yminc = ymin
        ymin = ymax
        ymax = yminc
    fullpath = os.path.join(folder,f'{d['prefix']}%04d.{d['suffix']}'%imageindex)
    with fabio.open(fullpath) as img:
        imgdata = img.data
    
    fig.suptitle('%s\n%s'%(ExperimentFolder,fullpath.rsplit('/RAW_DATA/')[1]))
    
        
    ax.imshow(imgdata, vmin = vmin, vmax =vmax, cmap=plt.cm.BuGn)
    if showroi:
        for pt in XYI_at_depth[:,:2]:
            ax.scatter(pt[0],pt[1],marker='o',color='yellow', alpha=0.3)
            ax.set_title('%s\n%s'%(ExperimentFolder,fullpath.rsplit('/RAW_DATA/')[1]))
    
interactive(plotimage, imageindex=(0,max(d['listindices'])-1),
            vmin=(500,1010),vmax=(1015,60000), showroi=[True, False])

In [ ]:
XYfound = XYI_at_depth[:,:2]
intensitiesfound = XYI_at_depth[:,2]
# sort by intensity 
ix_sorted = np.argsort(intensitiesfound)[::-1]
XYfound_sorted = np.take(XYfound,ix_sorted,axis=0 )
XYfound_sorted

In [ ]:
# helping function
# find a spot by XY coordinates and locate it XY array
XY = XYfound

Xtest, Ytest=[1160,1825]
Xtest, Ytest=[918,1730]
xcond =np.fabs(XY[:,0]-Xtest)<10
ycond =np.fabs(XY[:,1]-Ytest)<20
cond = np.logical_and(xcond,ycond)
ind_spos= np.where(cond)[0]
Ytest,posdip[ind_spos]


### Reading laue spot XY coordinates belonging to 1 grain from fit file

** set list of indices `spotsofg0_inXY`

In [ ]:
# uncomment to list files in d['folder']
#!ls {d['folder']}

In [ ]:
d['folder']

In [ ]:
'/data/visitor/ma6758/bm32/20260312/RAW_DATA/Al/Al_Al650um_daxm_h1.7_expo_5_nbsteps_650_hmicro1p7mm/scan0002/dat_eiger4m_0584_LT_2_G0_d100microns.fit

In [ ]:

#!cat {os.path.join(d['folder'],'dat_img_0004_LT_3_Zr_g0.fit')}
!cat {os.path.join(d['folder'],'img4_minus1grainZr_Zr_g1.fit')}
!cat {os.path.join(d['folder'],'dat_eiger4m_0584_LT_2_G0_d100microns.fit')}


In [ ]:
# from .fit files 

UB_g0 = np.array([[-0.715649128 , 0.083196626, -0.693582905],
[-0.274818874 ,-0.946425938 , 0.169636022],
[-0.642010804,  0.310566961,  0.701280083]])

UB_g1 = np.array([[ 0.448549611, 0.103010639, -0.887987377],
,[ 0.212069947,  0.953211693  ,0.217324011],
,[ 0.868557547, -0.284717106,  0.406654304]])


UB_g0_lauuenn = np.array([[0.715649128 , -0.083196626, -0.693582905],
[0.274818874 ,0.946425938 , 0.169636022],
[0.642010804,  -0.310566961,  +0.701280083]])

In [ ]:
!pwd

In [ ]:
## read .fit file of 1rst grain indexed
from LaueTools.IOLaueTools import readfile_fit

#"/data/visitor/me1701/bm32/20241113/RAW_DATA/A45/A45_line1daxms/scan0005/dat_img_0000_Al2O3_grain1.fit"
#resfit=readfile_fit(os.path.join(ExperimentFolder,'A45/A45_line1daxms/scan0005','dat_img_0000_Al2O3_grain1.fit'))

#resfit_g0=readfile_fit(os.path.join(d['folder'],'dat_img_0004_LT_3_Zr_g0.fit'))

resfit_g0=readfile_fit(os.path.join(d['folder'],'dat_eiger4m_0584_LT_2_G0_d100microns.fit'))

#img4_minus1grainZr_Zr_g1.fit
#resfit_g1=readfile_fit(os.path.join(d['folder'],'img4_minus1grainZr_Zr_g1.fit'))


In [ ]:
datag0 = resfit_g0[4]
XYg0 = datag0[:,7:9]
XYg0

In [ ]:
spotsofg0_inXY, _ , _= GT.getCommonPts(XY, XYg0, dist_tolerance=1)
spotsofg0_inXY

In [ ]:
np.take(XY, spotsofg0_inXY,axis=0)

### fitting the best line on which are the spots of a single grain

In [ ]:
# LOCATION spots of g0 in posdip - Yexp plot

spotsofgrain = spotsofg0_inXY

#--------------------------------------
Yexp_grain = Yexp[spotsofgrain]
posdip_grain = posdip[s_idx][spotsofgrain]

grain_lines = np.array([Yexp_grain, posdip_grain]).T

Yexp_division_43_min = 950
Yexp_division_43_max = 1150

Yexp_division_32_min = 1850
Yexp_division_32_max = 1850

# wire 4
condYexp4 = Yexp_grain<Yexp_division_43_min
params_w4 = np.polyfit(Yexp_grain[condYexp4], posdip_grain[condYexp4], 1)
grainfunc_4 = np.poly1d(params_w4)

# wire 3
condYexp3 = np.logical_and(Yexp_grain>Yexp_division_43_max,
                           Yexp_grain<Yexp_division_32_min)

params_w3 = np.polyfit(Yexp_grain[condYexp3], posdip_grain[condYexp3], 1)
grainfunc_3 = np.poly1d(params_w3)

figgrain, axgrain = plt.subplots()

X_yexp = np.linspace(-10,2018+10,20)

# w4
axgrain.plot(grain_lines[:,0], grain_lines[:,1], '.',X_yexp, grainfunc_4(X_yexp), '-')
# w3
axgrain.plot(X_yexp, grainfunc_3(X_yexp), '-')

axgrain.set_xlim((-10,2018+10))
axgrain.set_ylim((-10,np.amax(posdip[s_idx])+10))
axgrain.set_xlabel('pixel Yexp Laue Spot')
axgrain.set_ylabel('#image dip position')
axgrain.grid()
axgrain.set_title('Location of spots of grain 0')

for vertline in (Yexp_division_43_min, Yexp_division_43_max, Yexp_division_32_min, Yexp_division_32_max):
    axgrain.axvline(vertline, linestyle='dashed', color='grey')

print('W4 best line for g0: a/450, b')
print(params_w4[0]*450, params_w4[1])
print('W4 best line for g0: a/450, b')
print(params_w3[0]*450, params_w3[1])

In [ ]:
datag1 = resfit_g1[4]
XYg1 = datag1[:,7:9]
XYg1

### Location in depth abacus of spots in .fit file

In [ ]:
# LOCATION spots of g0 in posdip - Yexp plot

spotsofgrain = spotsofg0_inXY

#--------------------------------------
Yexp_grain = Yexp[spotsofgrain]
posdip_grain = posdip[s_idx][spotsofgrain]

grain_lines = np.array([Yexp_grain, posdip_grain]).T

Yexp_division_43_min = 950
Yexp_division_43_max = 1150

Yexp_division_32_min = 1850
Yexp_division_32_max = 1850

# wire 4
condYexp4 = Yexp_grain<Yexp_division_43_min
params_w4 = np.polyfit(Yexp_grain[condYexp4], posdip_grain[condYexp4], 1)
grainfunc_4 = np.poly1d(params_w4)

# wire 3
condYexp3 = np.logical_and(Yexp_grain>Yexp_division_43_max,
                           Yexp_grain<Yexp_division_32_min)

params_w3 = np.polyfit(Yexp_grain[condYexp3], posdip_grain[condYexp3], 1)
grainfunc_3 = np.poly1d(params_w3)

figgrain, axgrain = plt.subplots()

X_yexp = np.linspace(-10,2018+10,20)

# w4
axgrain.plot(grain_lines[:,0], grain_lines[:,1], '.',X_yexp, grainfunc_4(X_yexp), '-')
# w3
axgrain.plot(X_yexp, grainfunc_3(X_yexp), '-')

axgrain.set_xlim((-10,2018+10))
axgrain.set_ylim((-10,np.amax(posdip[s_idx])+10))
axgrain.set_xlabel('pixel Yexp Laue Spot')
axgrain.set_ylabel('#image dip position')
axgrain.grid()
axgrain.set_title('Location of spots of grain 1')

for vertline in (Yexp_division_43_min, Yexp_division_43_max, Yexp_division_32_min, Yexp_division_32_max):
    axgrain.axvline(vertline, linestyle='dashed', color='grey')

print('W4 best line for g0: a/450, b')
print(params_w4[0]*450, params_w4[1])
print('W4 best line for g0: a/450, b')
print(params_w3[0]*450, params_w3[1])

In [ ]:
# spotsofg1_inXY, _ , _= GT.getCommonPts(XY, XYg1, dist_tolerance=1)
# spotsofg1_inXY

### superimpose Laue spots belonging to the same grain

In [ ]:
# GUI   set model 
ADDLAUESPOTMARKERS = True

MinIntensityThreshold = -1000

s_idx = np.where(I>MinIntensityThreshold)[0]

XY = d['peaklist'][s_idx]

Xexp, Yexp = XY.T

# W4  Y from 0 380  top quarter  posdip = 200/(420-90)  Y + 86
# W3  Y 520 - 920
# W2  Y 1060 - 1480
# W1  Y 1570  -2000
Ytheo = np.arange(2048)

WP0 = [400, 525, 435, -485, 440, -955, 435, -1410]  # ma6758

dip4 = WP0[0]/(450)*Ytheo +WP0[1]
dip3 = WP0[2]/(450)*Ytheo +WP0[3]
dip2 = WP0[4]/(450)*Ytheo +WP0[5]
dip1 = WP0[6]/(450)*Ytheo +WP0[7]


figdd, axdd = plt.subplots()
axdd.plot(Yexp,posdip[s_idx], 'o')  # exp dip
axdd.plot(Ytheo,dip4, '-r')
axdd.plot(Ytheo,dip3, '-r')
axdd.plot(Ytheo,dip2, '-r')
axdd.plot(Ytheo,dip1, '-r')

axdd.set_xlim(Yexp[0]-10, Yexp[-1]+10)
axdd.set_ylim(np.amin(posdip[s_idx])-10, np.amax(posdip[s_idx])+10)


def addmarkeronspots(listindices, arraypts, ax, color='k'):
    selectedspots = np.take(arraypts,listindices, axis=0)
    _, Yselected = selectedspots.T
    
    selectedposdip = posdip[s_idx][listindices]
    ax.scatter(Yselected, selectedposdip, marker='x',c=color, s=200, alpha=0.5)
    
    #print(Yselected, selectedposdip)
    
if ADDLAUESPOTMARKERS:
    
    addmarkeronspots(spotsofg0_inXY, XY, axdd)
   # addmarkeronspots(spotsofg1_inXY, XY, axdd, color='g')

def plotposdip(w4_slope=WP0[0], w4_origordinate=WP0[1],
              w3_slope=WP0[2], w3_origordinate=WP0[3],
              w2_slope=WP0[4], w2_origordinate=WP0[5],
              w1_slope=WP0[6], w1_origordinate=WP0[7], showspotindex=False):
    
    ymin, ymax = axdd.get_ylim()
    xmin, xmax = axdd.get_xlim()

    axdd.clear()
    
   
    # --------------------------
    
#     def update_annot(ind):
    
#         pos = sc.get_offsets()[ind["ind"][0]]
#         annot.xy = pos
#         text = "{}, {}".format(" ".join(list(map(str,ind["ind"]))), 
#                                " ".join([names[n] for n in ind["ind"]]))
#         annot.set_text(text)
#         annot.get_bbox_patch().set_facecolor(cmap(norm(c[ind["ind"][0]])))
#         annot.get_bbox_patch().set_alpha(0.4)
    

#     def hover(event):
#         vis = annot.get_visible()
#         if event.inaxes == axdd:
#             cont, ind = sc.contains(event)
#             if cont:
#                 update_annot(ind)
#                 annot.set_visible(True)
#                 fig.canvas.draw_idle()
#             else:
#                 if vis:
#                     annot.set_visible(False)
#                     fig.canvas.draw_idle()

#     figdd.canvas.mpl_connect("motion_notify_event", hover)
    
    #--------------------------------
    dip4 = w4_slope/450.*Ytheo + w4_origordinate
    dip3 = w3_slope/450.*Ytheo + w3_origordinate
    dip2 = w2_slope/450.*Ytheo + w2_origordinate
    dip1 = w1_slope/450.*Ytheo + w1_origordinate
    
    axdd.plot(Yexp,posdip[s_idx], 'o', alpha=0.1)  # exp dip
    axdd.plot(Ytheo,dip4, '-r')
    axdd.plot(Ytheo,dip3, '-r')
    axdd.plot(Ytheo,dip2, '-r')
    axdd.plot(Ytheo,dip1, '-r')
    axdd.grid()
    axdd.set_xlabel('Y laue spot')
    axdd.set_ylabel('position of dip (image index)')
    axdd.set_ylim(0, tr.shape[1])
    
    if ADDLAUESPOTMARKERS:
    
        addmarkeronspots(spotsofg0_inXY, XY, axdd)
        #addmarkeronspots(spotsofg1_inXY, XY, axdd, color='g')

    print('W4,W3, W2, W1 parameters:')
    print([w4_slope, w4_origordinate, w3_slope,w3_origordinate,
              w2_slope, w2_origordinate,w1_slope,w1_origordinate])
    
    if showspotindex:
        # add annotations one by one with a loop
        for _k, _yexp in enumerate(Yexp):
            alldata_txt = '#spot %d %.1f, %.1f'%(_k, Xexp[_k],_yexp)
            index_txt = '%d'%_k
            
            annottxt = index_txt
            axdd.text(
                  _yexp+0.2,
                  posdip[s_idx][_k],
                  annottxt,
                  ha='left',
             )
                
    axdd.set_xlim(xmin, xmax)
    axdd.set_ylim(ymin, ymax)

interactive(plotposdip, w4_slope=(100,600,5), w4_origordinate=(-500,600,5),
           w3_slope=(100,600,5), w3_origordinate=(-800,200,5),
           w2_slope=(100,600,5), w2_origordinate=(-1000,0,5),
           w1_slope=(100,600,5), w1_origordinate=(-1500,0,5), showspotindex=[True, False])

### capture spots close to a straight line having similar depth under surface

In [ ]:
def distance(listpoint,coef):
    """compute ortho distance along vertical direction  ie imageindex """
    point= np.array(listpoint)
    return np.fabs((coef[0]*point[:,0])-point[:,1]+coef[1])/np.sqrt((coef[0]*coef[0])+1)

def vdistance(listpoint,coef):
    """compute distance along vertical direction  ie imageindex """
    point= np.array(listpoint)
    return np.fabs(point[:,1]-(coef[0]*point[:,0]+coef[1]))

distance([[30,46],[22,61]], (290./(450),20)), vdistance([[30,46],[22,61]], (290./(450),20))


In [ ]:
TabYexpPosDip = np.array([Yexp,posdip[s_idx]]).T
# dip4  ie at left side of graph, small Y pixel value, so wire shadow at very top of the laue camera
distance(TabYexpPosDip, (280./(450),55)), vdistance(TabYexpPosDip, (300./(450),-615))

In [ ]:
# wire4
surf4 = np.where(vdistance(TabYexpPosDip, (280./(450),55))<5)[0]
surf4

In [ ]:
# wire3
surf3 = np.where(vdistance(TabYexpPosDip, (300./(450),-615))<5)[0]
surf3

In [ ]:
XY[surf4],intensities[surf4]

In [ ]:
# concatenation

XYsurf = np.concatenate((XY[surf4], XY[surf3]))
XYsurf

In [ ]:
## read .cor file of 1rst grain indexed
from LaueTools.IOLaueTools import readfile_cor

rr = readfile_cor(os.path.join(d['folder'],'img4_minus2_Zrgrains.cor'),
                                           output_CCDparamsdict=True)

CCDcalibdict = rr[-1]
datacor = rr[:-1]

In [ ]:
def getPairsbetweenTwoSets(XY1, XY2, dist_tolerance=0.5, samelist=False):
    """
    return indices in XY1 and in XY2 of common pts (2D) and
    a flag is closest distances are below dist_tolerance

    :param XY1: list of 2D elements
    :param XY2: list of 2D elements
    :param dist_tolerance: largest distance (in unit of XY1, XY2) to consider two elements close enough
    :param samelist: boolean, default is False (when XY1 and XY2 are different). True if XY1=XY2 to
    find close spots in a single list of points

    :return:
    ind_XY1, ind_XY2: two arrays of indices which connect elementwise one element of XY1 to 1 element of XY2
    """
    x1, y1 = np.array(XY1).T

    x2, y2 = np.array(XY2).T

    diffx = x1[:, np.newaxis] - x2
    diffy = y1[:, np.newaxis] - y2

    _dist = np.hypot(diffx, diffy)

    if samelist:
        # add big distance in diagonal
        np.fill_diagonal(_dist, np.amax(_dist)+2*dist_tolerance)

    conddist = _dist <= dist_tolerance
    in1, in2 = np.where(conddist==True)
        
    return in1, in2



In [ ]:
XYIcor = datacor[0][:,2:5]
tthchi = datacor[0][:,:2]

spots_in_corfile, spots_in_XYsurf = getPairsbetweenTwoSets( XYIcor[:,:2],XYsurf, dist_tolerance=2)


In [ ]:
spots_in_XYsurf

In [ ]:
# pixel XY location on detector of surface grains spots
for _k in range(50):
    ix_cor = spots_in_corfile[_k]
    print(ix_cor, XYsurf[spots_in_XYsurf[_k]], XYIcor[ix_cor])

In [ ]:
# spots to keep or kill to improve indexing

# to keep
setspots_in_surface = set(spots_in_corfile.tolist())

# to kill
setspots_not_in_surface = set(range(len(XYIcor)))-setspots_in_surface

np.array(setspots_not_in_surface)

In [ ]:
alldata = datacor[0]
alldata_keep = np.take(alldata, list(setspots_in_surface), axis=0)
alldata_keep

In [ ]:

fileprefix = 'img4_surfacespots'
IOLT.writefile_cor(fileprefix, alldata_keep[:,0], alldata_keep[:,1], alldata_keep[:,2], alldata_keep[:,3], alldata_keep[:,4],
                    data_props=None, # add_props,
                    param=CCDcalibdict,
                    overwrite=1,
                  dirname_output = d['folder'])

### IN DVPT: tentative converting image index, wire #index to absolute depth

In [ ]:

Yexp = XY[:,1]

#roughdepth(Yexp, posdip)   # implicitly Wireparams are known from Ge calibration

figddd, axddd = plt.subplots()
                                     
axddd.plot(Yexp,roughdepth(Yexp, posdip), 'o')  

axddd.grid()
axddd.set_xlabel('Y laue spot')
axddd.set_ylabel('estimated depth (a. u.)')
axddd.set_title('Wire calibration from Ge')

### better model  (but still not very accurate from DAXM analysis)

In [ ]:
def getdepth(Y, w,w0=-200,stepw=0.001, H=1,dd=79,Ycen=900,pixelsize=0.0734):
    depth = stepw*(w-w0)/(Ycen-Y)/pixelsize + H/(dd-H)*(pixelsize*(Ycen-Y)+stepw*(w-w0))
    return depth

In [ ]:
print('all estimated depth')
idx_counter = 0
tabdips = []
for y, centerdip in zip(d['peaklist'][:,1],posdip):
    depth = getdepth(y,centerdip)
    tabdips.append([idx_counter,y,depth])
    idx_counter+=1
    
tabdips=np.array(tabdips)
tabdips

In [ ]:
# GUI to find the raw wire parameters ...

# need ti integrate sCMOS geometry calibration
# need to add the straight line trajectory angle 

figee, axee = plt.subplots()
axee.plot(tabdips[:,1],tabdips[:,2])
axee.grid()
axee.set_xlabel('index laue spot')

def plotdepth(w0=618, H=.7,dd=79,Ycen=900):
    
    axee.clear()
    def getdepth(Y, w,w0=w0,stepw=0.001, H=H,dd=dd,Ycen=Ycen,pixelsize=0.0734):
        depth = stepw*(w-w0)/(Ycen-Y)/pixelsize + H/(dd-H)*(pixelsize*(Ycen-Y)+stepw*(w-w0))
        return depth
    print('all estimated depth')
    idx_counter = 0
    tabdips = []
    for y, centerdip in zip(d['peaklist'][:,1],posdip):
        depth = getdepth(y,centerdip)
        tabdips.append([idx_counter,y,depth])
        idx_counter+=1
    
    tabdips=np.array(tabdips)
    
    axee.plot(tabdips[:,1],tabdips[:,2],'o')
    axee.grid()
    axee.set_xlabel('Y laue spot')
    axee.set_ylabel('depth')
    axee.set_ylim(-2,2)

interactive(plotdepth, w0=(-300,1000),H=(0.6,1.2,0.05),dd=(75.,85.,0.05),Ycen=(0,2000,1))

In [ ]:
# find major positive and negative step
bigsteps=np.where(np.fabs(bigdiff3)>.8,1,0)
bigsteps[58]

In [ ]:
fig5,ax5=plt.subplots()
ax5.imshow(bigsteps)

In [ ]:
# nb of steps fro each wire position
nbsteps=np.count_nonzero(bigsteps,axis=0)
nbsteps.shape

In [ ]:
nbsteps

In [ ]:
wireindex = 2  # ? / 600
# find the 16 lauespot (roicounter index) with largest step for a given depth or wire position thelargest step st wire position with largest
np.argsort(bigdiff[:,wireindex])[:16]

In [ ]:
bigdiff[:,2][579]

In [ ]:
# normally lauespot 579 and 643 mesut be shadowed fro the same wire position => should belon to the same grain

In [ ]:
def getspotindexclose(x,y, peaklist, tolerance=1):
    condx = (peaklist[:,0]-x)**2<tolerance**2
    condy = (peaklist[:,1]-y)**2<tolerance**2
    pos = np.where(np.logical_and(condx,condy))[0]
    
    if len(pos)==0:
        return None
    
    xyclose=peaklist[pos]
    return pos, xyclose

getspotindexclose(180,996,d['peaklist'], tolerance=5)

In [ ]:
xy=d['peaklist']
getspotindexclose(1026,1042,xy, tolerance=5)

In [ ]:
d['peaklist'][87], d['peaklist'][88],d['peaklist'][127], d['peaklist'][128]

In [ ]:
d['peaklist'][307]

# Find grains (from 2D map)

### monitor data if needed

set data to variable `monitor_data`

In [ ]:
d

In [ ]:
# reading monitor count for normalisation and plot ...
MONITORDATA_AVAILABLE = False
if 1:
    # read h5file for monitor correction...
    import h5py

    with h5py.File(d['localhdf5file'], 'r', locking=False) as f:
        monitor_data = f[f'{d['scanindex']}.1/measurement/mon'][()]
    #scanObj=iohdf5.Scan_hdf5(pathHDF5, d['sample_dataset_scanindex'], collectallscans=True)


    #logfilereader.getscanprops_lowest_hdf5(hdf5fullpath, 'P3_TTh1_longdia_3')
    
    #monitor_data = scanObj.getcounterdata('mon')

    fig000, ax000 = plt.subplots(figsize=(6,3))
    ax000.plot(monitor_data)
    ax000.set_xlabel('image index')
    ax000.set_ylabel('monitor intensity')
    ax000.set_title(f"Monitor \n"+d['sample_dataset_scanindex'])
    ax000.grid()

    MONITORDATA_AVAILABLE = True

    
    nbimagesperline = d['mapdimensions'][0]
    monitor_data2D = monitor_data.reshape((-1,nbimagesperline))
    print(monitor_data2D.shape)


## segmentation of roi counter map

In [ ]:
import scipy.stats as st
import scipy.ndimage as ndimage

def proprosethreshold(data, alpha, useskewness = True, skewnessthreshold=0.2, removeminimum=True, verbose=False):
    """from array of pixel intensity, and its histogram, return intensity in between most frequent intensity and max intensity.
    useskewness: True, for small area grain or object
    
    If skewness is too small, the proposed threshold is None"""
    if np.all(data == 0):
        return None
    
    if removeminimum:
        dd = data[data>np.amin(data)]
    else:
        dd = data

    
    freq, bins = np.histogram(dd, bins=40)
    binsize = bins[1]-bins[0]
    mostfreqintensity = bins[np.argmax(freq)]+binsize/2
    maxvalue = np.amax(dd)
    minval = (1-alpha)*mostfreqintensity + alpha*maxvalue

    skewness = st.skew(dd, axis=None)
    if skewness<skewnessthreshold:
        minval = None
    if verbose:
        print('mostfreqintensity', mostfreqintensity)
        print('max intensity', np.amax(dd))
        print('minval automatic value:', minval)
    return minval

In [ ]:
# dictlut=['Accent', 'Accent_r', 'Blues', 'Blues_r', 'BrBG', 'BrBG_r', 'BuGn', 'BuGn_r', 
#          'BuPu', 'BuPu_r', 'CMRmap', 'CMRmap_r', 'Dark2', 'Dark2_r', 'GnBu', 'GnBu_r', 
#          'Greens', 'Greens_r', 'Greys', 'Greys_r', 'OrRd', 'OrRd_r', 'Oranges', 
#          'Oranges_r', 'PRGn', 'PRGn_r', 'Paired', 'Paired_r', 'Pastel1', 'Pastel1_r',
#          'Pastel2', 'Pastel2_r', 'PiYG', 'PiYG_r', 'PuBu', 'PuBuGn', 'PuBuGn_r', 
#          'PuBu_r', 'PuOr', 'PuOr_r', 'PuRd', 'PuRd_r', 'Purples', 'Purples_r', 'RdBu', 
#          'RdBu_r', 'RdGy', 'RdGy_r', 'RdPu', 'RdPu_r', 'RdYlBu', 'RdYlBu_r', 'RdYlGn', 
#          'RdYlGn_r', 'Reds', 'Reds_r', 'Set1', 'Set1_r', 'Set2', 'Set2_r', 'Set3', 'Set3_r', 
#          'Spectral', 'Spectral_r', 'Wistia', 'Wistia_r', 'YlGn', 'YlGnBu', 'YlGnBu_r', 
#          'YlGn_r', 'YlOrBr', 'YlOrBr_r', 'YlOrRd', 'YlOrRd_r', 'afmhot', 'afmhot_r', 
#          'autumn', 'autumn_r', 'binary', 'binary_r', 'bone', 'bone_r', 'brg', 'brg_r', 
#          'bwr', 'bwr_r', 'cividis', 'cividis_r', 'cool', 'cool_r', 'coolwarm', 'coolwarm_r', 
#          'copper', 'copper_r', 'cubehelix', 'cubehelix_r', 'flag', 'flag_r', 'gist_earth', 
#          'gist_earth_r', 'gist_gray', 'gist_gray_r', 'gist_heat', 'gist_heat_r', 'gist_ncar', 
#          'gist_ncar_r', 'gist_rainbow', 'gist_rainbow_r', 'gist_stern', 'gist_stern_r', 
#          'gist_yarg', 'gist_yarg_r', 'gnuplot', 'gnuplot2', 'gnuplot2_r', 'gnuplot_r', 
#          'gray', 'gray_r', 'hot', 'hot_r', 'hsv', 'hsv_r', 'inferno', 'inferno_r', 'jet',
#          'jet_r', 'magma', 'magma_r', 'nipy_spectral', 'nipy_spectral_r', 'ocean', 
#          'ocean_r', 'pink', 'pink_r', 'plasma', 'plasma_r', 'prism', 'prism_r', 
#          'rainbow', 'rainbow_r', 'seismic', 'seismic_r', 'spring', 'spring_r', 'summer', 'summer_r', 
#          'tab10', 'tab10_r', 'tab20', 'tab20_r', 'tab20b', 'tab20b_r', 'tab20c', 'tab20c_r', 
#          'terrain', 'terrain_r', 'turbo', 'turbo_r', 'twilight', 'twilight_r',
#          'twilight_shifted', 'twilight_shifted_r', 'viridis', 'viridis_r', 'winter', 'winter_r']

In [ ]:
print('nb rois, nb images',tr.shape)
nbimagesperline = d['mapdimensions'][0]

In [ ]:
idx_counter0 = 2104 # 354 # 1287 # 1651
vmax = None  # max plot intensity or None (autocontrast) 
Threshold = 'auto'
alpha0 = 0.65

#----------------------------------
import scipy.stats as st

nbrois = tr.shape[0] # nb of rois
if idx_counter0>=nbrois: # 
    GT.printred(f'ERROR:\n"idx_counter0"={idx_counter0} exceeds the largest index of rois {nbrois-1}')
dd0= tr[idx_counter0]
dd0_2d = dd0.reshape((-1,nbimagesperline))
if vmax is not None:
    _vmax= max(np.amin(dd0), vmax)
else:
    _vmax = vmax

#print('tr.shape', tr.shape)
bigtr = tr.reshape((tr.shape[0],-1,nbimagesperline))
#print('bigtr.shape', bigtr.shape)

im = bigtr[idx_counter0]

if MONITORDATA_AVAILABLE:
    im = im / monitor_data2D
fig102, ax102 = plt.subplots(1,3, figsize=(8,5))
ax102[0].imshow(im, origin='lower', vmax=_vmax)
ax102[0].set_title(f'roi counter #{idx_counter0}')

freq, bins, _ = ax102[1].hist(dd0, bins=40)
ax102[1].grid()
binsize = bins[1]-bins[0]
mostfreqintensity = int(bins[np.argmax(freq)]+binsize/2)
std = np.std(im)
skewness = st.skew(im, axis=None)
txttitle = f'most frequent intensity around : {mostfreqintensity}\n with binsize {binsize:.0f}\n'
txttitle += f'std: {std:.0f}  maxval: {np.amax(im):.0f} skewness {skewness:.2f}'
pp = ax102[1].set_title( txttitle)

# thresholding map by map
if isinstance(Threshold, (int, float)):
    minval = Threshold
elif Threshold in ('auto',):
    if alpha0<0 or alpha0 >1:
        GT.printred(f'ERROR:\nalpha {alpha0} is not comprised between 0 and 1\n')
    minval=proprosethreshold(im, alpha0, skewnessthreshold=0.2)
    GOODSKEWNESS = True
    if minval is None:
        minval = np.amax(im) +1
        GOODSKEWNESS = False
    
mask = np.where(im > minval, 1, 0)
# connectivity
s_compact = [[1, 1, 1], [1,1,1], [1,1,1]]

label_im, nb_labels = ndimage.label(mask,structure=s_compact)
if nb_labels>0:
    totalsurface= np.sum(mask)
    meansurface=totalsurface/nb_labels

ax102[2].imshow(label_im, origin='lower', cmap='nipy_spectral_r')
txttitle3 = f'roi counter #{idx_counter0}\nnb of regions {nb_labels}'
if nb_labels>0:
    txttitle3+=f'\npixel**2 surface: total {totalsurface}, average/region {meansurface:.1f}'
ax102[2].set_title(txttitle3)

def plotmaphist(roiindex=idx_counter0, minimumskewness=0.2, alpha=alpha0):
    global current_roiindex, current_minimumskewness, current_alpha
    
    current_roiindex = roiindex
    current_minimumskewness = minimumskewness
    current_alpha = alpha
    
    dd0= tr[roiindex]
    dd0_2d = dd0.reshape((-1,nbimagesperline))
    if vmax is not None:
        _vmax= max(np.amin(dd0), vmax)
    else:
        _vmax = vmax
    
    #print('tr.shape', tr.shape)
    #bigtr = tr.reshape((tr.shape[0],-1,nbimagesperline))
    #print('bigtr.shape', bigtr.shape)
    
    im =dd0_2d # bigtr[roiindex]
    if MONITORDATA_AVAILABLE:
        im = im / monitor_data2D


    print('im.shape', im.shape)
    
    ax102[0].clear()
    ax102[1].clear()
    ax102[2].clear()

    
    ax102[0].imshow(im, origin='lower', vmax=_vmax)
    pp1 =  ax102[0].set_title(f'roi counter #{roiindex}')
    # kill spurious minimum value
    
    immodified = im[im>np.amin(im)]
    freq, bins, _ = ax102[1].hist(immodified.ravel(), bins=40)
    # print('freq', freq)
    # print('bins', bins)
    ax102[1].grid()
    binsize = bins[1]-bins[0]
    # print('binsize', binsize)
    # print('np.argmax(freq)',np.argmax(freq))
    mostfreqintensity = bins[np.argmax(freq)]+binsize/2
    std = np.std(immodified)
    skewness = st.skew(immodified, axis=None)
    
    txttitle = f'most frequent intensity:{mostfreqintensity:.4f}'
    #txttitle +=f'\n with binsize {binsize:.0f}'
    txttitle += f'\nstd: {std:.2f}\nmaxval: {np.amax(im):.2f}\nskewness {skewness:.2f}'
    ax102[1].text(0.5, 0.85, txttitle,
                        horizontalalignment='center',
                        verticalalignment='center',
                        fontsize=8, color='red',
                        transform=ax102[1].transAxes)
    
    # pp2 = ax102[1].set_title( txttitle)

    # thresholding map by map
    if isinstance(Threshold, (int, float)):
        minval = Threshold
    elif Threshold in ('auto',):
        if alpha<0 or alpha >1:
            GT.printred(f'ERROR:\nalpha {alpha} is not comprised between 0 and 1\n')
        minval=proprosethreshold(im, alpha, skewnessthreshold=minimumskewness)
        GOODSKEWNESS = True
        if minval is None:
            
            minval = np.amax(im) +1
            GOODSKEWNESS = False
    mask = np.where(im > minval, 1, 0)
    # connectivity
    s_compact = [[1, 1, 1], [1,1,1], [1,1,1]]
    
    label_im, nb_labels = ndimage.label(mask,structure=s_compact)
    if nb_labels>0:
        totalsurface= np.sum(mask)
        meansurface=totalsurface/nb_labels

    ax102[1].axvline(minval, color='black', ls='--')
    
    ax102[2].imshow(label_im, origin='lower', cmap='nipy_spectral_r')
    #txttitle3 = f'roi counter #{roiindex}'
    txttitle3 = f'\nnb of regions {nb_labels}'
    if nb_labels>0:
        txttitle3+=f'\nsurface: total {totalsurface}\naverage/region {meansurface:.1f}'
    pp3 = ax102[2].set_title(txttitle3)
    if not GOODSKEWNESS:
        ax102[2].text(0.5, 0.85, 'SKEWNESS too SMALL',
                        horizontalalignment='center',
                        verticalalignment='center',
                        fontsize=15, color='red',
                        transform=ax102[2].transAxes)

    plt.show()

interactive(plotmaphist, roiindex=(0,nbrois-1), minimumskewness=(-1,20,.1), alpha=(0,1,0.05))

### segmentation of a single roi counter map

pedagogy:
* use of `find_objects()`
* defintion of `regionmap` (2D array with a single region). For a given map, several `regionmap` can be built.

In [ ]:
# segment and plot  (to play and test)
if 1:
    idx_counter0 = current_roiindex # 2104 #150 #1369 #1742 #1274
    
    
    #Threshold = 6400
    Threshold = 'auto'
    minskewness = current_minimumskewness  #3
    alpha = current_alpha #.3
    
    
    
    #-----------------------------------------    
    dd=bigtr[idx_counter0]

    if MONITORDATA_AVAILABLE:
        dd = dd / monitor_data2D

    immodified = dd[dd>np.amin(dd)]
    # thresholding map by map
    if isinstance(Threshold, (int, float)):
        minval = Threshold
    elif Threshold in ('auto',):
        if alpha<0 or alpha >1:
            GT.printred(f'ERROR:\nalpha {alpha} is not comprised between 0 and 1\n')
        minval=proprosethreshold(dd, alpha,skewnessthreshold=minskewness,removeminimum=True)
        GOODSKEWNESS = True
        if minval is None:
            minval = np.amax(im) +1
            GOODSKEWNESS = False

        print('minval',minval)
    mask = np.where(dd > minval, 1, 0)
    # connectivity
    s_compact = [[1, 1, 1], [1,1,1], [1,1,1]]
    
    label_im, nb_labels = ndimage.label(mask,structure=s_compact)
    print('nb regions found', nb_labels)
    if nb_labels>0:
        totalsurface= np.sum(mask)
        meansurface=totalsurface/nb_labels
    # grain index = 0 = not a grain!
    
    figa, axa = plt.subplots()
    axa.imshow(label_im, origin='lower', cmap='nipy_spectral_r')
    txttitle = f'roi counter #{idx_counter0}\nnb of regions {nb_labels}'
    if nb_labels>0:
        txttitle+=f'\npixel**2 surface: total {totalsurface}, average/region {meansurface:.1f}'
    axa.set_title(txttitle)


In [ ]:
# region index starts from 1 (0 = background ie below threshold)
# and increased by increasing first index value on image

REGION_MINSURFACE = 4   # minimal segmented area in sample map pixel**2 unit
REGION_MAXSURFACE = 20  # maximal segmented area in sample map pixel**2 unit

label_im_purged = np.copy(label_im)
regions = ndimage.find_objects(label_im_purged)
print('Found regions',regions)

remainregion = nb_labels
for idx_region in range(len(regions)):
    regionmap = np.where(label_im_purged==idx_region+1, 1,0)
    regionsurface = np.sum(regionmap)
    #print(idx_region, regionsurface)
    if regionsurface<REGION_MINSURFACE or regionsurface>REGION_MAXSURFACE:
        label_im_purged[regions[idx_region]]=0
        remainregion=remainregion-1

remaining_regions = np.unique(label_im_purged)
print('-------\nremaining labeled regions', remaining_regions)
print('0 is the background\n-------')
figapurged, axapurged = plt.subplots()
axapurged.imshow(label_im_purged, origin='lower', cmap='nipy_spectral_r')
txttitle = f'PURGED roi counter #{idx_counter0}\nnb of regions {remainregion}'
axapurged.set_title(txttitle)

In [ ]:
# idx_region = 0 is background for small grains in the map
idx_region = 4  # label_index 

#------------------------------
if float(idx_region) not in remaining_regions:
    GT.printyellow(f'region {idx_region} is not in remaining regions list: {remaining_regions}')
else:
    regionmap = np.where(label_im==idx_region, idx_region,0)
    figa, axa = plt.subplots()
    axa.imshow(regionmap, origin='lower', cmap='nipy_spectral_r')
    txttitle = f'roi counter #{idx_counter0}\n#region {idx_region}'
    regionsurface = np.round(np.sum(regionmap)/(idx_region),1)
    txttitle+=f'\npixel**2 region surface {regionsurface}'
    axa.set_title(txttitle)

### from  Sergio

In [ ]:
import cv2 as cv
from copy import deepcopy

def to_binary_image(positions, shape):
    """Returns a matrix of given shape, where
        matrix[i,j] = 1, if peak is present in image[i,j]
        matrix[i,j] = 0 otherwise
        
    Parameters
    ----------
    positions  np.ndarray: flattened array with the index of the images where a given peak was found
    shape           tuple: size of the resulting image in pixels
    
    Example
    ----------
    >>> to_binary_image(np.array([3,5,7]), (3,3))
    array([[0, 0, 0],
           [1, 0, 1],
           [0, 1, 0]], dtype=uint8)
    """
    positions = positions.astype(np.uint64)
    
    image = np.zeros(shape, dtype=np.uint8)
    i = positions // shape[0]
    j = positions  % shape[1]
    image[i,j] = 1   
    return image

def _largest_cc_label(stats):
    """Return the label of the largest connected component, after the background. 
    (cv2.connectedComponentWithStats() does not guarantee that the largest component
    has index 1)
    
    Parameters
    ----------
    stats  np.ndarray: statistics on the result of connected component analysis 
                       as returned by cv2.connectedComponentWithStats()

    Returns
    ----------
    label    np.int64: label value, i.e. the index of the row of the stats array
                       corresponding to the largest connected component
    """
    # Order goes from smallest to largest. 
    # The background is expected to have the largest area, so the largest blob is the second last
    # (Background index would be -1)                                                     │
    return np.argsort(stats[:, cv.CC_STAT_AREA])[-2] # <─────────────────────────────────┘

def _get_cc(labeled_image, label):
    """Return only the components with the specified label.
    
    Parameters
    ----------
    labeled_image  np.ndarray: labeled image resulting from connected component analysis
                               as returned by cv2.connectedComponentWithStats()
                               
    Returns
    ----------
    out            np.ndarray: out[i,j] = 1 if (i,j) in component, 0 otherwise
    """
    return np.where(labeled_image != label, 0, 1).astype(np.uint8)

def get_largest_cc(image):
    """Perform connected component analysis and return a binary image with the largest
    component (after the background).

    Parameters
    ----------
    image    np.ndarray: array with dtype np.uint8 containing image data

    Returns:
    out      np.ndarray: array with data of the largest connected component
    """
    if np.count_nonzero(image) > image.size/2:
        return image # if the image has more ones than zeros do nothing
    
    num_labels, labeled_image, stats, centroids = cv.connectedComponentsWithStats(image)
    return _get_cc(labeled_image, _largest_cc_label(stats))
    
def filter_by_length(families, length=100, inplace=True):
    if not inplace:
        families = deepcopy(families)
    
    to_remove = []
    for i, family in enumerate(families):
        if len(family) < length:
            to_remove.append(i)
    
    for i in reversed(to_remove):
        del families[i]
        
    if not inplace:
        return families

def filter_by_largest_cc_size(images, minsize=None, return_excluded=False):
    # If not specified, choose 1% of the image area
    if minsize is None:
        minsize = int(0.01 * images.shape[1] * images.shape[2])
        
    to_remove = []
    for i, image in enumerate(images):
        # Compute connected components
        num_labels, labeled_image, stats, centroids = cv.connectedComponentsWithStats(image)
        # If largest component is too small, store its position in the image stack 
        if stats[_largest_cc_label(stats), cv.CC_STAT_AREA] < minsize:
            to_remove.append(i)       
    to_remove = np.array(to_remove)
    
    if return_excluded:
        return np.delete(images, to_remove, axis=0), images[to_remove]
    
    return np.delete(images, to_remove, axis=0)

## manual selection of a map point and find other roi counters with scattering intensity: `rawlistidx`

### single threshold applied for all maps on a single map point at `xmap,ymap` integer position on the map (xmap fast scan direction index)

In [ ]:
# NOT USED afterwards:
# roi counter which are fired at map coordinates (rather indices) x, y = xmap, ymap
#listidx = np.where(bigtr[:, 41,94]>Threshold)[0]
Threshold =500    # intensity in cts on Laue detector to select bright scattering spots

xmap, ymap =d['mapdimensions'][0]//2,d['mapdimensions'][1]//2 #
xmap, ymap = , 60  # 
img_idx = convertindex2Dto1D(xmap, ymap,d['mapdimensions'])
print('corresponding image index',img_idx)
rawlistidx = np.where(bigtr[:, ymap,xmap]>Threshold*1.1)[0]
print('nb of roi counters found', len(rawlistidx))
rawlistidx

## adaptative threshold to segment all invidual sample map seen by a roi counter.

For a segmented map with n found regions, it will create n new map `regionmap` with only 1 region present.

`all_labeled_im` is a list of all regionmap
`regions` is a an array of region properties (links to roi, local region, surface)

In [ ]:
# TODO  to parammemize

skewnessthreshold = current_minimumskewness  #3
alpha = current_alpha #.3

# REGION_MINSURFACE = 150  # minimal segmented region area in pixel*pixel
# REGION_MAXSURFACE = 1000  # maximal segmented regin area in pixel*pixel

#--------------------------------------
thresh_list = np.zeros(nbrois)
regionsanalysis = np.zeros((nbrois,4))
all_labeled_im = []
links_region_roi_region = []
regioncounter = 0
for roi_idx in range(nbrois):
    if roi_idx % (nbrois//10)==0: print('roiindex',roi_idx, "/",nbrois)
    dd = bigtr[roi_idx]
    minval = proprosethreshold(dd, alpha=alpha, skewnessthreshold=skewnessthreshold, verbose=False)
    GOODSKEWNESS = True
    if minval is None:
        minval = np.amax(im) +1
        GOODSKEWNESS = False
        
    thresh_list[roi_idx] = minval
    if not GOODSKEWNESS:
        continue
        
    mask = np.where(dd > minval, 1, 0)
    label_im, nb_labels = ndimage.label(mask,structure=s_compact)
    if nb_labels>0:
        Xroi, Yroi = d['peaklist'][roi_idx]
        totalsurface= np.sum(mask)
        meansurface=totalsurface/nb_labels
        regionsanalysis[roi_idx] = [roi_idx, totalsurface,nb_labels,meansurface]
        regions = ndimage.find_objects(label_im)
        for iregion in range(len(regions)):
            
            regionmap = np.where(label_im==iregion+1, 1,0)
            regionsurface= np.sum(regionmap)
            if regionsurface<REGION_MINSURFACE or regionsurface>REGION_MAXSURFACE:
                continue
            meanpositionji = ndimage.center_of_mass(regionmap)
            jc, ic = meanpositionji
            fastindex, slowindex  = int(ic),int(jc)
            centralimageindex = convertindex2Dto1D(fastindex, slowindex, d['mapdimensions'])
            links_region_roi_region.append([roi_idx, iregion, regioncounter, regionsurface, fastindex, slowindex, centralimageindex, Xroi, Yroi])
            all_labeled_im.append(regionmap)
            regioncounter += 1

GT.printgreen('Computation completed!!')
            
regionsanalysis = np.array(regionsanalysis)

regions = np.array(links_region_roi_region)
all_labeled_im = np.array(all_labeled_im)
regionidx_sorted = np.argsort(regions[:,3])[::-1]  # index in all regions list
#thresh_list, regionsanalysis

### sort regions by decreasing surface

In [ ]:
regions.shape, regions[regionidx_sorted].shape, regions[regionidx_sorted][0]

In [ ]:
# first N largest regions found
# Nlargest = 10
# bestregion = regionidx_sorted[:Nlargest]
# print('roiindex, localroiregionindex, regionindex, surface, fastaxisindex, slowaxisindex, centralimageindex')
# print(regions[bestregion])

print('Sorted regions by surface')
dfsortedregions = pd.DataFrame(regions[regionidx_sorted], columns=['roiindex', 'localroi_regionindex', 'regionindex','surface',
                                                             'fastaxisindex', 'slowaxisindex', 'centralimageindex','Xroi', 'Yroi'])
dfsortedregions

In [ ]:
if 1: # histogram of surface distribution
    figh, axh = plt.subplots()
    axh.hist(regions[regionidx_sorted][:,3], bins=40)
    axh.grid()

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt

# Assuming all_labeled_im, regionidx_sorted, and regions are already defined

# Create the initial figure and axis
figb, axb = plt.subplots()
plt.show(block=False)  # Display the figure without blocking

# Function to update the plot based on rankindex
def update_plot(rankindex):
    axb.clear()  # Clear the previous content

    # Update the image
    axb.imshow(all_labeled_im[regionidx_sorted[rankindex]], origin='lower')

    # Get the region data
    regionmap_index = regionidx_sorted[rankindex]
    surface, xmap, ymap, centralimageindex, Xroi, Yroi = regions[:, 3:][regionmap_index]

    # Update the title
    txttitle = f'largest region. rankindex: {rankindex} region index: {regionmap_index}'
    txttitle += f'\nseen in roi #{regions[:, 0][regionidx_sorted[rankindex]]} located @ pixel({Xroi},{Yroi})'
    txttitle += f'\nwith surface: {surface}'
    txttitle += f'\n mean position ({xmap}, {ymap}), mean imageindex {centralimageindex}'
    axb.set_title(txttitle, fontsize=8)

    # Redraw the figure
    figb.canvas.draw()

# Create a slider for rankindex
rankindex_slider = widgets.IntSlider(
    value=2,
    min=0,
    max=len(regionidx_sorted) - 1,
    step=1,
    description='Rank Index:',
    continuous_update=False
)

# Use interact to link the slider to the update_plot function
widgets.interact(update_plot, rankindex=rankindex_slider)

In [ ]:
# keep only region surface higher than `minimumsurface`  in pixel**2 unit on map
minimumsurface = 4

# sorting regionmap by surface
sorted_all_labeled_im = all_labeled_im[regionidx_sorted]
sorted_surfaceregions = regions[:,3][regionidx_sorted]
# filtering = rejecting small sized regions
cutindex = len(regionidx_sorted) - np.searchsorted(sorted_surfaceregions[::-1], minimumsurface)
#sorted_surfaceregions[:cutindex]

filtered_sorted_all_labeled_im=sorted_all_labeled_im[:cutindex]
print('remaining regionmap', len(filtered_sorted_all_labeled_im))
print('over ',len(all_labeled_im))

### compute common regions over all region maps

can take a while if `nbtotalregions` is larger than 2000


In [ ]:
nbtotalregions = filtered_sorted_all_labeled_im.shape[0]
print('nbtotalregions',nbtotalregions)

In [ ]:
# computation of mutual common area
ncalc =nbtotalregions
correlsurface_tab = np.inner(filtered_sorted_all_labeled_im[:ncalc].reshape(ncalc,-1),filtered_sorted_all_labeled_im[:ncalc].reshape(ncalc,-1))

In [ ]:
correlTab = np.copy(correlsurface_tab)
np.fill_diagonal(correlTab, 0)
correlTab

## query `correlTab`

In [ ]:
# rois  fired for a given region (grain or roi)
query_regionmapindex = 1110

print('local query_regionmapindex (among largest regions)',query_regionmapindex)
query_roi_index, query_corresponding_region_index, regionindex, _, xmap, ymap, centralimageindex, Xroi, Yroi = regions[regionidx_sorted][query_regionmapindex]
print('centralimageindex',centralimageindex)
print('roi counter index and region index', query_roi_index, query_corresponding_region_index)
query_roi_location = d['peaklist'][query_roi_index]

brightregions = np.where(correlsurface_tab[query_regionmapindex]>0)  # index of element in regionidx_sorted list
if 0:
    print('corresponding bright regions indices',brightregions)
    print('corresponding roi indices and their region index', regions[:,0:2][regionidx_sorted[brightregions]])

### list of roi center coordinates fired at the same location on map

In [ ]:
roicounterindex_correl = regions[:,0:2][regionidx_sorted[brightregions]][:,0]

txt = f'\n{len(roicounterindex_correl)} pixel position(s) of ROI fired at same sample position on the map than roi index {query_roi_index}'
txt += f'\nlocated at {query_roi_location}'
txt += f'\nset to `correlated_rois`'
print( txt)

correlated_rois = d['peaklist'][roicounterindex_correl]
print(correlated_rois.tolist())

In [ ]:
listidx = roicounterindex_correl
print(listidx)

In [ ]:
print('nb of roi counters satisfying the thresholding condition', len(listidx))
print('over total nb of rois', bigtr.shape[0])

### multiple plot of correlated rois

In [ ]:
nbcols = 4
autocontrast = True
nbrois_to_plot = 24

assert nbrois_to_plot < 61 # to avoid time consuming plot !!

nbimagesperline = d['nbimagesperline']
numrows, numcols = d['mapdimensions']

_list_idx_counters=list(listidx)[:nbrois_to_plot]  

nbcounters = len(_list_idx_counters)

#----------------------------------
nrows=nbcounters//nbcols
if nbcounters%nbcols != 0:
    nrows +=1

print('nrows',nrows)

if nbcounters<nrows*nbcols:

    _list_idx_counters = _list_idx_counters +['None'] * (nrows*nbcols - nbcounters)

print('nbcounters,nrows, nbcols')
print(nbcounters,nrows, nbcols)
fig3, axss = plt.subplots(ncols=nbcols, nrows=nrows, sharex=True, sharey=True, figsize=(8,16))

missingplot=np.zeros_like(tr[0]).reshape((-1,nbimagesperline))

k=0
axssflat=axss.flat
for ax, _idx in zip(axssflat,_list_idx_counters):
    if k<nbcounters:
        roidata = tr[_idx]
        if MONITORDATA_AVAILABLE:
            roidata = roidata / monitor_data2D.ravel()
        vmin,vmax=None,None
        if autocontrast:
            #print('mean',np.mean(roidata))
            #vmin, vmax = 1010, max(1010, 0.75*np.amax(roidata))
            pass
        
        ax.imshow(roidata.reshape((-1,nbimagesperline)), origin='lower', vmin=vmin,vmax=vmax)
        #ax.format_coord = format_coord
        #ax.set_title('%d'%_idx, fontsize=6, color='red')
        ax.text(0.5, 0.05, '%d'%_idx,
                        horizontalalignment='center',
                        verticalalignment='center',
                        fontsize=7, color='red',
                        transform=ax.transAxes)
    else:
        ax.imshow(missingplot, origin='lower')
        ax.text(0.5, 0.5, 'no data',
                        horizontalalignment='center',
                        verticalalignment='center',
                        fontsize=5, color='red',
                        transform=ax.transAxes)
    k+=1
plt.show()
plt.tight_layout(pad=0)

In [ ]:
# show all roi counters position that have also signal (above threshold) at the given point of the map.
commonspots = gridroi[listidx]
pixX, pixY = commonspots.T

print('centralimageindex', centralimageindex)

figco, axco = plt.subplots(figsize=(5,5))
axco.scatter(pixX, pixY)
axco.set_ylim(2100,0)
axco.set_xlim(0,2100)
txttitle =f'Laue Pattern of Common Roi centers\n at (fastindex, slowindex) = (%d, %d)\nimageindex: {centralimageindex}'%(xmap, ymap)
axco.set_title(txttitle, fontsize=8)
axco.set_xlabel('X')
axco.set_xlabel('Y')
axco.grid()

In [ ]:
fig12, ax12 = plt.subplots()

import LaueTools.indexingImageMatching as IIM
import LaueTools.LaueGeometry as LaueGeo

calibdetfile = '/data/visitor/a321217/bm32/20260707/RAW_DATA/LaB6_2/LaB6_2_gedaxm/scan0001/calibGe001_crude_eiger4M_july2026_A321217.det'
detectorplaneparameters = [99.66981616892743, 1077.5005899952898, 1177.652469333325, 0.4096519569722474, 0.4229933899487947]
tth, chi = LaueGeo.calc_uflab(pixX, pixY,detectorplaneparameters, pixelsize=0.075)

gnomonx, gmonony = IIM.ComputeGnomon_2([tth, chi])

ax12.scatter(gnomonx, gmonony)
ax12.grid()

ax12.set_xlabel('gnomonX')
ax12.set_ylabel('gnomonY')
ax12.set_title('gnomonic coordinates of roi centers')
plt.show()

In [ ]:
#imageindex=2733
#imageindex = convertindex2Dto1D(10,65,d['mapdimensions'])
imageindex = centralimageindex
print('imageindex',imageindex)


fullpath = os.path.join(d['folder'],f'{d['prefix']}%04d.{d['suffix']}'%imageindex)
with fabio.open(fullpath) as img:
    imgdata = img.data
    
fig,ax = plt.subplots()
#ax.imshow(np.log10(imgdata), vmin = 3, vmax = 3.5, cmap=plt.cm.inferno)
ax.imshow(imgdata, vmin = 0, vmax =500, cmap=plt.cm.OrRd)

for pt in commonspots:
    ax.scatter(pt[0],pt[1],marker='s',color='b', s=30, alpha=0.2)
    ax.set_title('%s\n%s'%(ExperimentFolder,fullpath.rsplit('/RAW_DATA/')[1]))
plt.show()

### Finding peak (hottestr pixel) for all `commonspots` rois (for a given image)

In [ ]:
halfboxsize = 5
squareboxlength = halfboxsize*2+1
fullpath = os.path.join(d['folder'],f'{d['prefix']}%04d.{d['suffix']}'%imageindex)
with fabio.open(fullpath) as img:
    imgdata = img.data

# ugly way without vectorization
positions_of_max = []
for pt in commonspots:
    #print(pt)
    jc, ic = pt
    if ic-halfboxsize<0 or  ic+halfboxsize>2015 or jc-halfboxsize<0 or  jc+halfboxsize>2015:
        continue
    tabI = imgdata[ic-halfboxsize:ic+halfboxsize+1, jc-halfboxsize:jc+halfboxsize+1]
    #print('max intensity:',np.amax(tabI))
    p0 = np.argmax(tabI)
    j_of_max, i_of_max = int(p0%squareboxlength-halfboxsize+jc), int(p0//squareboxlength-halfboxsize+ic)
    #print('at ', j_of_max, i_of_max)
    positions_of_max.append([j_of_max, i_of_max])

positions_of_max

In [ ]:
# plot updated common spots
fullpath = os.path.join(d['folder'],f'{d['prefix']}%04d.{d['suffix']}'%imageindex)
with fabio.open(fullpath) as img:
    imgdata = img.data
    
fig,ax = plt.subplots()
#ax.imshow(np.log10(imgdata), vmin = 3, vmax = 3.5, cmap=plt.cm.inferno)
ax.imshow(imgdata, vmin = 0, vmax =500, cmap=plt.cm.OrRd)

for pt_fine in positions_of_max:
    ax.scatter(pt_fine[0],pt_fine[1],marker='s',color='b', s=30, alpha=0.2)
    ax.set_title('%s\n%s'%(ExperimentFolder,fullpath.rsplit('/RAW_DATA/')[1]))

In [ ]:
# Create the initial figure and axis
fig, ax = plt.subplots()
plt.show(block=False)  # Display the figure without blocking
# Load the image data
imageindex0 = centralimageindex
fullpath = os.path.join(d['folder'], f'{d['prefix']}%04d.{d['suffix']}' % imageindex0)
with fabio.open(fullpath) as img:
    imgdata = img.data
im = ax.imshow(imgdata, vmin=vmin, vmax=vmax, cmap=plt.cm.OrRd)

# Store the current axis limits (initialized with default values)
current_xlim = ax.get_xlim()
current_ylim = ax.get_ylim()

# Function to update the plot based on vmin and vmax
def update_plot(vmin, vmax):
    # Store the current axis limits (zoom/pan state)
    global current_xlim, current_ylim
    current_xlim = ax.get_xlim()
    current_ylim = ax.get_ylim()

    ax.clear()  # Clear the previous content

    # Load the image data
    fullpath = os.path.join(d['folder'], f'{d['prefix']}%04d.{d['suffix']}' % imageindex)
    with fabio.open(fullpath) as img:
        imgdata = img.data

    # Update the image with new vmin/vmax
    im = ax.imshow(imgdata, vmin=vmin, vmax=vmax, cmap=plt.cm.OrRd)

    # Plot the scatter points
    for pt_fine in positions_of_max: #peaklist:#positions_of_max:
        ax.scatter(pt_fine[0], pt_fine[1], marker='s', color='b', s=30, alpha=0.2)

    # Set the title
    ax.set_title('%s\n%s' % (ExperimentFolder, fullpath.rsplit('/RAW_DATA/')[1]))

    # Restore the zoom/pan state
    ax.set_xlim(current_xlim)
    ax.set_ylim(current_ylim)

    # Redraw the figure
    fig.canvas.draw()

# Create sliders for vmin and vmax
vmin_slider = widgets.FloatSlider(
    value=0,
    min=0,
    max=7000,
    step=10,
    description='vmin:',
    continuous_update=False
)

vmax_slider = widgets.FloatSlider(
    value=7000,
    min=100,
    max=7000,
    step=10,
    description='vmax:',
    continuous_update=False
)

# Use interact to link the sliders to the update_plot function
widgets.interact(update_plot, vmin=vmin_slider, vmax=vmax_slider)

In [ ]:
# a way to fit several peaks on a single imahe: results are background , amplitude, X, Y , ...
# fitted_spots = collectroisfitpeak_singlefile(imageindex,positions_of_max,'img_',folder=d['folder'],
#                               boxsize_X=10,boxsize_Y=10,CCDLabel='sCMOS')

In [ ]:
# an other way to fit several peaks on a single image
guessed_peaksize = 0.9
MaxFitPixelDev = 5   # be careful not to exclude broad peaks or far initial guesses by a small value
tabIsorted, params_res, _ = RMCCD.fitoneimage_manypeaks(fullpath,
                                            np.array(positions_of_max),
                                            10,
                                            CCDLabel=CCDLabel,
                                            dirname=None,
                                            position_start="max",
                                            type_of_function="gaussian",
                                            guessed_peaksize=(guessed_peaksize, guessed_peaksize),
                                            xtol=0.001,  # accept all
                                            FitPixelDev=MaxFitPixelDev,  # accept all pixel deviation
                                            Ipixmax=None,
                                            verbose=0,
                                            position_definition=1,
                                            use_data_corrected=None,
                                            reject_negative_baseline=True,
                                            computerrorbars=False)

print('nb peak fitted', len(tabIsorted))
peaklist = tabIsorted[:,0:2]

### write .dat of restricted list of common peaks

In [ ]:
# write .dat file
writefolder=d['folder']
imagefilename = f'{d['prefix']}%04d.{d['suffix']}'%imageindex
fullpath_datfile = IOLT.writefile_Peaklist(f'{d['prefix']}%04d_selectedspots'%imageindex, tabIsorted,
                                                True,imagefilename ,
                                                None,writefolder, verbose=0)

fullpath_datfile

In [ ]:
#search for .det file in `ExperimentFolder`
import os

def find_files(root, extension='.det', sortbydate=False):
    det_files = []
    for folder, subfolders, files in os.walk(root):
        for file in files:
            if file.endswith(extension):
                full_path = os.path.join(folder, file)
                if sortbydate:
                    try:
                        mod_time = os.path.getmtime(full_path)
                        det_files.append((full_path, mod_time))
                    except OSError:
                        pass  # skip files that cause errors
                else:
                    det_files.append(full_path)
    if sortbydate:
        # Sort by modification time (newest first)
        det_files.sort(key=lambda x: x[1], reverse=True)
        return [path for path, _ in det_files]
    return det_files
    
def find_files_string(root, pattern=f'{d["prefix"]}', extension='.det', sortbydate=False):
    det_files = []
    for folder, subfolders, files in os.walk(root):
        for file in files:
            if file.endswith(extension) and pattern in file:
                full_path = os.path.join(folder, file)
                if sortbydate:
                    try:
                        mod_time = os.path.getmtime(full_path)
                        det_files.append((full_path, mod_time))
                    except OSError:
                        pass  # skip files that cause errors
                else:
                    det_files.append(full_path)
    if sortbydate:
        # Sort by modification time (newest first)
        det_files.sort(key=lambda x: x[1], reverse=True)
        return [path for path, _ in det_files]
    return det_files

print('in ExperimentFolder :', ExperimentFolder)
detfile_list = find_files(ExperimentFolder)
detfile_list = find_files(ExperimentFolder, extension='.det')
detfile_list

### write .cor of restricted list of common peaks

In [ ]:
# for blc 16200
detfile = '/data/visitor/blc16200/bm32/20250508/RAW_DATA/ech15/ech15_Gedaxm/scan0001/calibGe001_BLC16200_ech15_ZrCr.det'
# for ma6496
#detfile = detfile_list[1]
# for ihma714
#detfile = detfile_list[0]
# for 321217 
detfile = '/data/visitor/a321217/bm32/20260707/RAW_DATA/LaB6_2/LaB6_2_gedaxm/scan0001/calibGe001_crude_eiger4M_july2026_A321217.det'

import LaueTools.LaueGeometry as LaueGeo

In [ ]:
# write .cor file (with scattering angles) for indexation from .dat file
try:
    dictcalibparams = None
    #dictcalibparams = IOLT.readCalib_det_file('/data/bm32/inhouse/LAUE/Test02Sep20_LAUE/Jeudi/calibSilukas.det')
    dictcalibparams = IOLT.readCalib_det_file(detfile)
except NameError:
    print('please provide the path to a .det file')

# write . cor if calibration parameters are known
if dictcalibparams is not None:# 
    if isinstance(dictcalibparams, dict):

        datfolder, datfile = os.path.split(fullpath_datfile)

        fullpath_corfile = LaueGeo.convert2corfile(datfile,
                    [],  # should be useless and pixelsize argument is also missing...
                    dirname_in=datfolder,
                    dirname_out=datfolder,
                    CCDCalibdict=dictcalibparams,
                    add_props=True)

        print('fullpath_corfile', fullpath_corfile)

In [ ]:
# read exhaustive .dat file (or cor file)

In [ ]:
def getPairsbetweenTwoSets(XY1, XY2, dist_tolerance=0.5, samelist=False):
    """
    return indices in XY1 and in XY2 of common pts (2D) and
    a flag is closest distances are below dist_tolerance

    :param XY1: list of 2D elements
    :param XY2: list of 2D elements
    :param dist_tolerance: largest distance (in unit of XY1, XY2) to consider two elements close enough
    :param samelist: boolean, default is False (when XY1 and XY2 are different). True if XY1=XY2 to
    find close spots in a single list of points

    :return:
    ind_XY1, ind_XY2: two arrays of indices which connect elementwise one element of XY1 to 1 element of XY2
    """
    x1, y1 = np.array(XY1).T

    x2, y2 = np.array(XY2).T

    diffx = x1[:, np.newaxis] - x2
    diffy = y1[:, np.newaxis] - y2

    _dist = np.hypot(diffx, diffy)

    if samelist:
        # add big distance in diagonal
        np.fill_diagonal(_dist, np.amax(_dist)+2*dist_tolerance)

    conddist = _dist <= dist_tolerance
    in1, in2 = np.where(conddist==True)
        
    return in1, in2

In [ ]:

corfilefolder = d['folder']
corfile_list = find_files_string(corfilefolder, 'selected','.cor')

datfilefolder = d['folder']
datfile_list = find_files_string(datfilefolder, 'img_','.dat')
#corfilename = os.path.join(corfilefolder,'img_%04d.cor'%imageindex)

In [ ]:
corfile_list

In [ ]:
if 0:  # an other way
    # selecting .dat and convert to .cor file
    fullpath_datfile = datfile_list[1]
    fullpath_datfile
    # write . cor if calibration parameters are known
    if dictcalibparams is not None:# 
        if isinstance(dictcalibparams, dict):
    
            datfolder, datfile = os.path.split(fullpath_datfile)
    
            fullpath_corfile = LaueGeo.convert2corfile(datfile,
                        [],  # should be useless and pixelsize argument is also missing...
                        dirname_in=datfolder,
                        dirname_out=datfolder,
                        CCDCalibdict=dictcalibparams,
                        add_props=True)
    
            print('fullpath_corfile', fullpath_corfile)

In [ ]:
# read a corfile and located spots being found in tabIsorted

#corfilename = os.path.join(corfilefolder,'img_1865_selectedspots.cor')
corfilename = fullpath_corfile
datacor, *othercor= IOLT.readfile_cor(corfilename, output_CCDparamsdict=True, output_only5columns=False)
datacorXYI = datacor[:,2:5]
indexincorfile, _ = getPairsbetweenTwoSets(datacorXYI[:,:2],tabIsorted[:,:2],dist_tolerance=1)
#indexincorfile

setnotincorfile = set(range(len(datacorXYI)))-set(indexincorfile.tolist())
preferedorder = indexincorfile.tolist()+list(setnotincorfile)
#preferedorder

In [ ]:
#!less {corfilename}

In [ ]:
# rearrange datacor and dict_spotsproperties to put TabIsorted spots at the beginning of the spot list
# then write a starred_####.cor file

starrred_datacor = np.take(datacor[:,:5], preferedorder, axis=0)
starred_dictproperties = copy.copy(dict_spotsproperties)
starred_dictproperties['data_spotsproperties'] = np.take(dict_spotsproperties['data_spotsproperties'], preferedorder, axis=0)
# write .cor file with redordered list of peaks
tthchixyI = starrred_datacor[:,:5].T.tolist()

# from initial .cor file
CCDcalibdict = othercor[-2]
dict_spotsproperties = othercor[-1]
#CCDcalibdict

outputfolder = d['folder']
starredcorfilename = IOLT.writefile_cor(f'starred_{centralimageindex}', *tthchixyI, param=CCDcalibdict,
                   dirname_output=d['folder'], dict_data_spotsproperties=starred_dictproperties)
GT.printgreen(f'\nA .cor file optimized for indexing is written in:\n{outputfolder}\n {starredcorfilename}')

## SAND BOX selection of region of the map (from the segmentation of the map of 1 given roi counter)

In [ ]:
# nb of pixels per cluster
# grain index = 0 = not a grain!
sizes = ndimage.sum(mask, label_im, range(1,nb_labels + 1))
sizes

In [ ]:
#label of largest cluster
labelidx = np.argmax(sizes) + 1
labelidx

In [ ]:
#mean intensity  per object
# grain index = 0 = not a grain!
mean_vals = ndimage.mean(im, label_im, range(1, nb_labels + 1))
mean_vals

In [ ]:
mean_vals[labelidx-1]

In [ ]:
# sorting by mean intensity
#bestgrains_idx = np.argsort(mean_vals)[::-1]

# sorting by region size
bestgrains_idx = np.argsort(sizes)[::-1]
# 10 most intense grain
print('bestgrains_idx[:10]',bestgrains_idx[:10])
print('scattering intensity',np.take(mean_vals, bestgrains_idx[:10],axis=0))
print('grainsize',np.take(sizes, bestgrains_idx[:10],axis=0))

In [ ]:
# center of mass (ymap, xmap)
# grain index = 0 = not a grain!
#
coms = ndimage.center_of_mass(im, label_im, range(1, nb_labels + 1))
coms

In [ ]:
coms[labelidx-1]

In [ ]:
from scipy.ndimage import find_objects
# grain index = 0 = not a grain!
# now f[grain index-1] 
f = find_objects(label_im)
len(f), f[labelidx-1],#f
# slice1  along i or Ymap  slice2 along j or Xmap

In [ ]:
#im[f[27]], mask[f[27]]

In [ ]:
# # finding the number of positions on the map which are the region # 141 of image 1651

# cond = np.logical_and(np.where(label_im==141, True, False),np.where(im>Threshold, True, False))

# # where map 1015 intensity is larger than Threshold and at position defined by cond
# maskXY = np.where(cond==True)
# boolres = bigtr[1651][maskXY]>Threshold
# print('nb of True', boolres.sum())
# boolres = bigtr[1015][maskXY]>Threshold
# print('nb of True', boolres.sum())

In [ ]:
# finding the number of positions on the map which are the region # labelidx of image # idx_counter0
print('region with labelidx :',labelidx)
print('in segmented map of  roi counter :',idx_counter0)
cond = np.logical_and(np.where(label_im==labelidx, True, False),np.where(im>Threshold, True, False))

# where map 1937 intensity is larger than Threshold and at position defined by cond
maskXY = np.where(cond==True)


In [ ]:
bigtr.shape

In [ ]:
# array of common number of map pts
commonregionsize = np.sum(bigtr[:,maskXY[0],maskXY[1]]>Threshold, axis=1)
commonregionsize

In [ ]:
highcorrelatedcounters = np.argsort(commonregionsize)[::-1]
highcorrelatedcounters

In [ ]:
figco, axco = plt.subplots()
maxsize= commonregionsize[highcorrelatedcounters[0]]
axco.plot(commonregionsize[highcorrelatedcounters]/maxsize, '-o')
axco.grid()

In [ ]:
# find largest index for which correlation is larger than 2/3
maxindexroicounter = np.where(commonregionsize[highcorrelatedcounters]/maxsize<2./3)[0][0]
maxindexroicounter

In [ ]:
nbcols = 5
autocontrast = True

nbimagesperline = d['nbimagesperline']
numrows, numcols = d['mapdimensions']

# automated selection of largest grain
_list_idx_counters=highcorrelatedcounters[:60]


nbcounters = len(_list_idx_counters)

#----------------------------------
nrows=nbcounters//nbcols
if nbcounters%nbcols != 0:
    nrows +=1

print('nrows',nrows)

if nbcounters<nrows*nbcols:

    _list_idx_counters = _list_idx_counters +['None'] * (nrows*nbcols - nbcounters)

print(nbcounters,nrows, nbcols)
fig3, axss = plt.subplots(ncols=nbcols, nrows=nrows, sharex=True, sharey=True, figsize=(8,16))

missingplot=np.zeros_like(tr[0]).reshape((-1,nbimagesperline))

k=0
axssflat=axss.flat
for ax, _idx in zip(axssflat,_list_idx_counters):
    if k<nbcounters:
        roidata = tr[_idx]
        vmin,vmax=None,None
        if autocontrast:
            #print('mean',np.mean(roidata))
            vmin, vmax = 1010, max(1010, 0.75*np.amax(roidata))
        
        ax.imshow(roidata.reshape((-1,nbimagesperline)), origin='lower', vmin=vmin,vmax=vmax)
        #ax.format_coord = format_coord
        #ax.set_title('%d'%_idx, fontsize=6, color='red')
        ax.text(0.5, 0.05, '%d'%_idx,
                        horizontalalignment='center',
                        verticalalignment='center',
                        fontsize=7, color='red',
                        transform=ax.transAxes)
    else:
        ax.imshow(missingplot, origin='lower')
        ax.text(0.5, 0.5, 'no data',
                        horizontalalignment='center',
                        verticalalignment='center',
                        fontsize=5, color='red',
                        transform=ax.transAxes)
    k+=1

plt.tight_layout(pad=0)

## roi positions that are highly spatially correlated 

`commonspots2` should be better than `commonspots`  ,... but not the case yet (strange, maybe mixing data)

In [ ]:
commonspots2 = gridroi[highcorrelatedcounters][:maxindexroicounter]

pixX, pixY = commonspots.T

figco, axco = plt.subplots()
axco.scatter(pixX, pixY)
axco.set_ylim(2100,0)
axco.set_xlim(0,2100)
axco.set_title('Laue Pattern of Common Roi centers for Xmap, Ymap = %d, %d'%(xmap, ymap))
axco.set_xlabel('X')
axco.set_ylabel('Y')
axco.grid()

In [ ]:
# from X Y map to image index
mapdimensions = d['mapdimensions']
xmap, ymap = 95,21  # j, i
imageindex = ymap*mapdimensions[0]+xmap
imageindex

In [ ]:
imageindex=3602
fullpath = os.path.join(d['folder'],'img_%04d.tif'%imageindex)
with fabio.open(fullpath) as img:
    imgdata = img.data
    
fig,ax = plt.subplots()
#ax.imshow(np.log10(imgdata), vmin = 3, vmax = 3.5, cmap=plt.cm.inferno)
ax.imshow(imgdata, vmin = 1000, vmax =3000, cmap=plt.cm.OrRd)
for pt in commonspots2:
    ax.scatter(pt[0],pt[1],marker='o',color='b', alpha=0.2)
    ax.set_title('%s\n%s'%(ExperimentFolder,fullpath.rsplit('/RAW_DATA/')[1]))

## fitting strongest peak in each high correlated roi

In [ ]:
resfit = RMCCD.readoneimage_multiROIfit(os.path.join(d['folder'],'img_%04d.tif'%imageindex),
                                        commonspots2,
                                        [10,10],
                                        stackimageindex=-1,
                                        CCDLabel='sCMOS',
                                        baseline="auto",  # min in ROI box
                                        startangles=0.0,
                                        start_sigma1=1.,
                                        start_sigma2=1.,
                                        position_start='max',  # 'centers' or 'max'
                                        showfitresults=0,
                                        offsetposition=1,
                                        fitfunc="gaussian",
                                        xtol=0.00001,
                                        addImax=False,
                                        use_data_corrected=None)

In [ ]:
XYfit = np.array(resfit[0])[:,2:4]
XYfit

In [ ]:
XYfit = GT.purgeClosePoints2(XYfit,8)[0]
print('nb of spots', len(XYfit))

In [ ]:
imageindex=3602
fullpath = os.path.join(d['folder'],'img_%04d.tif'%imageindex)
with fabio.open(fullpath) as img:
    imgdata = img.data
    
fig,ax = plt.subplots()
#ax.imshow(np.log10(imgdata), vmin = 3, vmax = 3.5, cmap=plt.cm.inferno)
ax.imshow(imgdata, vmin = 1000, vmax =3000, cmap=plt.cm.OrRd)
for pt in XYfit:
    ax.scatter(pt[0],pt[1],marker='o',color='b', alpha=0.2)
    ax.set_title('%s\n%s'%(ExperimentFolder,fullpath.rsplit('/RAW_DATA/')[1]))

#  similarity

In [ ]:
from sewar.full_ref import mse, rmse, psnr, uqi, ssim, ergas, scc, rase, sam, msssim, vifp

##List of estimators to compare two images
#Mean Squared Error (MSE)
#Root Mean Squared Error (RMSE)
#Peak Signal-to-Noise Ratio (PSNR)
#Structural Similarity Index (SSIM)
#Universal Quality Image Index (UQI)
#Multi-scale Structural Similarity Index (MS-SSIM)
#Erreur Relative Globale Adimensionnelle de Synthèse (ERGAS)
#Spatial Correlation Coefficient (SCC)
#Relative Average Spectral Error (RASE)
#Spectral Angle Mapper (SAM)
#Visual Information Fidelity (VIF)

from tqdm import trange

index_of_interest = 87

index_ = []
for i in trange(len(tr)):
    ind = vifp(tr[i].reshape((-1,nbimagesperline)), tr[index_of_interest].reshape((-1,nbimagesperline)))
    index_.append(ind)
    
index_ = np.array(index_)

sim_list_idx_counters = np.where(index_>0.1)[0]

#    DIC    affine transform corioss correlation

In [ ]:
folder= '/home/esrf/micha/lauetools/notebooks'
img='grain2_5411.mccd'
ii,_,_=IOimage.readCCDimage(os.path.join(folder,img), CCDLabel='MARCCD165')
np.amax(ii)

In [ ]:
crop1 = ii[1350-25:1350+25, 1325-25:1325+25]
fig, ax =plt.subplots()
ax.imshow(crop1)
ii.dtype

In [ ]:
from scipy import signal

from scipy import misc

rng = np.random.default_rng()

face = misc.face(gray=True) - misc.face(gray=True).mean()

laueim = ii#-ii.mean()

#template = np.copy(face[300:365, 670:750])  # right eye
template = crop1.astype(np.float64)
template -= template.mean()

#face = face + rng.standard_normal(face.shape) * 50  # add noise

corr = signal.correlate2d(laueim, template, boundary='symm', mode='same')

y, x = np.unravel_index(np.argsort(corr.flatten())[::-1][:10], corr.shape)  # find the match


In [ ]:
corr.shape, np.argmax(corr), np.argsort(corr.flatten())[::-1]

In [ ]:
import matplotlib.pyplot as plt

fig, (ax_orig, ax_template, ax_corr) = plt.subplots(3, 1,

                                                    figsize=(6, 15))

ax_orig.imshow(laueim, cmap='gray')
ax_orig.set_title('Original')
ax_orig.set_axis_off()
ax_template.imshow(template, cmap='gray')
ax_template.set_title('Template')
ax_template.set_axis_off()
ax_corr.imshow(corr, cmap='gray')
ax_corr.set_title('Cross-correlation')
ax_corr.set_axis_off()
ax_orig.plot(x, y, 'ro')